> **Versión GitHub reproducible.** Esta copia no modifica el notebook original de Google Drive. Las rutas personales de Drive se han sustituido por rutas del repositorio. Los datos brutos de gran tamaño no incluidos en GitHub deben descargarse según `docs/DATA_SOURCES.md`.


# Preparación entorno y variables

In [ ]:
# ============================================================
# CONFIGURACIÓN DE RUTAS - VERSIÓN GITHUB / GOOGLE COLAB
# ============================================================
# Clone primero el repositorio con:
# !git clone https://github.com/ritaridh/tfm-crop-yield-prediction.git /content/TFM_Rendimiento_cultivos
# %cd /content/TFM_Rendimiento_cultivos

from pathlib import Path
import os

ROOT = Path("/content/TFM_Rendimiento_cultivos")
if not ROOT.exists():
    raise FileNotFoundError(
        "No se encuentra el repositorio. Ejecute primero el bloque de clonación indicado en el README."
    )

RUTA_TFM = str(ROOT)
RUTA_BASE = str(ROOT)
BASE = str(ROOT)
BASE_DIR = ROOT
RUTA_ORIGINALES = str(ROOT / "01_Datos_originales")
RUTA_EUROCROPS = str(ROOT / "01_Datos_originales" / "EuroCrops")
RUTA_MAPA = str(ROOT / "01_Datos_originales" / "MAPA")
RUTA_SENTINEL = str(ROOT / "01_Datos_originales" / "Sentinel2")
RUTA_PROCESADOS = str(ROOT / "02_Datos_procesados")
RUTA_RESULTADOS = str(ROOT / "05_Resultados")
PROC = RUTA_PROCESADOS
RESULTADOS = RUTA_RESULTADOS

Path(RUTA_PROCESADOS).mkdir(parents=True, exist_ok=True)
Path(RUTA_RESULTADOS).mkdir(parents=True, exist_ok=True)
print("Repositorio:", ROOT)


In [ ]:
# ============================================================
# PASO 2 - DEFINIR TARGET, IDENTIFICADORES Y PREDICTORES
# ============================================================

TARGET = "rendimiento_kg_ha"

IDENTIFICADORES = [
    "ano",
    "cultivo",
    "provincia_estandar"
]

PREDICTORES = [
    c for c in df.columns
    if c not in (
        IDENTIFICADORES
        + [TARGET]
    )
]

print("Target:")
print(TARGET)

print("\nNúmero de predictores:")
print(len(PREDICTORES))

print("\nPredictores:")
for i, c in enumerate(PREDICTORES, 1):
    print(f"{i:02d}. {c}")

In [ ]:
# ============================================================
# PASO 3 - DEFINIR BLOQUES DE PREDICTORES
# ============================================================

cols_era5 = [
    c for c in df.columns
    if (
        c.startswith("temp_media_")
        or c.startswith("temp_rocio_")
        or c.startswith("precipitacion_")
        or c.startswith("radiacion_")
    )
]

cols_soil = [
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm"
]

cols_sentinel = [
    c for c in df.columns
    if c.startswith(
        ("NDVI_", "EVI_", "NDMI_")
    )
]

cols_eurocrops = [
    "n_recintos",
    "superficie_eurocrops_ha",
    "superficie_media_recinto_ha"
]

bloques = {
    "ERA5": cols_era5,
    "SoilGrids": cols_soil,
    "Sentinel2": cols_sentinel,
    "EuroCrops": cols_eurocrops
}

for nombre, columnas in bloques.items():
    print(
        nombre,
        "->",
        len(columnas),
        "variables"
    )

print(
    "\nTotal:",
    sum(len(v) for v in bloques.values())
)

In [ ]:
# ============================================================
# PASO 4 - ESCENARIOS DE FUSIÓN DE DATOS
# ============================================================

feature_sets = {

    "ERA5":
        cols_era5,

    "Sentinel2":
        cols_sentinel,

    "ERA5_Sentinel2":
        cols_era5
        + cols_sentinel,

    "ERA5_Soil":
        cols_era5
        + cols_soil,

    "ERA5_Sentinel2_Soil":
        cols_era5
        + cols_sentinel
        + cols_soil,

    "Fusion_completa":
        cols_era5
        + cols_sentinel
        + cols_soil
        + cols_eurocrops
}

for nombre, columnas in feature_sets.items():

    print(
        f"{nombre:25s}",
        len(columnas),
        "variables"
    )

In [ ]:
# ============================================================
# PASO 5 - HOLDOUT TEMPORAL
# ============================================================

df_train = df[
    df["ano"] <= 2022
].copy()

df_test = df[
    df["ano"] == 2023
].copy()

print("TRAIN")
print("Años:", sorted(df_train["ano"].unique()))
print("Observaciones:", len(df_train))

print("\nTEST")
print("Años:", sorted(df_test["ano"].unique()))
print("Observaciones:", len(df_test))

print("\nTRAIN por cultivo:")
print(df_train["cultivo"].value_counts())

print("\nTEST por cultivo:")
print(df_test["cultivo"].value_counts())

In [ ]:
# ============================================================
# PASO 6 - CONTROL ANTI-LEAKAGE TEMPORAL
# ============================================================

print(
    "Máximo año TRAIN:",
    df_train["ano"].max()
)

print(
    "Mínimo año TEST:",
    df_test["ano"].min()
)

assert (
    df_train["ano"].max()
    < df_test["ano"].min()
)

print(
    "\nSeparación temporal correcta."
)

In [ ]:
# ============================================================
# PASO 7 - DEFINICIÓN DE LA VALIDACIÓN TEMPORAL PROGRESIVA
# ============================================================

folds_temporales = [
    {
        "fold": 1,
        "anos_train": [2017, 2018],
        "ano_validacion": 2019
    },
    {
        "fold": 2,
        "anos_train": [2017, 2018, 2019],
        "ano_validacion": 2020
    },
    {
        "fold": 3,
        "anos_train": [2017, 2018, 2019, 2020],
        "ano_validacion": 2021
    },
    {
        "fold": 4,
        "anos_train": [2017, 2018, 2019, 2020, 2021],
        "ano_validacion": 2022
    }
]

resumen_folds = []

for f in folds_temporales:

    train_fold = df_train[
        df_train["ano"].isin(f["anos_train"])
    ]

    val_fold = df_train[
        df_train["ano"] == f["ano_validacion"]
    ]

    resumen_folds.append({
        "fold": f["fold"],
        "train_anos": f"{min(f['anos_train'])}-{max(f['anos_train'])}",
        "validacion_ano": f["ano_validacion"],
        "n_train": len(train_fold),
        "n_validacion": len(val_fold),
        "train_barley": (train_fold["cultivo"] == "barley").sum(),
        "train_wheat": (train_fold["cultivo"] == "common_soft_wheat").sum(),
        "val_barley": (val_fold["cultivo"] == "barley").sum(),
        "val_wheat": (val_fold["cultivo"] == "common_soft_wheat").sum()
    })

resumen_folds = pd.DataFrame(resumen_folds)

display(resumen_folds)

In [ ]:
# ============================================================
# PASO 8 - CONTROLES ANTI-LEAKAGE
# ============================================================

for f in folds_temporales:

    train_fold = df_train[
        df_train["ano"].isin(f["anos_train"])
    ]

    val_fold = df_train[
        df_train["ano"] == f["ano_validacion"]
    ]

    assert train_fold["ano"].max() < val_fold["ano"].min()

    print(
        f"Fold {f['fold']}: "
        f"TRAIN hasta {train_fold['ano'].max()} -> "
        f"VALIDACIÓN {val_fold['ano'].min()} | OK"
    )

# Test final
assert df_train["ano"].max() < df_test["ano"].min()

print(
    f"\nTEST FINAL: "
    f"TRAIN hasta {df_train['ano'].max()} -> "
    f"TEST {df_test['ano'].min()} | OK"
)

In [ ]:
# ============================================================
# PASO 9 - VARIABLES PARA EL MODELADO
# ============================================================

VARIABLE_CATEGORICA = ["cultivo"]

VARIABLES_NUMERICAS = (
    cols_era5
    + cols_soil
    + cols_sentinel
    + cols_eurocrops
)

VARIABLES_MODELO_COMPLETO = (
    VARIABLES_NUMERICAS
    + VARIABLE_CATEGORICA
)

print("Variables numéricas:", len(VARIABLES_NUMERICAS))
print("Variables categóricas:", len(VARIABLE_CATEGORICA))
print("Total variables de entrada:", len(VARIABLES_MODELO_COMPLETO))

print("\nVariable categórica:")
print(VARIABLE_CATEGORICA)

print("\nVariables excluidas del modelo:")
print(["ano", "provincia_estandar"])

print("\nTarget:")
print(TARGET)

In [ ]:
# ============================================================
# PASO 10 - BASELINES CON VALIDACIÓN TEMPORAL PROGRESIVA
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def calcular_metricas(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_true, y_pred)

    return mae, rmse, r2


resultados_baseline = []

for f in folds_temporales:

    train_fold = df_train[
        df_train["ano"].isin(f["anos_train"])
    ].copy()

    val_fold = df_train[
        df_train["ano"] == f["ano_validacion"]
    ].copy()

    y_true = val_fold[TARGET]

    # ========================================================
    # BASELINE 1 - MEDIA GLOBAL
    # ========================================================

    media_global = train_fold[TARGET].mean()

    pred_global = np.repeat(
        media_global,
        len(val_fold)
    )

    mae, rmse, r2 = calcular_metricas(
        y_true,
        pred_global
    )

    resultados_baseline.append({
        "fold": f["fold"],
        "ano_validacion": f["ano_validacion"],
        "modelo": "Media_global",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })


    # ========================================================
    # BASELINE 2 - MEDIA HISTÓRICA POR CULTIVO
    # ========================================================

    medias_cultivo = (
        train_fold
        .groupby("cultivo")[TARGET]
        .mean()
    )

    pred_cultivo = (
        val_fold["cultivo"]
        .map(medias_cultivo)
    )

    mae, rmse, r2 = calcular_metricas(
        y_true,
        pred_cultivo
    )

    resultados_baseline.append({
        "fold": f["fold"],
        "ano_validacion": f["ano_validacion"],
        "modelo": "Media_por_cultivo",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })


    # ========================================================
    # BASELINE 3 - MEDIA HISTÓRICA POR PROVINCIA
    # ========================================================

    medias_provincia = (
        train_fold
        .groupby("provincia_estandar")[TARGET]
        .mean()
    )

    pred_provincia = (
        val_fold["provincia_estandar"]
        .map(medias_provincia)
    )

    # Si alguna provincia no existiese previamente en train,
    # usar media global del train
    pred_provincia = pred_provincia.fillna(
        media_global
    )

    mae, rmse, r2 = calcular_metricas(
        y_true,
        pred_provincia
    )

    resultados_baseline.append({
        "fold": f["fold"],
        "ano_validacion": f["ano_validacion"],
        "modelo": "Media_por_provincia",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })


resultados_baseline_df = pd.DataFrame(
    resultados_baseline
)

display(
    resultados_baseline_df.round(2)
)

In [ ]:
# ============================================================
# PASO 11 - RESUMEN DE BASELINES
# ============================================================

resumen_baselines = (
    resultados_baseline_df
    .groupby("modelo")
    .agg(
        MAE_medio=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_medio=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        R2_medio=("R2", "mean"),
        R2_std=("R2", "std"),
        R2_min=("R2", "min"),
        R2_max=("R2", "max")
    )
    .sort_values(
        "RMSE_medio"
    )
)

display(
    resumen_baselines.round(3)
)

In [ ]:
# ============================================================
# PASO 12 - EVOLUCIÓN DE R2 DE BASELINES
# ============================================================

plt.figure(figsize=(9, 5))

for modelo in resultados_baseline_df["modelo"].unique():

    temp = resultados_baseline_df[
        resultados_baseline_df["modelo"] == modelo
    ]

    plt.plot(
        temp["ano_validacion"],
        temp["R2"],
        marker="o",
        label=modelo
    )

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Año de validación")
plt.ylabel("R²")
plt.title(
    "Estabilidad temporal de los modelos baseline"
)

plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
# ============================================================
# PASO 13 - DEFINICIÓN DE VARIABLES PARA MODELIZACIÓN
# ============================================================

TARGET = "rendimiento_kg_ha"

# Variables que NO entran directamente como predictores numéricos
columnas_id = [
    TARGET,
    "ano"
]

# Variables categóricas
variables_categoricas = [
    "cultivo",
    "provincia_estandar"
]

# Variables numéricas
variables_numericas = [
    c for c in df_train.columns
    if c not in columnas_id + variables_categoricas
]

print("=" * 70)
print("VARIABLES PARA MODELIZACIÓN")
print("=" * 70)

print("\nVariables numéricas:", len(variables_numericas))
print("Variables categóricas:", len(variables_categoricas))

print("\nCategóricas:")
for v in variables_categoricas:
    print(" -", v)

print("\nNuméricas:")
for v in variables_numericas:
    print(" -", v)

print("\nTotal predictores originales:",
      len(variables_numericas) + len(variables_categoricas))

print("\nTarget:", TARGET)

print("\nAño:")
print("No se utiliza inicialmente como predictor.")
print("Se utiliza para definir la estructura temporal train/validación/test.")

In [ ]:
# ============================================================
# PASO 14 - PIPELINE DE PREPROCESAMIENTO
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# Variables numéricas
# ------------------------------------------------------------

pipeline_numerico = Pipeline(
    steps=[
        (
            "imputacion",
            SimpleImputer(strategy="median")
        ),
        (
            "escalado",
            StandardScaler()
        )
    ]
)


# ------------------------------------------------------------
# Variables categóricas
# ------------------------------------------------------------

pipeline_categorico = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)


# ------------------------------------------------------------
# Preprocesador completo
# ------------------------------------------------------------

preprocesador = ColumnTransformer(
    transformers=[
        (
            "numericas",
            pipeline_numerico,
            variables_numericas
        ),
        (
            "categoricas",
            pipeline_categorico,
            variables_categoricas
        )
    ],
    remainder="drop"
)

print("Pipeline de preprocesamiento creado correctamente.")
print("\nNuméricas:", len(variables_numericas))
print("Categóricas:", len(variables_categoricas))

# Modelización - modelos lineales

# Ridge Regression con validación temporal

In [ ]:
# ============================================================
# PASO 15.1 - FUNCIÓN DE EVALUACIÓN
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


def evaluar_modelo_temporal(
    modelo,
    df,
    anos_validacion=(2019, 2020, 2021, 2022)
):

    resultados = []

    for fold, ano_val in enumerate(anos_validacion, start=1):

        # Entrenamiento: todos los años anteriores
        train_fold = df[df["ano"] < ano_val].copy()

        # Validación: año inmediatamente posterior
        val_fold = df[df["ano"] == ano_val].copy()

        X_train = train_fold[
            variables_numericas + variables_categoricas
        ]

        y_train = train_fold[TARGET]

        X_val = val_fold[
            variables_numericas + variables_categoricas
        ]

        y_val = val_fold[TARGET]

        modelo_fold = clone(modelo)
        modelo_fold.fit(X_train, y_train)

        pred = modelo_fold.predict(X_val)

        mae = mean_absolute_error(y_val, pred)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        r2 = r2_score(y_val, pred)

        resultados.append({
            "fold": fold,
            "ano_validacion": ano_val,
            "n_train": len(train_fold),
            "n_validacion": len(val_fold),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    return pd.DataFrame(resultados)


print("Función de evaluación temporal creada correctamente.")

In [ ]:
# ============================================================
# PASO 15.2 - BÚSQUEDA DE ALPHA PARA RIDGE
# ============================================================

from sklearn.pipeline import Pipeline

alphas_ridge = [
    0.001,
    0.01,
    0.1,
    1,
    10,
    100,
    1000
]

resultados_alpha = []

for alpha in alphas_ridge:

    modelo_ridge = Pipeline(
        steps=[
            ("preprocesado", clone(preprocesador)),
            ("modelo", Ridge(alpha=alpha))
        ]
    )

    resultados = evaluar_modelo_temporal(
        modelo=modelo_ridge,
        df=df_train
    )

    resultados_alpha.append({
        "alpha": alpha,
        "MAE_medio": resultados["MAE"].mean(),
        "RMSE_medio": resultados["RMSE"].mean(),
        "R2_medio": resultados["R2"].mean(),
        "R2_std": resultados["R2"].std(),
        "R2_min": resultados["R2"].min(),
        "R2_max": resultados["R2"].max()
    })


resultados_alpha_ridge = pd.DataFrame(resultados_alpha)

resultados_alpha_ridge = (
    resultados_alpha_ridge
    .sort_values("RMSE_medio")
    .reset_index(drop=True)
)

display(
    resultados_alpha_ridge.round(3)
)

In [ ]:
# ============================================================
# PASO 15.3 - MEJOR RIDGE: RESULTADOS POR FOLD
# ============================================================

mejor_alpha_ridge = resultados_alpha_ridge.iloc[0]["alpha"]

print("Mejor alpha Ridge:", mejor_alpha_ridge)


ridge_final_validacion = Pipeline(
    steps=[
        ("preprocesado", clone(preprocesador)),
        (
            "modelo",
            Ridge(alpha=mejor_alpha_ridge)
        )
    ]
)


resultados_ridge = evaluar_modelo_temporal(
    modelo=ridge_final_validacion,
    df=df_train
)


display(
    resultados_ridge.round(3)
)


print("\nRESUMEN RIDGE")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_ridge["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_ridge["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_ridge["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_ridge["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_ridge["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_ridge["R2"].max(), 3)
)

# Lasso

In [ ]:
# ============================================================
# PASO 16.1 - BÚSQUEDA DE ALPHA PARA LASSO
# ============================================================

from sklearn.linear_model import Lasso

alphas_lasso = [
    0.01,
    0.1,
    1,
    5,
    10,
    25,
    50,
    100,
    250,
    500
]

resultados_alpha = []

for alpha in alphas_lasso:

    modelo_lasso = Pipeline(
        steps=[
            ("preprocesado", clone(preprocesador)),
            (
                "modelo",
                Lasso(
                    alpha=alpha,
                    max_iter=50000
                )
            )
        ]
    )

    resultados = evaluar_modelo_temporal(
        modelo=modelo_lasso,
        df=df_train
    )

    resultados_alpha.append({
        "alpha": alpha,
        "MAE_medio": resultados["MAE"].mean(),
        "RMSE_medio": resultados["RMSE"].mean(),
        "R2_medio": resultados["R2"].mean(),
        "R2_std": resultados["R2"].std(),
        "R2_min": resultados["R2"].min(),
        "R2_max": resultados["R2"].max()
    })


resultados_alpha_lasso = pd.DataFrame(
    resultados_alpha
)

resultados_alpha_lasso = (
    resultados_alpha_lasso
    .sort_values("RMSE_medio")
    .reset_index(drop=True)
)

display(
    resultados_alpha_lasso.round(3)
)

In [ ]:
# ============================================================
# PASO 16.2 - MEJOR LASSO: RESULTADOS POR FOLD
# ============================================================

mejor_alpha_lasso = (
    resultados_alpha_lasso.iloc[0]["alpha"]
)

print(
    "Mejor alpha Lasso:",
    mejor_alpha_lasso
)


lasso_final_validacion = Pipeline(
    steps=[
        ("preprocesado", clone(preprocesador)),
        (
            "modelo",
            Lasso(
                alpha=mejor_alpha_lasso,
                max_iter=50000
            )
        )
    ]
)


resultados_lasso = evaluar_modelo_temporal(
    modelo=lasso_final_validacion,
    df=df_train
)

display(
    resultados_lasso.round(3)
)


print("\nRESUMEN LASSO")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_lasso["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_lasso["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_lasso["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_lasso["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_lasso["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_lasso["R2"].max(), 3)
)

In [ ]:
# ============================================================
# PASO 16.3 - SELECCIÓN DE VARIABLES DE LASSO
# ============================================================

# Entrenamos únicamente con 2017-2022.
# 2023 continúa completamente reservado.

X_train_completo = df_train[
    variables_numericas + variables_categoricas
]

y_train_completo = df_train[TARGET]


lasso_interpretacion = clone(
    lasso_final_validacion
)

lasso_interpretacion.fit(
    X_train_completo,
    y_train_completo
)


# Nombres después del preprocesamiento
nombres_features = (
    lasso_interpretacion
    .named_steps["preprocesado"]
    .get_feature_names_out()
)

coeficientes = (
    lasso_interpretacion
    .named_steps["modelo"]
    .coef_
)


coef_lasso = pd.DataFrame({
    "variable": nombres_features,
    "coeficiente": coeficientes
})

coef_lasso["coef_abs"] = (
    coef_lasso["coeficiente"].abs()
)


n_total = len(coef_lasso)

n_seleccionadas = (
    coef_lasso["coeficiente"] != 0
).sum()

n_eliminadas = (
    coef_lasso["coeficiente"] == 0
).sum()


print("Variables después de One-Hot Encoding:", n_total)

print(
    "Coeficientes distintos de cero:",
    n_seleccionadas
)

print(
    "Coeficientes eliminados por Lasso:",
    n_eliminadas
)

print(
    "Porcentaje eliminado:",
    round(
        100 * n_eliminadas / n_total,
        1
    ),
    "%"
)


print("\n20 coeficientes de mayor magnitud:")

display(
    coef_lasso[
        coef_lasso["coeficiente"] != 0
    ]
    .sort_values(
        "coef_abs",
        ascending=False
    )
    .head(20)
)

# Elastic Net

In [ ]:
# ============================================================
# PASO 17.1 - ELASTIC NET
# BÚSQUEDA DE ALPHA Y L1_RATIO
# ============================================================

from sklearn.linear_model import ElasticNet

alphas_enet = [
    0.01,
    0.1,
    1,
    5,
    10,
    25,
    50,
    100
]

l1_ratios = [
    0.1,
    0.25,
    0.5,
    0.75,
    0.9
]

resultados_enet = []

for alpha in alphas_enet:

    for l1_ratio in l1_ratios:

        modelo_enet = Pipeline(
            steps=[
                ("preprocesado", clone(preprocesador)),
                (
                    "modelo",
                    ElasticNet(
                        alpha=alpha,
                        l1_ratio=l1_ratio,
                        max_iter=50000
                    )
                )
            ]
        )

        resultados = evaluar_modelo_temporal(
            modelo=modelo_enet,
            df=df_train
        )

        resultados_enet.append({
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "MAE_medio": resultados["MAE"].mean(),
            "RMSE_medio": resultados["RMSE"].mean(),
            "R2_medio": resultados["R2"].mean(),
            "R2_std": resultados["R2"].std(),
            "R2_min": resultados["R2"].min(),
            "R2_max": resultados["R2"].max()
        })


resultados_enet_df = pd.DataFrame(
    resultados_enet
)

resultados_enet_df = (
    resultados_enet_df
    .sort_values("RMSE_medio")
    .reset_index(drop=True)
)

print("10 mejores combinaciones Elastic Net:")

display(
    resultados_enet_df
    .head(10)
    .round(3)
)

In [ ]:
# ============================================================
# PASO 17.2 - MEJOR ELASTIC NET
# ============================================================

mejor_alpha_enet = (
    resultados_enet_df.iloc[0]["alpha"]
)

mejor_l1_enet = (
    resultados_enet_df.iloc[0]["l1_ratio"]
)


print(
    "Mejor alpha Elastic Net:",
    mejor_alpha_enet
)

print(
    "Mejor l1_ratio Elastic Net:",
    mejor_l1_enet
)


elasticnet_final_validacion = Pipeline(
    steps=[
        ("preprocesado", clone(preprocesador)),
        (
            "modelo",
            ElasticNet(
                alpha=mejor_alpha_enet,
                l1_ratio=mejor_l1_enet,
                max_iter=50000
            )
        )
    ]
)


resultados_elasticnet = evaluar_modelo_temporal(
    modelo=elasticnet_final_validacion,
    df=df_train
)


display(
    resultados_elasticnet.round(3)
)


print("\nRESUMEN ELASTIC NET")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_elasticnet["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_elasticnet["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_elasticnet["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_elasticnet["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_elasticnet["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_elasticnet["R2"].max(), 3)
)

#Comparación provisional

In [ ]:
# ============================================================
# PASO 17.3 - COMPARACIÓN PROVISIONAL DE MODELOS
# ============================================================

comparacion_modelos = pd.DataFrame([

    {
        "modelo": "Baseline provincia",
        "MAE_medio": 638.557,
        "RMSE_medio": 804.780,
        "R2_medio": 0.393,
        "R2_std": 0.262
    },

    {
        "modelo": "Ridge",
        "MAE_medio": resultados_ridge["MAE"].mean(),
        "RMSE_medio": resultados_ridge["RMSE"].mean(),
        "R2_medio": resultados_ridge["R2"].mean(),
        "R2_std": resultados_ridge["R2"].std()
    },

    {
        "modelo": "Lasso",
        "MAE_medio": resultados_lasso["MAE"].mean(),
        "RMSE_medio": resultados_lasso["RMSE"].mean(),
        "R2_medio": resultados_lasso["R2"].mean(),
        "R2_std": resultados_lasso["R2"].std()
    },

    {
        "modelo": "Elastic Net",
        "MAE_medio": resultados_elasticnet["MAE"].mean(),
        "RMSE_medio": resultados_elasticnet["RMSE"].mean(),
        "R2_medio": resultados_elasticnet["R2"].mean(),
        "R2_std": resultados_elasticnet["R2"].std()
    }
])


comparacion_modelos = (
    comparacion_modelos
    .sort_values(
        "RMSE_medio"
    )
    .reset_index(drop=True)
)


display(
    comparacion_modelos.round(3)
)

# Modelización - Modelos no lineales

# Random Forest

In [ ]:
# ============================================================
# PASO 18.1 - PREPROCESAMIENTO PARA MODELOS DE ÁRBOLES
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer


pipeline_numerico_arbol = Pipeline(
    steps=[
        (
            "imputacion",
            SimpleImputer(strategy="median")
        )
    ]
)


pipeline_categorico_arbol = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


preprocesador_arbol = ColumnTransformer(
    transformers=[
        (
            "numericas",
            pipeline_numerico_arbol,
            variables_numericas
        ),
        (
            "categoricas",
            pipeline_categorico_arbol,
            variables_categoricas
        )
    ],
    remainder="drop"
)


print("Preprocesador para árboles creado correctamente.")
print("Variables numéricas:", len(variables_numericas))
print("Variables categóricas:", len(variables_categoricas))

In [ ]:
# ============================================================
# PASO 18.2 - RANDOM FOREST INICIAL
# ============================================================

from sklearn.ensemble import RandomForestRegressor


random_forest = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador_arbol)
        ),
        (
            "modelo",
            RandomForestRegressor(
                n_estimators=500,
                max_depth=None,
                min_samples_split=5,
                min_samples_leaf=2,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


resultados_rf = evaluar_modelo_temporal(
    modelo=random_forest,
    df=df_train
)


display(
    resultados_rf.round(3)
)


print("\nRESUMEN RANDOM FOREST")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_rf["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_rf["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_rf["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_rf["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_rf["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_rf["R2"].max(), 3)
)

# Comparación provisional

In [ ]:
# ============================================================
# PASO 18.3 - COMPARACIÓN CON MODELOS ANTERIORES
# ============================================================

fila_rf = pd.DataFrame([{
    "modelo": "Random Forest",
    "MAE_medio": resultados_rf["MAE"].mean(),
    "RMSE_medio": resultados_rf["RMSE"].mean(),
    "R2_medio": resultados_rf["R2"].mean(),
    "R2_std": resultados_rf["R2"].std()
}])


comparacion_modelos = pd.concat(
    [
        comparacion_modelos[
            comparacion_modelos["modelo"] != "Random Forest"
        ],
        fila_rf
    ],
    ignore_index=True
)


comparacion_modelos = (
    comparacion_modelos
    .sort_values("RMSE_medio")
    .reset_index(drop=True)
)


display(
    comparacion_modelos.round(3)
)

# Extra Trees

In [ ]:
# ============================================================
# PASO 19.1 - EXTRA TREES
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.base import clone


extra_trees = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador_arbol)
        ),
        (
            "modelo",
            ExtraTreesRegressor(
                n_estimators=500,
                max_depth=None,
                min_samples_split=5,
                min_samples_leaf=2,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


resultados_et = evaluar_modelo_temporal(
    modelo=extra_trees,
    df=df_train
)


display(
    resultados_et.round(3)
)


print("\nRESUMEN EXTRA TREES")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_et["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_et["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_et["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_et["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_et["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_et["R2"].max(), 3)
)

# Comparación provisional

In [ ]:
# ============================================================
# PASO 19.2 - COMPARACIÓN DE MODELOS
# ============================================================

fila_et = pd.DataFrame([{
    "modelo": "Extra Trees",
    "MAE_medio": resultados_et["MAE"].mean(),
    "RMSE_medio": resultados_et["RMSE"].mean(),
    "R2_medio": resultados_et["R2"].mean(),
    "R2_std": resultados_et["R2"].std()
}])


comparacion_modelos = pd.concat(
    [
        comparacion_modelos[
            comparacion_modelos["modelo"] != "Extra Trees"
        ],
        fila_et
    ],
    ignore_index=True
)


comparacion_modelos = (
    comparacion_modelos
    .sort_values("RMSE_medio")
    .reset_index(drop=True)
)


display(
    comparacion_modelos.round(3)
)

# HistGradientBoosting

In [ ]:
# ============================================================
# PASO 20.1 - HISTGRADIENTBOOSTING
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.base import clone


hist_gb = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador_arbol)
        ),
        (
            "modelo",
            HistGradientBoostingRegressor(
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=15,
                min_samples_leaf=10,
                l2_regularization=1.0,
                random_state=42
            )
        )
    ]
)


resultados_hgb = evaluar_modelo_temporal(
    modelo=hist_gb,
    df=df_train
)


display(
    resultados_hgb.round(3)
)


print("\nRESUMEN HISTGRADIENTBOOSTING")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_hgb["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_hgb["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_hgb["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_hgb["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_hgb["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_hgb["R2"].max(), 3)
)

# Comparación provisional

In [ ]:
# ============================================================
# PASO 20.2 - ACTUALIZAR COMPARACIÓN DE MODELOS
# ============================================================

fila_hgb = pd.DataFrame([{
    "modelo": "HistGradientBoosting",
    "MAE_medio": resultados_hgb["MAE"].mean(),
    "RMSE_medio": resultados_hgb["RMSE"].mean(),
    "R2_medio": resultados_hgb["R2"].mean(),
    "R2_std": resultados_hgb["R2"].std()
}])


comparacion_modelos = pd.concat(
    [
        comparacion_modelos[
            comparacion_modelos["modelo"] != "HistGradientBoosting"
        ],
        fila_hgb
    ],
    ignore_index=True
)


comparacion_modelos = (
    comparacion_modelos
    .sort_values("RMSE_medio")
    .reset_index(drop=True)
)


display(
    comparacion_modelos.round(3)
)

# Gradient Boosting Regressor

In [ ]:
# ============================================================
# PASO 21.1 - GRADIENT BOOSTING REGRESSOR
# ============================================================

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.base import clone
from sklearn.pipeline import Pipeline


gradient_boosting = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador_arbol)
        ),
        (
            "modelo",
            GradientBoostingRegressor(
                n_estimators=300,
                learning_rate=0.03,
                max_depth=2,
                min_samples_split=10,
                min_samples_leaf=5,
                subsample=0.8,
                random_state=42
            )
        )
    ]
)


resultados_gb = evaluar_modelo_temporal(
    modelo=gradient_boosting,
    df=df_train
)


display(
    resultados_gb.round(3)
)


print("\nRESUMEN GRADIENT BOOSTING")
print("-" * 50)

print("MAE medio:",
      round(resultados_gb["MAE"].mean(), 2))

print("RMSE medio:",
      round(resultados_gb["RMSE"].mean(), 2))

print("R² medio:",
      round(resultados_gb["R2"].mean(), 3))

print("R² std:",
      round(resultados_gb["R2"].std(), 3))

print("R² mínimo:",
      round(resultados_gb["R2"].min(), 3))

print("R² máximo:",
      round(resultados_gb["R2"].max(), 3))

# Comparación provisional

In [ ]:
# ============================================================
# PASO 21.2 - ACTUALIZAR COMPARACIÓN
# ============================================================

fila_gb = pd.DataFrame([{
    "modelo": "Gradient Boosting",
    "MAE_medio": resultados_gb["MAE"].mean(),
    "RMSE_medio": resultados_gb["RMSE"].mean(),
    "R2_medio": resultados_gb["R2"].mean(),
    "R2_std": resultados_gb["R2"].std()
}])


comparacion_modelos = pd.concat(
    [
        comparacion_modelos[
            comparacion_modelos["modelo"] != "Gradient Boosting"
        ],
        fila_gb
    ],
    ignore_index=True
)


comparacion_modelos = (
    comparacion_modelos
    .sort_values("RMSE_medio")
    .reset_index(drop=True)
)


display(comparacion_modelos.round(3))

# XGBoost

In [ ]:
# ============================================================
# PASO 22.1 - PREPARAR XGBOOST
# ============================================================

import xgboost as xgb

print("XGBoost version:", xgb.__version__)

In [ ]:
# ============================================================
# PASO 22.2 - XGBOOST
# ============================================================

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone


xgboost_model = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador_arbol)
        ),
        (
            "modelo",
            XGBRegressor(
                n_estimators=500,
                learning_rate=0.03,
                max_depth=3,
                min_child_weight=5,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


resultados_xgb = evaluar_modelo_temporal(
    modelo=xgboost_model,
    df=df_train
)


display(
    resultados_xgb.round(3)
)


print("\nRESUMEN XGBOOST")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_xgb["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_xgb["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_xgb["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_xgb["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_xgb["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_xgb["R2"].max(), 3)
)

# LightGBM

In [ ]:
# ============================================================
# PASO 23.1 - PREPARAR LIGHTGBM
# ============================================================

!pip -q install lightgbm

import lightgbm as lgb

print("LightGBM version:", lgb.__version__)

In [ ]:
# ============================================================
# PASO 23.2 - LIGHTGBM
# ============================================================

from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone


lightgbm_model = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador_arbol)
        ),
        (
            "modelo",
            LGBMRegressor(
                n_estimators=500,
                learning_rate=0.03,
                num_leaves=15,
                max_depth=-1,
                min_child_samples=15,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=42,
                n_jobs=-1,
                verbosity=-1
            )
        )
    ]
)


resultados_lgbm = evaluar_modelo_temporal(
    modelo=lightgbm_model,
    df=df_train
)


display(resultados_lgbm.round(3))


print("\nRESUMEN LIGHTGBM")
print("-" * 50)

print("MAE medio:",
      round(resultados_lgbm["MAE"].mean(), 2))

print("RMSE medio:",
      round(resultados_lgbm["RMSE"].mean(), 2))

print("R² medio:",
      round(resultados_lgbm["R2"].mean(), 3))

print("R² std:",
      round(resultados_lgbm["R2"].std(), 3))

print("R² mínimo:",
      round(resultados_lgbm["R2"].min(), 3))

print("R² máximo:",
      round(resultados_lgbm["R2"].max(), 3))

# CatBoost

In [ ]:
# ============================================================
# PASO 24.1 - PREPARAR CATBOOST
# ============================================================

!pip -q install catboost

from catboost import CatBoostRegressor

import catboost
print("CatBoost version:", catboost.__version__)

In [ ]:
# ============================================================
# PASO 24.2 - EVALUACIÓN TEMPORAL CATBOOST
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


def evaluar_catboost_temporal(df):

    resultados = []

    anos_validacion = [2019, 2020, 2021, 2022]

    predictores = [
        col for col in df.columns
        if col not in ["ano", "rendimiento_kg_ha"]
    ]

    categoricas = [
        "cultivo",
        "provincia_estandar"
    ]

    for fold, ano_val in enumerate(anos_validacion, start=1):

        train_fold = df[df["ano"] < ano_val].copy()
        val_fold = df[df["ano"] == ano_val].copy()

        X_train = train_fold[predictores].copy()
        y_train = train_fold["rendimiento_kg_ha"].copy()

        X_val = val_fold[predictores].copy()
        y_val = val_fold["rendimiento_kg_ha"].copy()

        # CatBoost necesita categorías sin NaN
        for col in categoricas:
            X_train[col] = X_train[col].astype(str)
            X_val[col] = X_val[col].astype(str)

        modelo = CatBoostRegressor(
            iterations=500,
            learning_rate=0.03,
            depth=5,
            loss_function="RMSE",
            l2_leaf_reg=3,
            random_seed=42,
            verbose=False,
            allow_writing_files=False
        )

        modelo.fit(
            X_train,
            y_train,
            cat_features=categoricas
        )

        pred = modelo.predict(X_val)

        mae = mean_absolute_error(y_val, pred)

        rmse = np.sqrt(
            mean_squared_error(y_val, pred)
        )

        r2 = r2_score(y_val, pred)

        resultados.append({
            "fold": fold,
            "ano_validacion": ano_val,
            "n_train": len(train_fold),
            "n_validacion": len(val_fold),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    return pd.DataFrame(resultados)

In [ ]:
# ============================================================
# PASO 24.3 - EJECUTAR CATBOOST
# ============================================================

resultados_cat = evaluar_catboost_temporal(
    df=df_train
)

display(
    resultados_cat.round(3)
)


print("\nRESUMEN CATBOOST")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_cat["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_cat["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_cat["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_cat["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_cat["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_cat["R2"].max(), 3)
)

# Support Vector Regression (SVR)

In [ ]:
# ============================================================
# PASO 25 - SUPPORT VECTOR REGRESSION (SVR)
# ============================================================

from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.base import clone


svr_model = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador)
        ),
        (
            "modelo",
            SVR(
                kernel="rbf",
                C=100,
                epsilon=0.1,
                gamma="scale"
            )
        )
    ]
)


resultados_svr = evaluar_modelo_temporal(
    modelo=svr_model,
    df=df_train
)


display(resultados_svr.round(3))


print("\nRESUMEN SVR")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_svr["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_svr["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_svr["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_svr["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_svr["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_svr["R2"].max(), 3)
)

# AdaBoost Regressor

In [ ]:
# ============================================================
# PASO 26 - ADABOOST REGRESSOR
# ============================================================

from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone


adaboost_model = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador_arbol)
        ),
        (
            "modelo",
            AdaBoostRegressor(
                estimator=DecisionTreeRegressor(
                    max_depth=3,
                    min_samples_leaf=5,
                    random_state=42
                ),
                n_estimators=300,
                learning_rate=0.03,
                loss="linear",
                random_state=42
            )
        )
    ]
)


resultados_ada = evaluar_modelo_temporal(
    modelo=adaboost_model,
    df=df_train
)


display(resultados_ada.round(3))


print("\nRESUMEN ADABOOST")
print("-" * 50)

print("MAE medio:",
      round(resultados_ada["MAE"].mean(), 2))

print("RMSE medio:",
      round(resultados_ada["RMSE"].mean(), 2))

print("R² medio:",
      round(resultados_ada["R2"].mean(), 3))

print("R² std:",
      round(resultados_ada["R2"].std(), 3))

print("R² mínimo:",
      round(resultados_ada["R2"].min(), 3))

print("R² máximo:",
      round(resultados_ada["R2"].max(), 3))

# K-Nearest Neighbors Regression

In [ ]:
# ============================================================
# PASO 27 - K-NEAREST NEIGHBORS REGRESSION
# ============================================================

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone


knn_model = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador)
        ),
        (
            "modelo",
            KNeighborsRegressor(
                n_neighbors=10,
                weights="distance",
                metric="minkowski",
                p=2
            )
        )
    ]
)


resultados_knn = evaluar_modelo_temporal(
    modelo=knn_model,
    df=df_train
)


display(resultados_knn.round(3))


print("\nRESUMEN KNN")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_knn["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_knn["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_knn["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_knn["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_knn["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_knn["R2"].max(), 3)
)

# PLS Regression

In [ ]:
# ============================================================
# PASO 28 - PARTIAL LEAST SQUARES REGRESSION (PLS)
# ============================================================

from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.base import clone


pls_model = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador)
        ),
        (
            "modelo",
            PLSRegression(
                n_components=10,
                scale=False,
                max_iter=1000
            )
        )
    ]
)


resultados_pls = evaluar_modelo_temporal(
    modelo=pls_model,
    df=df_train
)


display(resultados_pls.round(3))


print("\nRESUMEN PLS REGRESSION")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_pls["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_pls["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_pls["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_pls["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_pls["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_pls["R2"].max(), 3)
)

# Huber Regressor

In [ ]:
# ============================================================
# PASO 29 - HUBER REGRESSOR
# ============================================================

from sklearn.linear_model import HuberRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone


huber_model = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador)
        ),
        (
            "modelo",
            HuberRegressor(
                epsilon=1.35,
                alpha=1.0,
                max_iter=2000,
                tol=1e-5
            )
        )
    ]
)


resultados_huber = evaluar_modelo_temporal(
    modelo=huber_model,
    df=df_train
)


display(resultados_huber.round(3))


print("\nRESUMEN HUBER REGRESSION")
print("-" * 50)

print("MAE medio:",
      round(resultados_huber["MAE"].mean(), 2))

print("RMSE medio:",
      round(resultados_huber["RMSE"].mean(), 2))

print("R² medio:",
      round(resultados_huber["R2"].mean(), 3))

print("R² std:",
      round(resultados_huber["R2"].std(), 3))

print("R² mínimo:",
      round(resultados_huber["R2"].min(), 3))

print("R² máximo:",
      round(resultados_huber["R2"].max(), 3))

# MLPRegressor

In [ ]:
# ============================================================
# PASO 30 - MLP REGRESSOR
# Red neuronal para datos tabulares
# ============================================================

from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone


mlp_model = Pipeline(
    steps=[
        (
            "preprocesado",
            clone(preprocesador)
        ),
        (
            "modelo",
            MLPRegressor(
                hidden_layer_sizes=(64, 32),
                activation="relu",
                solver="adam",
                alpha=1.0,
                learning_rate="adaptive",
                learning_rate_init=0.001,
                max_iter=3000,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=50,
                random_state=42
            )
        )
    ]
)


resultados_mlp = evaluar_modelo_temporal(
    modelo=mlp_model,
    df=df_train
)


display(resultados_mlp.round(3))


print("\nRESUMEN MLP REGRESSOR")
print("-" * 50)

print("MAE medio:",
      round(resultados_mlp["MAE"].mean(), 2))

print("RMSE medio:",
      round(resultados_mlp["RMSE"].mean(), 2))

print("R² medio:",
      round(resultados_mlp["R2"].mean(), 3))

print("R² std:",
      round(resultados_mlp["R2"].std(), 3))

print("R² mínimo:",
      round(resultados_mlp["R2"].min(), 3))

print("R² máximo:",
      round(resultados_mlp["R2"].max(), 3))

# Comparación de modelos

In [ ]:
# ============================================================
# PASO 31 - TABLA MAESTRA DE BENCHMARKING
# ============================================================

def resumen_resultados_modelo(nombre, resultados_df):
    return {
        "modelo": nombre,
        "MAE_medio": resultados_df["MAE"].mean(),
        "MAE_std": resultados_df["MAE"].std(),
        "RMSE_medio": resultados_df["RMSE"].mean(),
        "RMSE_std": resultados_df["RMSE"].std(),
        "R2_medio": resultados_df["R2"].mean(),
        "R2_std": resultados_df["R2"].std(),
        "R2_min": resultados_df["R2"].min(),
        "R2_max": resultados_df["R2"].max()
    }


benchmark_modelos = []

# ------------------------------------------------------------
# Baseline
# ------------------------------------------------------------

benchmark_modelos.append({
    "modelo": "Baseline provincia",
    "MAE_medio": 638.557,
    "MAE_std": 189.482,
    "RMSE_medio": 804.780,
    "RMSE_std": 225.469,
    "R2_medio": 0.393,
    "R2_std": 0.262,
    "R2_min": 0.051,
    "R2_max": 0.631
})


# ------------------------------------------------------------
# Modelos evaluados
# ------------------------------------------------------------

benchmark_modelos.append(
    resumen_resultados_modelo(
        "Ridge",
        resultados_ridge
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "Lasso",
        resultados_lasso
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "Elastic Net",
        resultados_elasticnet
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "Random Forest",
        resultados_rf
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "Extra Trees",
        resultados_et
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "HistGradientBoosting",
        resultados_hgb
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "Gradient Boosting",
        resultados_gb
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "XGBoost",
        resultados_xgb
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "LightGBM",
        resultados_lgbm
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "CatBoost",
        resultados_cat
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "SVR",
        resultados_svr
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "AdaBoost",
        resultados_ada
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "KNN",
        resultados_knn
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "PLS Regression",
        resultados_pls
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "Huber",
        resultados_huber
    )
)

benchmark_modelos.append(
    resumen_resultados_modelo(
        "MLP",
        resultados_mlp
    )
)


# ------------------------------------------------------------
# Crear DataFrame
# ------------------------------------------------------------

benchmark_df = pd.DataFrame(
    benchmark_modelos
)


# ------------------------------------------------------------
# Orden principal: menor RMSE
# ------------------------------------------------------------

benchmark_df = (
    benchmark_df
    .sort_values(
        ["RMSE_medio", "R2_std"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)


print("=" * 90)
print("BENCHMARKING GLOBAL DE MODELOS")
print("=" * 90)

display(
    benchmark_df.round(3)
)

In [ ]:
# ============================================================
# PASO 31.2 - RANKING POR R2 MEDIO
# ============================================================

ranking_r2 = (
    benchmark_df
    .sort_values(
        ["R2_medio", "R2_std"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

display(
    ranking_r2[
        [
            "modelo",
            "R2_medio",
            "R2_std",
            "R2_min",
            "R2_max",
            "RMSE_medio",
            "MAE_medio"
        ]
    ].round(3)
)

In [ ]:
# ============================================================
# PASO 31.3 - MEJORA FRENTE AL BASELINE PROVINCIAL
# ============================================================

baseline_rmse = benchmark_df.loc[
    benchmark_df["modelo"] == "Baseline provincia",
    "RMSE_medio"
].iloc[0]

baseline_mae = benchmark_df.loc[
    benchmark_df["modelo"] == "Baseline provincia",
    "MAE_medio"
].iloc[0]

benchmark_df["mejora_RMSE_pct"] = (
    100 *
    (baseline_rmse - benchmark_df["RMSE_medio"])
    / baseline_rmse
)

benchmark_df["mejora_MAE_pct"] = (
    100 *
    (baseline_mae - benchmark_df["MAE_medio"])
    / baseline_mae
)

display(
    benchmark_df[
        [
            "modelo",
            "RMSE_medio",
            "mejora_RMSE_pct",
            "MAE_medio",
            "mejora_MAE_pct",
            "R2_medio",
            "R2_std"
        ]
    ].round(3)
)

In [ ]:
# ============================================================
# PASO 31.4 - COMPARACIÓN VISUAL DE MODELOS
# ============================================================

benchmark_plot = benchmark_df.sort_values(
    "R2_medio"
)

plt.figure(figsize=(10, 8))

plt.barh(
    benchmark_plot["modelo"],
    benchmark_plot["R2_medio"]
)

plt.axvline(
    0.393,
    linestyle="--",
    label="Baseline provincia"
)

plt.xlabel("R² medio - validación temporal 2019-2022")
plt.ylabel("Modelo")

plt.title(
    "Comparación global de modelos predictivos"
)

plt.legend()
plt.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASO 32 - BENCHMARK TEMPORAL COMPLETO POR AÑO
# ============================================================

resultados_por_modelo = {
    "Ridge": resultados_ridge,
    "Lasso": resultados_lasso,
    "Elastic Net": resultados_elasticnet,
    "Random Forest": resultados_rf,
    "Extra Trees": resultados_et,
    "HistGradientBoosting": resultados_hgb,
    "Gradient Boosting": resultados_gb,
    "XGBoost": resultados_xgb,
    "LightGBM": resultados_lgbm,
    "CatBoost": resultados_cat,
    "SVR": resultados_svr,
    "AdaBoost": resultados_ada,
    "KNN": resultados_knn,
    "PLS Regression": resultados_pls,
    "Huber": resultados_huber,
    "MLP": resultados_mlp
}

filas_temporales = []

for nombre, resultados in resultados_por_modelo.items():

    for _, fila in resultados.iterrows():

        filas_temporales.append({
            "modelo": nombre,
            "ano": int(fila["ano_validacion"]),
            "MAE": fila["MAE"],
            "RMSE": fila["RMSE"],
            "R2": fila["R2"]
        })

benchmark_temporal = pd.DataFrame(filas_temporales)

# Baseline provincial
baseline_temporal = resultados_baseline_df[
    resultados_baseline_df["modelo"] == "Media_por_provincia"
].copy()

for _, fila in baseline_temporal.iterrows():

    benchmark_temporal.loc[
        len(benchmark_temporal)
    ] = {
        "modelo": "Baseline provincia",
        "ano": int(fila["ano_validacion"]),
        "MAE": fila["MAE"],
        "RMSE": fila["RMSE"],
        "R2": fila["R2"]
    }


tabla_r2_anual = benchmark_temporal.pivot(
    index="modelo",
    columns="ano",
    values="R2"
)

tabla_r2_anual["R2_medio"] = tabla_r2_anual[
    [2019, 2020, 2021, 2022]
].mean(axis=1)

tabla_r2_anual["R2_std"] = tabla_r2_anual[
    [2019, 2020, 2021, 2022]
].std(axis=1)

tabla_r2_anual = tabla_r2_anual.sort_values(
    "R2_medio",
    ascending=False
)

display(tabla_r2_anual.round(3))

In [ ]:
# ============================================================
# PASO 33 - IMPACTO DE 2022 EN EL BENCHMARK
# ============================================================

impacto_2022 = []

for modelo in benchmark_temporal["modelo"].unique():

    temp = benchmark_temporal[
        benchmark_temporal["modelo"] == modelo
    ]

    r2_2019_2021 = temp[
        temp["ano"].isin([2019, 2020, 2021])
    ]["R2"].mean()

    r2_2022 = temp.loc[
        temp["ano"] == 2022,
        "R2"
    ].iloc[0]

    impacto_2022.append({
        "modelo": modelo,
        "R2_medio_2019_2021": r2_2019_2021,
        "R2_2022": r2_2022,
        "caida_R2_2022": r2_2022 - r2_2019_2021
    })

impacto_2022_df = (
    pd.DataFrame(impacto_2022)
    .sort_values("caida_R2_2022")
)

display(
    impacto_2022_df.round(3)
)

In [ ]:
# ============================================================
# PASO 34 - DIAGNÓSTICO DE DISTRIBUTION SHIFT EN 2022
# ============================================================

variables_clave_shift = [
    "rendimiento_kg_ha",
    "precipitacion_mm_campana",
    "precipitacion_mm_mar_abr",
    "precipitacion_mm_may_jun",
    "temp_media_c_mar_abr",
    "temp_media_c_may_jun",
    "NDVI_mar_apr",
    "NDVI_may_jun",
    "NDMI_mar_apr",
    "NDMI_may_jun"
]

resumen_shift = (
    df_train
    .groupby("ano")[variables_clave_shift]
    .mean()
    .round(3)
)

display(resumen_shift)

In [ ]:
# ============================================================
# PASO 34.2 - Z-SCORE DE 2022 RESPECTO AL HISTÓRICO 2017-2021
# ============================================================

historico = df_train[
    df_train["ano"] <= 2021
]

ano_2022 = df_train[
    df_train["ano"] == 2022
]

resultados_shift = []

for variable in variables_clave_shift:

    media_hist = historico[variable].mean()
    std_hist = historico[variable].std()

    media_2022 = ano_2022[variable].mean()

    z = (
        (media_2022 - media_hist) / std_hist
        if std_hist > 0
        else np.nan
    )

    resultados_shift.append({
        "variable": variable,
        "media_2017_2021": media_hist,
        "media_2022": media_2022,
        "z_shift_2022": z
    })

shift_2022_df = pd.DataFrame(
    resultados_shift
).sort_values(
    "z_shift_2022",
    key=abs,
    ascending=False
)

display(
    shift_2022_df.round(3)
)

In [ ]:
# ============================================================
# PASO 35 - ABLATION STUDY: EFECTO DE PROVINCIA
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import clone


# ------------------------------------------------------------
# 1. Predictores SIN provincia
# ------------------------------------------------------------

categoricas_sin_provincia = ["cultivo"]

numericas_sin_provincia = [
    col for col in variables_numericas
    if col != "ano"
]


# ------------------------------------------------------------
# 2. Pipeline numérico
# ------------------------------------------------------------

pipeline_numerico_sin_provincia = Pipeline(
    steps=[
        ("imputacion", SimpleImputer(strategy="median")),
        ("escalado", StandardScaler())
    ]
)


# ------------------------------------------------------------
# 3. Pipeline categórico
# ------------------------------------------------------------

pipeline_categorico_sin_provincia = Pipeline(
    steps=[
        (
            "imputacion",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# ------------------------------------------------------------
# 4. Preprocesador SIN provincia
# ------------------------------------------------------------

preprocesador_sin_provincia = ColumnTransformer(
    transformers=[
        (
            "numericas",
            pipeline_numerico_sin_provincia,
            numericas_sin_provincia
        ),
        (
            "categoricas",
            pipeline_categorico_sin_provincia,
            categoricas_sin_provincia
        )
    ]
)


print("Preprocesador SIN provincia creado.")
print(
    "Variables numéricas:",
    len(numericas_sin_provincia)
)
print(
    "Variables categóricas:",
    len(categoricas_sin_provincia)
)

# Repetir modelos sin Provincia

In [ ]:
# ============================================================
# PASO 35.2 - EVALUACIÓN SIN PROVINCIA
# Versión autocontenida
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    GradientBoostingRegressor
)

from sklearn.svm import SVR
from sklearn.linear_model import HuberRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


# ============================================================
# 1. MODELOS PARA EL ABLATION STUDY
# ============================================================

modelos_ablation = {

    "Random Forest": RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror"
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=500,
        random_state=42,
        verbosity=-1
    ),

    "CatBoost": CatBoostRegressor(
        iterations=500,
        random_seed=42,
        verbose=False
    ),

    "SVR": SVR(
        kernel="rbf",
        C=100,
        epsilon=0.1,
        gamma="scale"
    ),

    "Huber": HuberRegressor(
        epsilon=1.35,
        alpha=0.0001,
        max_iter=2000
    )
}


# ============================================================
# 2. EVALUACIÓN TEMPORAL SIN PROVINCIA
# ============================================================

resultados_sin_provincia = {}

for nombre, estimador in modelos_ablation.items():

    print("\n" + "=" * 70)
    print(nombre.upper(), "- SIN PROVINCIA")
    print("=" * 70)

    modelo_sin_provincia = Pipeline(
        steps=[
            (
                "preprocesado",
                clone(preprocesador_sin_provincia)
            ),
            (
                "modelo",
                clone(estimador)
            )
        ]
    )

    resultados = evaluar_modelo_temporal(
        modelo=modelo_sin_provincia,
        df=df_train
    )

    resultados_sin_provincia[nombre] = resultados

    print(
        f"R² medio = {resultados['R2'].mean():.3f} | "
        f"RMSE medio = {resultados['RMSE'].mean():.2f} | "
        f"MAE medio = {resultados['MAE'].mean():.2f}"
    )


print("\n" + "=" * 70)
print("EVALUACIÓN SIN PROVINCIA FINALIZADA")
print("=" * 70)

# Ablation por fuentes de datos

In [ ]:
# ============================================================
# PASO 36.1 - DEFINICIÓN DE BLOQUES DE VARIABLES POR FUENTE
# ============================================================

bloques_features = {

    "ERA5": [
        "temp_media_c_ene_feb",
        "temp_media_c_jul_ago",
        "temp_media_c_mar_abr",
        "temp_media_c_may_jun",
        "temp_media_c_sep_dic",

        "temp_rocio_c_ene_feb",
        "temp_rocio_c_jul_ago",
        "temp_rocio_c_mar_abr",
        "temp_rocio_c_may_jun",
        "temp_rocio_c_sep_dic",

        "precipitacion_mm_ene_feb",
        "precipitacion_mm_jul_ago",
        "precipitacion_mm_mar_abr",
        "precipitacion_mm_may_jun",
        "precipitacion_mm_sep_dic",

        "radiacion_mj_m2_ene_feb",
        "radiacion_mj_m2_jul_ago",
        "radiacion_mj_m2_mar_abr",
        "radiacion_mj_m2_may_jun",
        "radiacion_mj_m2_sep_dic",

        "temp_media_c_campana",
        "temp_rocio_c_campana",
        "precipitacion_mm_campana",
        "radiacion_mj_m2_campana"
    ],

    "SoilGrids": [
        "clay_pct_0_30cm",
        "sand_pct_0_30cm",
        "soc_g_kg_0_30cm",
        "ph_0_30cm",
        "bdod_g_cm3_0_30cm"
    ],

    "Sentinel-2": [
        "NDVI_jan_feb",
        "NDVI_jul",
        "NDVI_mar_apr",
        "NDVI_may_jun",
        "NDVI_nov_dec",

        "EVI_jan_feb",
        "EVI_jul",
        "EVI_mar_apr",
        "EVI_may_jun",
        "EVI_nov_dec",

        "NDMI_jan_feb",
        "NDMI_jul",
        "NDMI_mar_apr",
        "NDMI_may_jun",
        "NDMI_nov_dec"
    ],

    "EuroCrops": [
        "n_recintos",
        "superficie_eurocrops_ha",
        "superficie_media_recinto_ha"
    ]
}


print("=" * 70)
print("BLOQUES DE VARIABLES")
print("=" * 70)

for bloque, variables in bloques_features.items():
    print(
        f"{bloque:12s}: {len(variables):2d} variables"
    )

print(
    "\nTotal:",
    sum(len(v) for v in bloques_features.values())
)

In [ ]:
# ============================================================
# PASO 36.2 - ESCENARIOS DE FUSIÓN DE DATOS
# ============================================================

escenarios_fusion = {

    # Fuentes individuales
    "ERA5":
        bloques_features["ERA5"],

    "SoilGrids":
        bloques_features["SoilGrids"],

    "Sentinel-2":
        bloques_features["Sentinel-2"],

    "EuroCrops":
        bloques_features["EuroCrops"],


    # Fusiones principales
    "ERA5 + Sentinel-2":
        bloques_features["ERA5"]
        + bloques_features["Sentinel-2"],

    "ERA5 + SoilGrids":
        bloques_features["ERA5"]
        + bloques_features["SoilGrids"],

    "Sentinel-2 + SoilGrids":
        bloques_features["Sentinel-2"]
        + bloques_features["SoilGrids"],


    # Fusión biofísica principal
    "ERA5 + Sentinel-2 + SoilGrids":
        bloques_features["ERA5"]
        + bloques_features["Sentinel-2"]
        + bloques_features["SoilGrids"],


    # Todas las fuentes
    "Todas":
        bloques_features["ERA5"]
        + bloques_features["Sentinel-2"]
        + bloques_features["SoilGrids"]
        + bloques_features["EuroCrops"]
}


for nombre, variables in escenarios_fusion.items():

    print(
        f"{nombre:40s}: "
        f"{len(variables):2d} variables"
    )

In [ ]:
# ============================================================
# PASO 36.3 - PREPROCESADOR DINÁMICO POR ESCENARIO
# ============================================================

def crear_preprocesador_fusion(variables):

    pipeline_numerico = Pipeline(
        steps=[
            (
                "imputacion",
                SimpleImputer(strategy="median")
            ),
            (
                "escalado",
                StandardScaler()
            )
        ]
    )

    pipeline_categorico = Pipeline(
        steps=[
            (
                "imputacion",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]
    )

    preprocesador = ColumnTransformer(
        transformers=[
            (
                "numericas",
                pipeline_numerico,
                variables
            ),
            (
                "cultivo",
                pipeline_categorico,
                ["cultivo"]
            )
        ]
    )

    return preprocesador

In [ ]:
# ============================================================
# PASO 36.4 - DATA FUSION CON EXTRA TREES
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor

resultados_fusion_et = []

resultados_fusion_et_detalle = {}


for escenario, variables in escenarios_fusion.items():

    print("\n" + "=" * 75)
    print(escenario.upper())
    print("=" * 75)

    preprocesador_escenario = crear_preprocesador_fusion(
        variables
    )

    modelo = Pipeline(
        steps=[
            (
                "preprocesado",
                preprocesador_escenario
            ),
            (
                "modelo",
                ExtraTreesRegressor(
                    n_estimators=500,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]
    )

    resultados = evaluar_modelo_temporal(
        modelo=modelo,
        df=df_train
    )

    resultados_fusion_et_detalle[
        escenario
    ] = resultados

    resultados_fusion_et.append({
        "escenario": escenario,
        "n_variables": len(variables),

        "MAE_medio":
            resultados["MAE"].mean(),

        "RMSE_medio":
            resultados["RMSE"].mean(),

        "R2_medio":
            resultados["R2"].mean(),

        "R2_std":
            resultados["R2"].std(),

        "R2_min":
            resultados["R2"].min(),

        "R2_2022":
            resultados.loc[
                resultados["ano_validacion"] == 2022,
                "R2"
            ].iloc[0]
    })


fusion_et_df = (
    pd.DataFrame(resultados_fusion_et)
    .sort_values(
        "R2_medio",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 90)
print("RESULTADOS DATA FUSION - EXTRA TREES")
print("=" * 90)

display(
    fusion_et_df.round(3)
)

# Preparar validación temporal para optimización de hiperparámetros

In [ ]:
# ============================================================
# PASO 37 - PREPARAR CV TEMPORAL PARA HYPERPARAMETER TUNING
# Todas las variables se mantienen
# 2023 permanece completamente bloqueado
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Comprobar dataset utilizado para tuning
# ------------------------------------------------------------

print("=" * 70)
print("DATASET DISPONIBLE PARA TUNING")
print("=" * 70)

print("Años disponibles:")
print(sorted(df_train["ano"].unique()))

print("\nNúmero de observaciones:")
print(len(df_train))

print("\nDistribución por cultivo:")
print(df_train["cultivo"].value_counts())


# ------------------------------------------------------------
# 2. Variables del modelo
# ------------------------------------------------------------

variables_modelo = (
    variables_numericas
    + variables_categoricas
)

print("\n" + "=" * 70)
print("VARIABLES DEL MODELO")
print("=" * 70)

print("Variables numéricas:", len(variables_numericas))
print("Variables categóricas:", len(variables_categoricas))
print("Total variables originales:", len(variables_modelo))

print("\nCategóricas:")
print(variables_categoricas)


# ------------------------------------------------------------
# 3. X e y para tuning
# ------------------------------------------------------------

X_tuning = df_train[
    variables_modelo
].copy()

y_tuning = df_train[
    TARGET
].copy()


# ------------------------------------------------------------
# 4. Crear folds temporales EXPANDING WINDOW
# ------------------------------------------------------------

config_folds_tuning = [
    ([2017, 2018], 2019),
    ([2017, 2018, 2019], 2020),
    ([2017, 2018, 2019, 2020], 2021),
    ([2017, 2018, 2019, 2020, 2021], 2022)
]

cv_temporal = []

resumen_cv = []

for fold, (anos_train, ano_val) in enumerate(
    config_folds_tuning,
    start=1
):

    idx_train = np.where(
        df_train["ano"].isin(anos_train)
    )[0]

    idx_val = np.where(
        df_train["ano"] == ano_val
    )[0]

    cv_temporal.append(
        (idx_train, idx_val)
    )

    resumen_cv.append({
        "fold": fold,
        "train_desde": min(anos_train),
        "train_hasta": max(anos_train),
        "ano_validacion": ano_val,
        "n_train": len(idx_train),
        "n_validacion": len(idx_val)
    })


resumen_cv_df = pd.DataFrame(
    resumen_cv
)

print("\n" + "=" * 70)
print("FOLDS TEMPORALES PARA TUNING")
print("=" * 70)

display(resumen_cv_df)


# ------------------------------------------------------------
# 5. Controles anti-leakage
# ------------------------------------------------------------

for fold, (idx_train, idx_val) in enumerate(
    cv_temporal,
    start=1
):

    max_train = df_train.iloc[
        idx_train
    ]["ano"].max()

    min_val = df_train.iloc[
        idx_val
    ]["ano"].min()

    assert max_train < min_val

    print(
        f"Fold {fold}: "
        f"TRAIN hasta {max_train} -> "
        f"VALIDACIÓN {min_val} | OK"
    )


# ------------------------------------------------------------
# 6. Confirmar que 2023 NO aparece
# ------------------------------------------------------------

assert 2023 not in df_train["ano"].unique()

print("\n2023 fuera del tuning: OK")
print(
    "Observaciones reservadas para TEST FINAL:",
    len(df_test)
)

print("\nValidación temporal preparada correctamente.")

# Tuning de Random Forest

In [ ]:
# ============================================================
# PASO 38 - HYPERPARAMETER TUNING: RANDOM FOREST
# Validación temporal expanding-window
# 2023 NO se utiliza
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. Preprocesador específico para Random Forest
# ------------------------------------------------------------

preprocessor_rf_tuning = ColumnTransformer(
    transformers=[
        (
            "numericas",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                )
            ]),
            variables_numericas
        ),
        (
            "categoricas",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            variables_categoricas
        )
    ]
)

# ------------------------------------------------------------
# 2. Pipeline
# ------------------------------------------------------------

pipeline_rf = Pipeline([
    (
        "preprocessor",
        preprocessor_rf_tuning
    ),
    (
        "modelo",
        RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        )
    )
])

# ------------------------------------------------------------
# 3. Espacio de hiperparámetros
# ------------------------------------------------------------

parametros_rf = {

    "modelo__n_estimators": [
        300, 500, 800, 1200
    ],

    "modelo__max_depth": [
        None, 5, 8, 12, 16, 20
    ],

    "modelo__min_samples_split": [
        2, 4, 6, 10
    ],

    "modelo__min_samples_leaf": [
        1, 2, 3, 5, 8
    ],

    "modelo__max_features": [
        0.4, 0.6, 0.8, 1.0, "sqrt"
    ],

    "modelo__bootstrap": [
        True, False
    ]
}

# ------------------------------------------------------------
# 4. Randomized Search
# ------------------------------------------------------------

busqueda_rf = RandomizedSearchCV(

    estimator=pipeline_rf,

    param_distributions=parametros_rf,

    n_iter=60,

    scoring="neg_root_mean_squared_error",

    cv=cv_temporal,

    random_state=42,

    n_jobs=-1,

    verbose=1,

    return_train_score=True
)

print("=" * 70)
print("INICIANDO TUNING RANDOM FOREST")
print("=" * 70)

busqueda_rf.fit(
    X_tuning,
    y_tuning
)

print("\nTuning finalizado.")

# ------------------------------------------------------------
# 5. Mejores hiperparámetros
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MEJORES HIPERPARÁMETROS")
print("=" * 70)

for parametro, valor in busqueda_rf.best_params_.items():
    print(f"{parametro}: {valor}")

print(
    "\nRMSE CV usado para selección:",
    round(-busqueda_rf.best_score_, 2)
)

# ------------------------------------------------------------
# 6. Evaluación fold por fold del mejor modelo
# ------------------------------------------------------------

mejores_parametros_rf = busqueda_rf.best_params_

resultados_rf_tuned = []

for fold, (idx_train, idx_val) in enumerate(
    cv_temporal,
    start=1
):

    X_train_fold = X_tuning.iloc[idx_train]
    y_train_fold = y_tuning.iloc[idx_train]

    X_val_fold = X_tuning.iloc[idx_val]
    y_val_fold = y_tuning.iloc[idx_val]

    modelo_fold = Pipeline([
        (
            "preprocessor",
            preprocessor_rf_tuning
        ),
        (
            "modelo",
            RandomForestRegressor(
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    modelo_fold.set_params(
        **mejores_parametros_rf
    )

    modelo_fold.fit(
        X_train_fold,
        y_train_fold
    )

    pred = modelo_fold.predict(
        X_val_fold
    )

    mae = mean_absolute_error(
        y_val_fold,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            pred
        )
    )

    r2 = r2_score(
        y_val_fold,
        pred
    )

    ano_validacion = (
        df_train
        .iloc[idx_val]["ano"]
        .unique()[0]
    )

    resultados_rf_tuned.append({
        "fold": fold,
        "ano_validacion": ano_validacion,
        "n_train": len(idx_train),
        "n_validacion": len(idx_val),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })


resultados_rf_tuned_df = pd.DataFrame(
    resultados_rf_tuned
)

print("\n" + "=" * 70)
print("RESULTADOS RANDOM FOREST OPTIMIZADO")
print("=" * 70)

display(
    resultados_rf_tuned_df.round(3)
)

# ------------------------------------------------------------
# 7. Resumen
# ------------------------------------------------------------

print("\nRESUMEN RANDOM FOREST OPTIMIZADO")
print("-" * 50)

print(
    "MAE medio:",
    round(
        resultados_rf_tuned_df["MAE"].mean(),
        2
    )
)

print(
    "RMSE medio:",
    round(
        resultados_rf_tuned_df["RMSE"].mean(),
        2
    )
)

print(
    "R² medio:",
    round(
        resultados_rf_tuned_df["R2"].mean(),
        3
    )
)

print(
    "R² std:",
    round(
        resultados_rf_tuned_df["R2"].std(),
        3
    )
)

print(
    "R² mínimo:",
    round(
        resultados_rf_tuned_df["R2"].min(),
        3
    )
)

print(
    "R² máximo:",
    round(
        resultados_rf_tuned_df["R2"].max(),
        3
    )
)

# ------------------------------------------------------------
# 8. Guardar mejor configuración
# ------------------------------------------------------------

rf_tuned = busqueda_rf.best_estimator_

print("\nModelo guardado como: rf_tuned")

# Tuning de Extra Trees

In [ ]:
# ============================================================
# PASO 39 - HYPERPARAMETER TUNING: EXTRA TREES
# Validación temporal expanding-window
# 2023 permanece BLOQUEADO
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. Pipeline Extra Trees
# Utilizamos el mismo preprocesamiento definido para árboles
# ------------------------------------------------------------

pipeline_et = Pipeline([
    (
        "preprocessor",
        preprocessor_rf_tuning
    ),
    (
        "modelo",
        ExtraTreesRegressor(
            random_state=42,
            n_jobs=-1
        )
    )
])


# ------------------------------------------------------------
# 2. Espacio de hiperparámetros
# ------------------------------------------------------------

parametros_et = {

    "modelo__n_estimators": [
        300, 500, 800
    ],

    "modelo__max_depth": [
        None, 8, 12, 16, 20, 25
    ],

    "modelo__min_samples_split": [
        2, 4, 6, 10
    ],

    "modelo__min_samples_leaf": [
        1, 2, 3, 5
    ],

    "modelo__max_features": [
        0.4, 0.6, 0.8, 1.0, "sqrt"
    ],

    "modelo__bootstrap": [
        False, True
    ]
}


# ------------------------------------------------------------
# 3. Randomized Search
# ------------------------------------------------------------

busqueda_et = RandomizedSearchCV(

    estimator=pipeline_et,

    param_distributions=parametros_et,

    n_iter=40,

    scoring="neg_root_mean_squared_error",

    cv=cv_temporal,

    random_state=42,

    n_jobs=-1,

    verbose=1,

    return_train_score=True
)


print("=" * 70)
print("INICIANDO TUNING EXTRA TREES")
print("=" * 70)

busqueda_et.fit(
    X_tuning,
    y_tuning
)

print("\nTuning finalizado.")


# ------------------------------------------------------------
# 4. Mejores hiperparámetros
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MEJORES HIPERPARÁMETROS EXTRA TREES")
print("=" * 70)

for parametro, valor in busqueda_et.best_params_.items():
    print(f"{parametro}: {valor}")

print(
    "\nRMSE CV usado para selección:",
    round(-busqueda_et.best_score_, 2)
)


# ------------------------------------------------------------
# 5. Evaluación temporal fold por fold
# ------------------------------------------------------------

mejores_parametros_et = busqueda_et.best_params_

resultados_et_tuned = []

for fold, (idx_train, idx_val) in enumerate(
    cv_temporal,
    start=1
):

    X_train_fold = X_tuning.iloc[idx_train]
    y_train_fold = y_tuning.iloc[idx_train]

    X_val_fold = X_tuning.iloc[idx_val]
    y_val_fold = y_tuning.iloc[idx_val]

    modelo_fold = Pipeline([
        (
            "preprocessor",
            preprocessor_rf_tuning
        ),
        (
            "modelo",
            ExtraTreesRegressor(
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    modelo_fold.set_params(
        **mejores_parametros_et
    )

    modelo_fold.fit(
        X_train_fold,
        y_train_fold
    )

    pred = modelo_fold.predict(
        X_val_fold
    )

    mae = mean_absolute_error(
        y_val_fold,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            pred
        )
    )

    r2 = r2_score(
        y_val_fold,
        pred
    )

    ano_validacion = (
        df_train
        .iloc[idx_val]["ano"]
        .unique()[0]
    )

    resultados_et_tuned.append({
        "fold": fold,
        "ano_validacion": ano_validacion,
        "n_train": len(idx_train),
        "n_validacion": len(idx_val),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })


resultados_et_tuned_df = pd.DataFrame(
    resultados_et_tuned
)


# ------------------------------------------------------------
# 6. Resultados
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESULTADOS EXTRA TREES OPTIMIZADO")
print("=" * 70)

display(
    resultados_et_tuned_df.round(3)
)


# ------------------------------------------------------------
# 7. Resumen
# ------------------------------------------------------------

print("\nRESUMEN EXTRA TREES OPTIMIZADO")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_et_tuned_df["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_et_tuned_df["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_et_tuned_df["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_et_tuned_df["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_et_tuned_df["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_et_tuned_df["R2"].max(), 3)
)


# ------------------------------------------------------------
# 8. Guardar mejor modelo
# ------------------------------------------------------------

et_tuned = busqueda_et.best_estimator_

print("\nModelo guardado como: et_tuned")

# Tuning de CatBoost

In [ ]:
# ============================================================
# PASO 40 - HYPERPARAMETER TUNING: CATBOOST
# Validación temporal expanding-window
# 2023 permanece BLOQUEADO
# ============================================================

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. Pipeline
# ------------------------------------------------------------

pipeline_cat = Pipeline([
    (
        "preprocessor",
        preprocessor_rf_tuning
    ),
    (
        "modelo",
        CatBoostRegressor(
            random_seed=42,
            verbose=0,
            allow_writing_files=False
        )
    )
])


# ------------------------------------------------------------
# 2. Espacio de hiperparámetros
# ------------------------------------------------------------

parametros_cat = {

    "modelo__iterations": [
        300, 500, 800, 1200
    ],

    "modelo__depth": [
        4, 5, 6, 7, 8
    ],

    "modelo__learning_rate": [
        0.01, 0.03, 0.05, 0.08, 0.1
    ],

    "modelo__l2_leaf_reg": [
        1, 3, 5, 10, 20
    ],

    "modelo__random_strength": [
        0, 0.5, 1, 2, 5
    ],

    "modelo__subsample": [
        0.7, 0.8, 0.9, 1.0
    ]
}


# ------------------------------------------------------------
# 3. Randomized Search
# ------------------------------------------------------------

busqueda_cat = RandomizedSearchCV(

    estimator=pipeline_cat,

    param_distributions=parametros_cat,

    n_iter=35,

    scoring="neg_root_mean_squared_error",

    cv=cv_temporal,

    random_state=42,

    n_jobs=-1,

    verbose=1,

    return_train_score=True
)


print("=" * 70)
print("INICIANDO TUNING CATBOOST")
print("=" * 70)

busqueda_cat.fit(
    X_tuning,
    y_tuning
)

print("\nTuning finalizado.")


# ------------------------------------------------------------
# 4. Mejores hiperparámetros
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MEJORES HIPERPARÁMETROS CATBOOST")
print("=" * 70)

for parametro, valor in busqueda_cat.best_params_.items():
    print(f"{parametro}: {valor}")

print(
    "\nRMSE CV usado para selección:",
    round(-busqueda_cat.best_score_, 2)
)


# ------------------------------------------------------------
# 5. Evaluación temporal fold por fold
# ------------------------------------------------------------

mejores_parametros_cat = busqueda_cat.best_params_

resultados_cat_tuned = []

for fold, (idx_train, idx_val) in enumerate(
    cv_temporal,
    start=1
):

    X_train_fold = X_tuning.iloc[idx_train]
    y_train_fold = y_tuning.iloc[idx_train]

    X_val_fold = X_tuning.iloc[idx_val]
    y_val_fold = y_tuning.iloc[idx_val]

    modelo_fold = Pipeline([
        (
            "preprocessor",
            preprocessor_rf_tuning
        ),
        (
            "modelo",
            CatBoostRegressor(
                random_seed=42,
                verbose=0,
                allow_writing_files=False
            )
        )
    ])

    modelo_fold.set_params(
        **mejores_parametros_cat
    )

    modelo_fold.fit(
        X_train_fold,
        y_train_fold
    )

    pred = modelo_fold.predict(
        X_val_fold
    )

    mae = mean_absolute_error(
        y_val_fold,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            pred
        )
    )

    r2 = r2_score(
        y_val_fold,
        pred
    )

    ano_validacion = (
        df_train
        .iloc[idx_val]["ano"]
        .unique()[0]
    )

    resultados_cat_tuned.append({
        "fold": fold,
        "ano_validacion": ano_validacion,
        "n_train": len(idx_train),
        "n_validacion": len(idx_val),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })


resultados_cat_tuned_df = pd.DataFrame(
    resultados_cat_tuned
)


# ------------------------------------------------------------
# 6. Mostrar resultados
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESULTADOS CATBOOST OPTIMIZADO")
print("=" * 70)

display(
    resultados_cat_tuned_df.round(3)
)


# ------------------------------------------------------------
# 7. Resumen
# ------------------------------------------------------------

print("\nRESUMEN CATBOOST OPTIMIZADO")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_cat_tuned_df["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_cat_tuned_df["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_cat_tuned_df["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_cat_tuned_df["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_cat_tuned_df["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_cat_tuned_df["R2"].max(), 3)
)


# ------------------------------------------------------------
# 8. Guardar modelo
# ------------------------------------------------------------

cat_tuned = busqueda_cat.best_estimator_

print("\nModelo guardado como: cat_tuned")

# Tuning de XGBoost

In [ ]:
# ============================================================
# PASO 41 - HYPERPARAMETER TUNING: XGBOOST
# Validación temporal expanding-window
# 2023 permanece BLOQUEADO
# ============================================================

import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. Pipeline
# ------------------------------------------------------------

pipeline_xgb = Pipeline([
    (
        "preprocessor",
        preprocessor_rf_tuning
    ),
    (
        "modelo",
        XGBRegressor(
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )
    )
])


# ------------------------------------------------------------
# 2. Espacio de hiperparámetros
# ------------------------------------------------------------

parametros_xgb = {

    "modelo__n_estimators": [
        200, 400, 600, 800
    ],

    "modelo__max_depth": [
        2, 3, 4, 5, 6
    ],

    "modelo__learning_rate": [
        0.01, 0.03, 0.05, 0.08, 0.1
    ],

    "modelo__subsample": [
        0.6, 0.7, 0.8, 0.9, 1.0
    ],

    "modelo__colsample_bytree": [
        0.5, 0.6, 0.7, 0.8, 1.0
    ],

    "modelo__min_child_weight": [
        1, 3, 5, 10
    ],

    "modelo__reg_alpha": [
        0, 0.01, 0.1, 1
    ],

    "modelo__reg_lambda": [
        1, 3, 5, 10
    ]
}


# ------------------------------------------------------------
# 3. Randomized Search
# ------------------------------------------------------------

busqueda_xgb = RandomizedSearchCV(

    estimator=pipeline_xgb,

    param_distributions=parametros_xgb,

    n_iter=40,

    scoring="neg_root_mean_squared_error",

    cv=cv_temporal,

    random_state=42,

    n_jobs=-1,

    verbose=1,

    return_train_score=True
)


print("=" * 70)
print("INICIANDO TUNING XGBOOST")
print("=" * 70)

busqueda_xgb.fit(
    X_tuning,
    y_tuning
)

print("\nTuning finalizado.")


# ------------------------------------------------------------
# 4. Mejores hiperparámetros
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MEJORES HIPERPARÁMETROS XGBOOST")
print("=" * 70)

for parametro, valor in busqueda_xgb.best_params_.items():
    print(f"{parametro}: {valor}")

print(
    "\nRMSE CV usado para selección:",
    round(-busqueda_xgb.best_score_, 2)
)


# ------------------------------------------------------------
# 5. Evaluación temporal fold por fold
# ------------------------------------------------------------

mejores_parametros_xgb = busqueda_xgb.best_params_

resultados_xgb_tuned = []

for fold, (idx_train, idx_val) in enumerate(
    cv_temporal,
    start=1
):

    X_train_fold = X_tuning.iloc[idx_train]
    y_train_fold = y_tuning.iloc[idx_train]

    X_val_fold = X_tuning.iloc[idx_val]
    y_val_fold = y_tuning.iloc[idx_val]

    modelo_fold = Pipeline([
        (
            "preprocessor",
            preprocessor_rf_tuning
        ),
        (
            "modelo",
            XGBRegressor(
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1,
                verbosity=0
            )
        )
    ])

    modelo_fold.set_params(
        **mejores_parametros_xgb
    )

    modelo_fold.fit(
        X_train_fold,
        y_train_fold
    )

    pred = modelo_fold.predict(
        X_val_fold
    )

    resultados_xgb_tuned.append({

        "fold": fold,

        "ano_validacion":
            df_train.iloc[idx_val]["ano"].unique()[0],

        "n_train": len(idx_train),

        "n_validacion": len(idx_val),

        "MAE":
            mean_absolute_error(y_val_fold, pred),

        "RMSE":
            np.sqrt(
                mean_squared_error(y_val_fold, pred)
            ),

        "R2":
            r2_score(y_val_fold, pred)
    })


resultados_xgb_tuned_df = pd.DataFrame(
    resultados_xgb_tuned
)


# ------------------------------------------------------------
# 6. Mostrar resultados
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESULTADOS XGBOOST OPTIMIZADO")
print("=" * 70)

display(
    resultados_xgb_tuned_df.round(3)
)


# ------------------------------------------------------------
# 7. Resumen
# ------------------------------------------------------------

print("\nRESUMEN XGBOOST OPTIMIZADO")
print("-" * 50)

print(
    "MAE medio:",
    round(resultados_xgb_tuned_df["MAE"].mean(), 2)
)

print(
    "RMSE medio:",
    round(resultados_xgb_tuned_df["RMSE"].mean(), 2)
)

print(
    "R² medio:",
    round(resultados_xgb_tuned_df["R2"].mean(), 3)
)

print(
    "R² std:",
    round(resultados_xgb_tuned_df["R2"].std(), 3)
)

print(
    "R² mínimo:",
    round(resultados_xgb_tuned_df["R2"].min(), 3)
)

print(
    "R² máximo:",
    round(resultados_xgb_tuned_df["R2"].max(), 3)
)


# ------------------------------------------------------------
# 8. Guardar modelo
# ------------------------------------------------------------

xgb_tuned = busqueda_xgb.best_estimator_

print("\nModelo guardado como: xgb_tuned")

# Tuning de LightGBM

In [ ]:
# ============================================================
# PASO 42 - HYPERPARAMETER TUNING: LIGHTGBM
# Validación temporal expanding-window
# 2023 permanece BLOQUEADO
# ============================================================

import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor

from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. Pipeline
# ------------------------------------------------------------

pipeline_lgbm = Pipeline([
    (
        "preprocessor",
        preprocessor_rf_tuning
    ),
    (
        "modelo",
        LGBMRegressor(
            objective="regression",
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        )
    )
])


# ------------------------------------------------------------
# 2. Espacio de hiperparámetros
# ------------------------------------------------------------

parametros_lgbm = {

    "modelo__n_estimators": [
        200, 400, 600, 800
    ],

    "modelo__learning_rate": [
        0.01, 0.03, 0.05, 0.08, 0.1
    ],

    "modelo__num_leaves": [
        7, 15, 31, 50
    ],

    "modelo__max_depth": [
        -1, 3, 5, 7, 10
    ],

    "modelo__min_child_samples": [
        10, 20, 30, 40, 50
    ],

    "modelo__subsample": [
        0.6, 0.7, 0.8, 0.9, 1.0
    ],

    "modelo__colsample_bytree": [
        0.5, 0.6, 0.7, 0.8, 1.0
    ],

    "modelo__reg_alpha": [
        0, 0.01, 0.1, 1
    ],

    "modelo__reg_lambda": [
        0, 1, 3, 5, 10
    ]
}


# ------------------------------------------------------------
# 3. Randomized Search
# ------------------------------------------------------------

busqueda_lgbm = RandomizedSearchCV(

    estimator=pipeline_lgbm,

    param_distributions=parametros_lgbm,

    n_iter=40,

    scoring="neg_root_mean_squared_error",

    cv=cv_temporal,

    random_state=42,

    n_jobs=-1,

    verbose=1,

    return_train_score=True
)


print("=" * 70)
print("INICIANDO TUNING LIGHTGBM")
print("=" * 70)

busqueda_lgbm.fit(
    X_tuning,
    y_tuning
)

print("\nTuning finalizado.")


# ------------------------------------------------------------
# 4. Mejores hiperparámetros
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MEJORES HIPERPARÁMETROS LIGHTGBM")
print("=" * 70)

for parametro, valor in busqueda_lgbm.best_params_.items():
    print(f"{parametro}: {valor}")

print(
    "\nRMSE CV usado para selección:",
    round(-busqueda_lgbm.best_score_, 2)
)


# ------------------------------------------------------------
# 5. Evaluación fold por fold
# ------------------------------------------------------------

mejores_parametros_lgbm = busqueda_lgbm.best_params_

resultados_lgbm_tuned = []

for fold, (idx_train, idx_val) in enumerate(
    cv_temporal,
    start=1
):

    X_train_fold = X_tuning.iloc[idx_train]
    y_train_fold = y_tuning.iloc[idx_train]

    X_val_fold = X_tuning.iloc[idx_val]
    y_val_fold = y_tuning.iloc[idx_val]

    modelo_fold = Pipeline([
        (
            "preprocessor",
            preprocessor_rf_tuning
        ),
        (
            "modelo",
            LGBMRegressor(
                objective="regression",
                random_state=42,
                n_jobs=-1,
                verbosity=-1
            )
        )
    ])

    modelo_fold.set_params(
        **mejores_parametros_lgbm
    )

    modelo_fold.fit(
        X_train_fold,
        y_train_fold
    )

    pred = modelo_fold.predict(
        X_val_fold
    )

    resultados_lgbm_tuned.append({

        "fold": fold,

        "ano_validacion":
            df_train.iloc[idx_val]["ano"].unique()[0],

        "n_train": len(idx_train),

        "n_validacion": len(idx_val),

        "MAE":
            mean_absolute_error(
                y_val_fold,
                pred
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y_val_fold,
                    pred
                )
            ),

        "R2":
            r2_score(
                y_val_fold,
                pred
            )
    })


resultados_lgbm_tuned_df = pd.DataFrame(
    resultados_lgbm_tuned
)


# ------------------------------------------------------------
# 6. Mostrar resultados
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESULTADOS LIGHTGBM OPTIMIZADO")
print("=" * 70)

display(
    resultados_lgbm_tuned_df.round(3)
)


# ------------------------------------------------------------
# 7. Resumen
# ------------------------------------------------------------

print("\nRESUMEN LIGHTGBM OPTIMIZADO")
print("-" * 50)

print(
    "MAE medio:",
    round(
        resultados_lgbm_tuned_df["MAE"].mean(),
        2
    )
)

print(
    "RMSE medio:",
    round(
        resultados_lgbm_tuned_df["RMSE"].mean(),
        2
    )
)

print(
    "R² medio:",
    round(
        resultados_lgbm_tuned_df["R2"].mean(),
        3
    )
)

print(
    "R² std:",
    round(
        resultados_lgbm_tuned_df["R2"].std(),
        3
    )
)

print(
    "R² mínimo:",
    round(
        resultados_lgbm_tuned_df["R2"].min(),
        3
    )
)

print(
    "R² máximo:",
    round(
        resultados_lgbm_tuned_df["R2"].max(),
        3
    )
)


# ------------------------------------------------------------
# 8. Guardar modelo
# ------------------------------------------------------------

lgbm_tuned = busqueda_lgbm.best_estimator_

print("\nModelo guardado como: lgbm_tuned")

# Comparar modelos optimizados

In [ ]:
# ============================================================
# PASO 43 - COMPARACIÓN DE MODELOS OPTIMIZADOS
# Validación temporal 2019-2022
# 2023 NO SE UTILIZA
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Reunir resultados de los modelos optimizados
# ------------------------------------------------------------

modelos_tuned = {
    "Random Forest": resultados_rf_tuned_df,
    "Extra Trees": resultados_et_tuned_df,
    "CatBoost": resultados_cat_tuned_df,
    "XGBoost": resultados_xgb_tuned_df,
    "LightGBM": resultados_lgbm_tuned_df
}


# ------------------------------------------------------------
# 2. Crear tabla resumen
# ------------------------------------------------------------

resumen_tuned = []

for nombre, resultados in modelos_tuned.items():

    resumen_tuned.append({

        "Modelo": nombre,

        "MAE_medio":
            resultados["MAE"].mean(),

        "RMSE_medio":
            resultados["RMSE"].mean(),

        "R2_medio":
            resultados["R2"].mean(),

        "R2_std":
            resultados["R2"].std(),

        "R2_min":
            resultados["R2"].min(),

        "R2_max":
            resultados["R2"].max()
    })


resumen_tuned_df = pd.DataFrame(resumen_tuned)

resumen_tuned_df = resumen_tuned_df.sort_values(
    by="R2_medio",
    ascending=False
).reset_index(drop=True)


print("=" * 80)
print("COMPARACIÓN GLOBAL - MODELOS OPTIMIZADOS")
print("=" * 80)

display(
    resumen_tuned_df.round(3)
)


# ------------------------------------------------------------
# 3. Ranking por diferentes criterios
# ------------------------------------------------------------

ranking = resumen_tuned_df.copy()

ranking["Rank_R2"] = ranking["R2_medio"].rank(
    ascending=False,
    method="min"
)

ranking["Rank_RMSE"] = ranking["RMSE_medio"].rank(
    ascending=True,
    method="min"
)

ranking["Rank_MAE"] = ranking["MAE_medio"].rank(
    ascending=True,
    method="min"
)

ranking["Rank_estabilidad"] = ranking["R2_std"].rank(
    ascending=True,
    method="min"
)

ranking["Rank_R2_min"] = ranking["R2_min"].rank(
    ascending=False,
    method="min"
)

ranking["Rank_medio"] = ranking[
    [
        "Rank_R2",
        "Rank_RMSE",
        "Rank_MAE",
        "Rank_estabilidad",
        "Rank_R2_min"
    ]
].mean(axis=1)


ranking = ranking.sort_values(
    "Rank_medio"
).reset_index(drop=True)


print("\n" + "=" * 80)
print("RANKING MULTICRITERIO")
print("=" * 80)

display(
    ranking[
        [
            "Modelo",
            "Rank_R2",
            "Rank_RMSE",
            "Rank_MAE",
            "Rank_estabilidad",
            "Rank_R2_min",
            "Rank_medio"
        ]
    ].round(2)
)


# ------------------------------------------------------------
# 4. Comparación R² por año
# ------------------------------------------------------------

r2_por_ano = pd.DataFrame()

for nombre, resultados in modelos_tuned.items():

    temp = resultados[
        ["ano_validacion", "R2"]
    ].copy()

    temp = temp.rename(
        columns={"R2": nombre}
    )

    if r2_por_ano.empty:
        r2_por_ano = temp
    else:
        r2_por_ano = r2_por_ano.merge(
            temp,
            on="ano_validacion",
            how="outer"
        )


r2_por_ano = r2_por_ano.sort_values(
    "ano_validacion"
).reset_index(drop=True)


print("\n" + "=" * 80)
print("R² POR AÑO DE VALIDACIÓN")
print("=" * 80)

display(
    r2_por_ano.round(3)
)


# ------------------------------------------------------------
# 5. RMSE por año
# ------------------------------------------------------------

rmse_por_ano = pd.DataFrame()

for nombre, resultados in modelos_tuned.items():

    temp = resultados[
        ["ano_validacion", "RMSE"]
    ].copy()

    temp = temp.rename(
        columns={"RMSE": nombre}
    )

    if rmse_por_ano.empty:
        rmse_por_ano = temp
    else:
        rmse_por_ano = rmse_por_ano.merge(
            temp,
            on="ano_validacion",
            how="outer"
        )


rmse_por_ano = rmse_por_ano.sort_values(
    "ano_validacion"
).reset_index(drop=True)


print("\n" + "=" * 80)
print("RMSE POR AÑO DE VALIDACIÓN")
print("=" * 80)

display(
    rmse_por_ano.round(2)
)


# ------------------------------------------------------------
# 6. Gráfico R² por año
# ------------------------------------------------------------

plt.figure(figsize=(11, 6))

for modelo in modelos_tuned.keys():

    plt.plot(
        r2_por_ano["ano_validacion"],
        r2_por_ano[modelo],
        marker="o",
        label=modelo
    )

plt.axhline(
    y=0,
    linestyle="--",
    linewidth=1
)

plt.xlabel("Año de validación")
plt.ylabel("R²")
plt.title(
    "Generalización temporal de los modelos optimizados"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 7. Gráfico RMSE medio
# ------------------------------------------------------------

orden_rmse = resumen_tuned_df.sort_values(
    "RMSE_medio"
)

plt.figure(figsize=(9, 5))

plt.bar(
    orden_rmse["Modelo"],
    orden_rmse["RMSE_medio"]
)

plt.ylabel("RMSE medio (kg/ha)")
plt.title(
    "RMSE medio - Validación temporal 2019-2022"
)

plt.xticks(
    rotation=30,
    ha="right"
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 8. Gráfico R² medio
# ------------------------------------------------------------

orden_r2 = resumen_tuned_df.sort_values(
    "R2_medio",
    ascending=False
)

plt.figure(figsize=(9, 5))

plt.bar(
    orden_r2["Modelo"],
    orden_r2["R2_medio"]
)

plt.ylabel("R² medio")
plt.title(
    "R² medio - Validación temporal 2019-2022"
)

plt.xticks(
    rotation=30,
    ha="right"
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 9. Identificar mejor modelo por año
# ------------------------------------------------------------

mejor_por_ano = []

for _, fila in r2_por_ano.iterrows():

    valores = fila.drop(
        labels="ano_validacion"
    )

    mejor_modelo = valores.idxmax()

    mejor_por_ano.append({

        "ano_validacion":
            int(fila["ano_validacion"]),

        "mejor_modelo":
            mejor_modelo,

        "R2":
            valores.max()
    })


mejor_por_ano_df = pd.DataFrame(
    mejor_por_ano
)


print("\n" + "=" * 80)
print("MEJOR MODELO EN CADA AÑO")
print("=" * 80)

display(
    mejor_por_ano_df.round(3)
)


# ------------------------------------------------------------
# 10. Comprobación explícita
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CONTROL METODOLÓGICO")
print("=" * 80)

print(
    "Años utilizados en esta comparación:",
    sorted(r2_por_ano["ano_validacion"].unique())
)

print(
    "2023 utilizado:",
    2023 in r2_por_ano["ano_validacion"].values
)

print(
    "\nTEST 2023 continúa BLOQUEADO."
)

# Reducción controlada de redundancia

In [ ]:
# ============================================================
# PASO 44 - DEFINICIÓN DE CONJUNTO REDUCIDO DE VARIABLES
# VERSIÓN AUTOCONTENIDA
# NO modifica df
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Definir columnas que NO son predictores
# ------------------------------------------------------------

target = "rendimiento_kg_ha"

columnas_excluir = [
    target,
    "ano"
]


# ------------------------------------------------------------
# 2. Reconstruir predictores directamente desde df
# ------------------------------------------------------------

categoricas = [
    "cultivo",
    "provincia_estandar"
]

# Comprobar que existen
categoricas = [
    c for c in categoricas
    if c in df.columns
]

# Todas las columnas predictoras excepto categóricas
variables_completas = [
    c for c in df.columns
    if c not in columnas_excluir + categoricas
    and pd.api.types.is_numeric_dtype(df[c])
]


print("=" * 70)
print("CONJUNTO ORIGINAL")
print("=" * 70)

print("Variables numéricas:", len(variables_completas))
print("Variables categóricas:", len(categoricas))
print(
    "Total predictores:",
    len(variables_completas) + len(categoricas)
)


# ------------------------------------------------------------
# 3. Variables redundantes candidatas a eliminar
# ------------------------------------------------------------

# Sentinel-2:
# Eliminamos EVI porque presenta correlación muy elevada con NDVI
# en todas las ventanas temporales.

eliminar_sentinel = [
    "EVI_nov_dec",
    "EVI_jan_feb",
    "EVI_mar_apr",
    "EVI_may_jun",
    "EVI_jul"
]


# ERA5:
# Eliminamos agregados de campaña porque conservamos
# las ventanas temporales más detalladas.

eliminar_era5_campana = [
    "temp_media_c_campana",
    "temp_rocio_c_campana",
    "precipitacion_mm_campana",
    "radiacion_mj_m2_campana"
]


eliminar_redundancia = (
    eliminar_sentinel +
    eliminar_era5_campana
)


# ------------------------------------------------------------
# 4. Comprobar existencia antes de eliminar
# ------------------------------------------------------------

encontradas = [
    v for v in eliminar_redundancia
    if v in variables_completas
]

no_encontradas = [
    v for v in eliminar_redundancia
    if v not in variables_completas
]


print("\n" + "=" * 70)
print("VARIABLES CANDIDATAS A ELIMINAR")
print("=" * 70)

for v in encontradas:
    print("OK  ", v)

if no_encontradas:
    print("\nNo encontradas:")
    for v in no_encontradas:
        print("   ", v)


# ------------------------------------------------------------
# 5. Crear conjunto reducido
# ------------------------------------------------------------

variables_reducidas = [
    v for v in variables_completas
    if v not in encontradas
]


print("\n" + "=" * 70)
print("CONJUNTO REDUCIDO")
print("=" * 70)

print("Variables numéricas:", len(variables_reducidas))
print("Variables categóricas:", len(categoricas))

print(
    "Total predictores:",
    len(variables_reducidas) + len(categoricas)
)

print(
    "Variables eliminadas:",
    len(variables_completas) - len(variables_reducidas)
)

print(
    "Reducción de variables numéricas:",
    round(
        100 *
        (len(variables_completas) - len(variables_reducidas))
        / len(variables_completas),
        1
    ),
    "%"
)


# ------------------------------------------------------------
# 6. Mostrar variables eliminadas
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VARIABLES ELIMINADAS")
print("=" * 70)

for v in encontradas:
    print("-", v)


# ------------------------------------------------------------
# 7. Mostrar variables conservadas
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VARIABLES CONSERVADAS")
print("=" * 70)

for i, v in enumerate(variables_reducidas, 1):
    print(f"{i:02d}. {v}")


# ------------------------------------------------------------
# 8. Controles
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONTROL METODOLÓGICO")
print("=" * 70)

print("df modificado: NO")
print("Target incluido como predictor:",
      target in variables_reducidas)

print("Año incluido como predictor:",
      "ano" in variables_reducidas)

print(
    "2023 usado para decidir la reducción: NO"
)

# Random Forest optimizado: 49 vs. 40 predictores

In [ ]:
# ============================================================
# PASO 45
# RANDOM FOREST OPTIMIZADO:
# CONJUNTO COMPLETO (49) vs REDUCIDO (40)
#
# Misma arquitectura
# Mismos hiperparámetros
# Mismos folds temporales
# 2023 NO SE UTILIZA
# ============================================================

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ------------------------------------------------------------
# 1. Configuración
# ------------------------------------------------------------

target = "rendimiento_kg_ha"

# Hiperparámetros obtenidos previamente en el tuning de RF
rf_params = {
    "n_estimators": 300,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "max_depth": 20,
    "bootstrap": False,
    "random_state": 42,
    "n_jobs": -1
}


# ------------------------------------------------------------
# 2. Función de evaluación temporal
# ------------------------------------------------------------

def evaluar_rf_variables(
    df,
    variables_numericas,
    variables_categoricas,
    nombre_escenario
):

    predictores = (
        variables_numericas +
        variables_categoricas
    )

    resultados = []

    # Validación expanding-window:
    # 2017-18 -> 2019
    # 2017-19 -> 2020
    # 2017-20 -> 2021
    # 2017-21 -> 2022

    folds = [
        ([2017, 2018], 2019),
        ([2017, 2018, 2019], 2020),
        ([2017, 2018, 2019, 2020], 2021),
        ([2017, 2018, 2019, 2020, 2021], 2022)
    ]

    for fold, (anos_train, ano_val) in enumerate(
        folds,
        start=1
    ):

        train = df[
            df["ano"].isin(anos_train)
        ].copy()

        val = df[
            df["ano"] == ano_val
        ].copy()

        X_train = train[predictores]
        y_train = train[target]

        X_val = val[predictores]
        y_val = val[target]


        # ----------------------------------------------------
        # Preprocesamiento
        # ----------------------------------------------------

        preprocessor = ColumnTransformer(
            transformers=[

                (
                    "num",
                    Pipeline([
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="median"
                            )
                        )
                    ]),
                    variables_numericas
                ),

                (
                    "cat",
                    Pipeline([
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="most_frequent"
                            )
                        ),
                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="ignore"
                            )
                        )
                    ]),
                    variables_categoricas
                )
            ]
        )


        # ----------------------------------------------------
        # Modelo
        # ----------------------------------------------------

        modelo = Pipeline([
            (
                "preprocesador",
                preprocessor
            ),
            (
                "modelo",
                RandomForestRegressor(
                    **rf_params
                )
            )
        ])


        modelo.fit(
            X_train,
            y_train
        )

        pred = modelo.predict(
            X_val
        )


        # ----------------------------------------------------
        # Métricas
        # ----------------------------------------------------

        mae = mean_absolute_error(
            y_val,
            pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                pred
            )
        )

        r2 = r2_score(
            y_val,
            pred
        )


        resultados.append({

            "escenario":
                nombre_escenario,

            "fold":
                fold,

            "ano_validacion":
                ano_val,

            "n_train":
                len(train),

            "n_validacion":
                len(val),

            "n_numericas":
                len(variables_numericas),

            "n_predictores_originales":
                len(predictores),

            "MAE":
                mae,

            "RMSE":
                rmse,

            "R2":
                r2
        })


    return pd.DataFrame(
        resultados
    )


# ------------------------------------------------------------
# 3. Evaluar conjunto COMPLETO
# ------------------------------------------------------------

resultados_rf_completo = evaluar_rf_variables(
    df=df,
    variables_numericas=variables_completas,
    variables_categoricas=categoricas,
    nombre_escenario="Completo_49"
)


# ------------------------------------------------------------
# 4. Evaluar conjunto REDUCIDO
# ------------------------------------------------------------

resultados_rf_reducido = evaluar_rf_variables(
    df=df,
    variables_numericas=variables_reducidas,
    variables_categoricas=categoricas,
    nombre_escenario="Reducido_40"
)


# ------------------------------------------------------------
# 5. Unir resultados
# ------------------------------------------------------------

comparacion_rf_variables = pd.concat(
    [
        resultados_rf_completo,
        resultados_rf_reducido
    ],
    ignore_index=True
)


print("=" * 80)
print("RESULTADOS POR FOLD")
print("=" * 80)

display(
    comparacion_rf_variables.round(3)
)


# ------------------------------------------------------------
# 6. Resumen global
# ------------------------------------------------------------

resumen_rf_variables = (
    comparacion_rf_variables
    .groupby("escenario")
    .agg(
        n_predictores=(
            "n_predictores_originales",
            "first"
        ),
        MAE_medio=(
            "MAE",
            "mean"
        ),
        RMSE_medio=(
            "RMSE",
            "mean"
        ),
        R2_medio=(
            "R2",
            "mean"
        ),
        R2_std=(
            "R2",
            "std"
        ),
        R2_min=(
            "R2",
            "min"
        ),
        R2_max=(
            "R2",
            "max"
        )
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("COMPARACIÓN GLOBAL")
print("=" * 80)

display(
    resumen_rf_variables.round(3)
)


# ------------------------------------------------------------
# 7. Comparación directa por año
# ------------------------------------------------------------

comparacion_anos = (
    comparacion_rf_variables
    .pivot(
        index="ano_validacion",
        columns="escenario",
        values=["MAE", "RMSE", "R2"]
    )
)


print("\n" + "=" * 80)
print("COMPARACIÓN POR AÑO")
print("=" * 80)

display(
    comparacion_anos.round(3)
)


# ------------------------------------------------------------
# 8. Diferencias reducido - completo
# ------------------------------------------------------------

completo = (
    resultados_rf_completo
    .set_index("ano_validacion")
)

reducido = (
    resultados_rf_reducido
    .set_index("ano_validacion")
)


diferencias = pd.DataFrame({

    "delta_MAE":
        reducido["MAE"] -
        completo["MAE"],

    "delta_RMSE":
        reducido["RMSE"] -
        completo["RMSE"],

    "delta_R2":
        reducido["R2"] -
        completo["R2"]
})


print("\n" + "=" * 80)
print("EFECTO DE LA REDUCCIÓN")
print("Reducido - Completo")
print("=" * 80)

display(
    diferencias.round(3)
)


# ------------------------------------------------------------
# 9. Resumen del efecto
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RESUMEN DEL EFECTO DE REDUCIR VARIABLES")
print("=" * 80)

print(
    "Δ MAE medio:",
    round(
        resultados_rf_reducido["MAE"].mean()
        -
        resultados_rf_completo["MAE"].mean(),
        2
    ),
    "kg/ha"
)

print(
    "Δ RMSE medio:",
    round(
        resultados_rf_reducido["RMSE"].mean()
        -
        resultados_rf_completo["RMSE"].mean(),
        2
    ),
    "kg/ha"
)

print(
    "Δ R² medio:",
    round(
        resultados_rf_reducido["R2"].mean()
        -
        resultados_rf_completo["R2"].mean(),
        3
    )
)


# ------------------------------------------------------------
# 10. Control metodológico
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CONTROL METODOLÓGICO")
print("=" * 80)

anos_evaluados = sorted(
    comparacion_rf_variables[
        "ano_validacion"
    ].unique()
)

print(
    "Años evaluados:",
    anos_evaluados
)

print(
    "2023 utilizado:",
    2023 in anos_evaluados
)

print(
    "Mismos hiperparámetros RF:",
    True
)

print(
    "Dataset df modificado:",
    False
)

print(
    "\nTEST FINAL 2023 CONTINÚA BLOQUEADO."
)

# Importancia interna de variables en Random Forest optimizado

In [ ]:
# ============================================================
# PASO 46 - FEATURE IMPORTANCE
# RANDOM FOREST OPTIMIZADO
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Entrenar rf_tuned con TODO el desarrollo 2017-2022
# ------------------------------------------------------------

X_train_total = df_train[
    variables_modelo
].copy()

y_train_total = df_train[
    TARGET
].copy()


rf_interpretacion = rf_tuned

rf_interpretacion.fit(
    X_train_total,
    y_train_total
)


# ------------------------------------------------------------
# 2. Recuperar nombres después del preprocesamiento
# ------------------------------------------------------------

preprocessor_fit = rf_interpretacion.named_steps[
    "preprocessor"
]

nombres_features = (
    preprocessor_fit
    .get_feature_names_out()
)


# ------------------------------------------------------------
# 3. Recuperar importancias
# ------------------------------------------------------------

importancias = (
    rf_interpretacion
    .named_steps["modelo"]
    .feature_importances_
)


# ------------------------------------------------------------
# 4. Crear tabla
# ------------------------------------------------------------

feature_importance_rf = pd.DataFrame({
    "variable": nombres_features,
    "importancia": importancias
})


feature_importance_rf["importancia_pct"] = (
    100 *
    feature_importance_rf["importancia"]
)


feature_importance_rf = (
    feature_importance_rf
    .sort_values(
        "importancia",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 80)
print("IMPORTANCIA DE VARIABLES - RANDOM FOREST OPTIMIZADO")
print("=" * 80)

display(
    feature_importance_rf
    .head(30)
    .round(4)
)

In [ ]:
# ============================================================
# PASO 46.2 - LIMPIAR NOMBRES DE FEATURES
# ============================================================

feature_importance_rf["variable_limpia"] = (
    feature_importance_rf["variable"]
    .str.replace(
        "numericas__",
        "",
        regex=False
    )
    .str.replace(
        "categoricas__",
        "",
        regex=False
    )
    .str.replace(
        "preprocessor__",
        "",
        regex=False
    )
)


display(
    feature_importance_rf[
        [
            "variable_limpia",
            "importancia",
            "importancia_pct"
        ]
    ]
    .head(30)
    .round(4)
)

In [ ]:
# ============================================================
# PASO 46.3 - TOP 20 VARIABLES
# ============================================================

top20_rf = (
    feature_importance_rf
    .head(20)
    .sort_values(
        "importancia",
        ascending=True
    )
)


plt.figure(
    figsize=(10, 8)
)

plt.barh(
    top20_rf["variable_limpia"],
    top20_rf["importancia"]
)

plt.xlabel(
    "Importancia relativa"
)

plt.ylabel(
    "Variable"
)

plt.title(
    "Top 20 variables - Random Forest optimizado"
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# PASO 46.4 - IMPORTANCIA ACUMULADA
# ============================================================

feature_importance_rf[
    "importancia_acumulada"
] = (
    feature_importance_rf[
        "importancia"
    ].cumsum()
)


print("=" * 70)
print("IMPORTANCIA ACUMULADA")
print("=" * 70)

display(
    feature_importance_rf[
        [
            "variable_limpia",
            "importancia",
            "importancia_acumulada"
        ]
    ]
    .head(30)
    .round(4)
)


# Número de variables necesarias para alcanzar:
# 50 %, 75 %, 90 % y 95 %

for umbral in [
    0.50,
    0.75,
    0.90,
    0.95
]:

    n = (
        feature_importance_rf[
            "importancia_acumulada"
        ] < umbral
    ).sum() + 1

    print(
        f"Variables para alcanzar "
        f"{int(umbral*100)} % "
        f"de la importancia: {n}"
    )

# Permutation Importance temporal

In [ ]:
# ============================================================
# PASO 47.1
# PERMUTATION IMPORTANCE OUT-OF-SAMPLE
# RANDOM FOREST OPTIMIZADO
# VALIDACIÓN TEMPORAL 2019-2022
# ============================================================

import pandas as pd
import numpy as np

from sklearn.inspection import permutation_importance
from sklearn.base import clone


# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------

resultados_perm = []

anos_validacion = [2019, 2020, 2021, 2022]


# ------------------------------------------------------------
# Evaluación temporal
# ------------------------------------------------------------

for ano_val in anos_validacion:

    print("=" * 80)
    print(f"VALIDACIÓN: {ano_val}")
    print("=" * 80)

    # -----------------------------------------
    # TRAIN = todos los años anteriores
    # VALIDACIÓN = año correspondiente
    # -----------------------------------------

    train_fold = df_train[
        df_train["ano"] < ano_val
    ].copy()

    val_fold = df_train[
        df_train["ano"] == ano_val
    ].copy()


    X_train_fold = train_fold[
        variables_modelo
    ].copy()

    y_train_fold = train_fold[
        TARGET
    ].copy()

    X_val_fold = val_fold[
        variables_modelo
    ].copy()

    y_val_fold = val_fold[
        TARGET
    ].copy()


    # -----------------------------------------
    # Clonar RF optimizado
    # -----------------------------------------

    modelo_fold = clone(rf_tuned)

    modelo_fold.fit(
        X_train_fold,
        y_train_fold
    )


    # -----------------------------------------
    # Permutation Importance
    #
    # scoring = neg_root_mean_squared_error
    #
    # Si al permutar una variable aumenta mucho
    # el RMSE, esa variable es importante.
    # -----------------------------------------

    perm = permutation_importance(
        modelo_fold,
        X_val_fold,
        y_val_fold,
        scoring="neg_root_mean_squared_error",
        n_repeats=30,
        random_state=42,
        n_jobs=-1
    )


    # -----------------------------------------
    # Guardar resultados
    # -----------------------------------------

    resultado_fold = pd.DataFrame({
        "variable": variables_modelo,
        "importancia": perm.importances_mean,
        "importancia_std": perm.importances_std,
        "ano_validacion": ano_val
    })

    resultados_perm.append(
        resultado_fold
    )


    # Mostrar TOP 10 del año
    print("\nTop 10 variables:")

    display(
        resultado_fold
        .sort_values(
            "importancia",
            ascending=False
        )
        .head(10)
        .round(3)
    )


# ------------------------------------------------------------
# Unir resultados
# ------------------------------------------------------------

permutation_temporal = pd.concat(
    resultados_perm,
    ignore_index=True
)


print("\n" + "=" * 80)
print("PERMUTATION IMPORTANCE TEMPORAL FINALIZADA")
print("=" * 80)

print(
    "Años evaluados:",
    sorted(
        permutation_temporal[
            "ano_validacion"
        ].unique()
    )
)

print(
    "2023 utilizado:",
    2023 in permutation_temporal[
        "ano_validacion"
    ].unique()
)

In [ ]:
# ============================================================
# PASO 47.2
# RESUMEN TEMPORAL DE PERMUTATION IMPORTANCE
# ============================================================

resumen_permutation = (
    permutation_temporal
    .groupby("variable")
    .agg(
        importancia_media=(
            "importancia",
            "mean"
        ),
        importancia_std_temporal=(
            "importancia",
            "std"
        ),
        importancia_min=(
            "importancia",
            "min"
        ),
        importancia_max=(
            "importancia",
            "max"
        ),
        n_anos_positiva=(
            "importancia",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)


resumen_permutation = (
    resumen_permutation
    .sort_values(
        "importancia_media",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 80)
print("PERMUTATION IMPORTANCE - RESUMEN TEMPORAL")
print("=" * 80)

display(
    resumen_permutation
    .head(30)
    .round(3)
)

In [ ]:
# ============================================================
# PASO 47.3
# TOP 20 - PERMUTATION IMPORTANCE
# ============================================================

import matplotlib.pyplot as plt


top20_perm = (
    resumen_permutation
    .head(20)
    .sort_values(
        "importancia_media",
        ascending=True
    )
)


plt.figure(
    figsize=(10, 8)
)

plt.barh(
    top20_perm["variable"],
    top20_perm["importancia_media"]
)

plt.xlabel(
    "Incremento medio del RMSE al permutar la variable (kg/ha)"
)

plt.ylabel(
    "Variable"
)

plt.title(
    "Permutation Importance temporal - Random Forest optimizado"
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# PASO 47.4
# PERMUTATION IMPORTANCE POR AÑO
# ============================================================

top15_variables = (
    resumen_permutation
    .head(15)["variable"]
    .tolist()
)


tabla_perm_anual = (
    permutation_temporal[
        permutation_temporal[
            "variable"
        ].isin(top15_variables)
    ]
    .pivot(
        index="variable",
        columns="ano_validacion",
        values="importancia"
    )
)


# Ordenar según importancia media global

tabla_perm_anual = (
    tabla_perm_anual
    .reindex(top15_variables)
)


print("=" * 80)
print("TOP 15 - IMPORTANCIA POR AÑO DE VALIDACIÓN")
print("=" * 80)

display(
    tabla_perm_anual.round(2)
)

# SHAP del Random Forest

In [ ]:
# ============================================================
# PASO 48.1
# PREPARACIÓN SHAP
# RANDOM FOREST OPTIMIZADO
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Instalar SHAP si fuera necesario
try:
    import shap
except ImportError:
    !pip -q install shap
    import shap


# ------------------------------------------------------------
# Dataset de desarrollo
# SOLO 2017-2022
# ------------------------------------------------------------

df_shap = df_train.copy()

print("=" * 70)
print("CONTROL DATASET SHAP")
print("=" * 70)

print("Años:",
      sorted(df_shap["ano"].unique()))

print("Observaciones:",
      len(df_shap))

print(
    "2023 utilizado:",
    2023 in df_shap["ano"].unique()
)


# ------------------------------------------------------------
# Predictores / target
# ------------------------------------------------------------

X_shap_original = df_shap[
    variables_modelo
].copy()

y_shap = df_shap[
    TARGET
].copy()


# ------------------------------------------------------------
# Entrenar una copia del RF optimizado
# ------------------------------------------------------------

from sklearn.base import clone

rf_shap = clone(rf_tuned)

rf_shap.fit(
    X_shap_original,
    y_shap
)

print("\nModelo RF preparado.")

In [ ]:
# ============================================================
# PASO 48.2
# TRANSFORMACIÓN DE VARIABLES PARA SHAP
# ============================================================

preprocessor_shap = (
    rf_shap
    .named_steps["preprocessor"]
)

modelo_shap = (
    rf_shap
    .named_steps["modelo"]
)


# Transformar X
X_shap_trans = (
    preprocessor_shap
    .transform(X_shap_original)
)


# Convertir sparse -> dense si fuera necesario
if hasattr(
    X_shap_trans,
    "toarray"
):
    X_shap_trans = (
        X_shap_trans.toarray()
    )


# Recuperar nombres
feature_names_shap = (
    preprocessor_shap
    .get_feature_names_out()
)


# Limpiar nombres
feature_names_shap = [
    x.replace("numericas__", "")
     .replace("categoricas__", "")
     .replace("preprocessor__", "")
    for x in feature_names_shap
]


# DataFrame
X_shap_df = pd.DataFrame(
    X_shap_trans,
    columns=feature_names_shap,
    index=X_shap_original.index
)


print("=" * 70)
print("MATRIZ SHAP")
print("=" * 70)

print(
    "Observaciones:",
    X_shap_df.shape[0]
)

print(
    "Variables tras transformación:",
    X_shap_df.shape[1]
)

display(
    X_shap_df.head()
)

In [ ]:
# ============================================================
# PASO 48.3
# CÁLCULO SHAP VALUES
# ============================================================

explainer = shap.TreeExplainer(
    modelo_shap
)

shap_values = explainer.shap_values(
    X_shap_df
)


print("=" * 70)
print("SHAP CALCULADO")
print("=" * 70)

print(
    "Shape SHAP:",
    np.array(shap_values).shape
)

In [ ]:
# ============================================================
# PASO 48.4
# SHAP SUMMARY PLOT
# ============================================================

shap.summary_plot(
    shap_values,
    X_shap_df,
    max_display=20,
    show=True
)

In [ ]:
# ============================================================
# PASO 48.5
# RANKING GLOBAL SHAP
# ============================================================

mean_abs_shap = np.abs(
    shap_values
).mean(axis=0)


ranking_shap = pd.DataFrame({
    "variable": X_shap_df.columns,
    "mean_abs_SHAP": mean_abs_shap
})


ranking_shap = (
    ranking_shap
    .sort_values(
        "mean_abs_SHAP",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 70)
print("TOP 30 VARIABLES SEGÚN SHAP")
print("=" * 70)

display(
    ranking_shap
    .head(30)
    .round(3)
)

# Ensembles de los mejores modelos
# Random Forest + XGBoost
# Random Forest + CatBoost
# Random Forest + XGBoost + CatBoost

In [ ]:
# ============================================================
# PASO 49 - ENSEMBLES DE LOS MEJORES MODELOS
# Validación temporal 2019-2022
# 2023 permanece BLOQUEADO
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


modelos_ensemble = {
    "RF": rf_tuned,
    "XGB": xgb_tuned,
    "CAT": cat_tuned
}


resultados_ensemble = []
predicciones_folds = []


# ------------------------------------------------------------
# 1. Evaluación fold por fold
# ------------------------------------------------------------

for fold, (idx_train, idx_val) in enumerate(
    cv_temporal,
    start=1
):

    X_train_fold = X_tuning.iloc[idx_train]
    y_train_fold = y_tuning.iloc[idx_train]

    X_val_fold = X_tuning.iloc[idx_val]
    y_val_fold = y_tuning.iloc[idx_val]

    ano_val = int(
        df_train.iloc[idx_val]["ano"].unique()[0]
    )

    pred_fold = {}

    # --------------------------------------------------------
    # Modelos individuales
    # --------------------------------------------------------

    for nombre, modelo in modelos_ensemble.items():

        m = clone(modelo)

        m.fit(
            X_train_fold,
            y_train_fold
        )

        pred_fold[nombre] = m.predict(
            X_val_fold
        )


    # --------------------------------------------------------
    # Ensembles
    # --------------------------------------------------------

    pred_fold["RF_XGB"] = (
        pred_fold["RF"] +
        pred_fold["XGB"]
    ) / 2

    pred_fold["RF_CAT"] = (
        pred_fold["RF"] +
        pred_fold["CAT"]
    ) / 2

    pred_fold["RF_XGB_CAT"] = (
        pred_fold["RF"] +
        pred_fold["XGB"] +
        pred_fold["CAT"]
    ) / 3


    # --------------------------------------------------------
    # Guardar métricas
    # --------------------------------------------------------

    for nombre in [
        "RF",
        "XGB",
        "CAT",
        "RF_XGB",
        "RF_CAT",
        "RF_XGB_CAT"
    ]:

        pred = pred_fold[nombre]

        resultados_ensemble.append({

            "modelo": nombre,

            "fold": fold,

            "ano_validacion": ano_val,

            "MAE":
                mean_absolute_error(
                    y_val_fold,
                    pred
                ),

            "RMSE":
                np.sqrt(
                    mean_squared_error(
                        y_val_fold,
                        pred
                    )
                ),

            "R2":
                r2_score(
                    y_val_fold,
                    pred
                )
        })


# ------------------------------------------------------------
# 2. DataFrame completo
# ------------------------------------------------------------

resultados_ensemble_df = pd.DataFrame(
    resultados_ensemble
)


# ------------------------------------------------------------
# 3. Resumen global
# ------------------------------------------------------------

resumen_ensemble = (
    resultados_ensemble_df
    .groupby("modelo")
    .agg(
        MAE_medio=("MAE", "mean"),
        RMSE_medio=("RMSE", "mean"),
        R2_medio=("R2", "mean"),
        R2_std=("R2", "std"),
        R2_min=("R2", "min"),
        R2_max=("R2", "max")
    )
    .reset_index()
    .sort_values(
        "RMSE_medio"
    )
)


print("=" * 80)
print("COMPARACIÓN MODELOS INDIVIDUALES Y ENSEMBLES")
print("=" * 80)

display(
    resumen_ensemble.round(3)
)


# ------------------------------------------------------------
# 4. R² por año
# ------------------------------------------------------------

r2_ensemble_anual = (
    resultados_ensemble_df
    .pivot(
        index="ano_validacion",
        columns="modelo",
        values="R2"
    )
)


print("\n" + "=" * 80)
print("R² POR AÑO")
print("=" * 80)

display(
    r2_ensemble_anual.round(3)
)


# ------------------------------------------------------------
# 5. RMSE por año
# ------------------------------------------------------------

rmse_ensemble_anual = (
    resultados_ensemble_df
    .pivot(
        index="ano_validacion",
        columns="modelo",
        values="RMSE"
    )
)


print("\n" + "=" * 80)
print("RMSE POR AÑO")
print("=" * 80)

display(
    rmse_ensemble_anual.round(2)
)


# ------------------------------------------------------------
# 6. Control metodológico
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CONTROL METODOLÓGICO")
print("=" * 80)

print(
    "Años evaluados:",
    sorted(
        resultados_ensemble_df[
            "ano_validacion"
        ].unique()
    )
)

print(
    "2023 utilizado:",
    2023 in resultados_ensemble_df[
        "ano_validacion"
    ].unique()
)

print(
    "\nTEST FINAL 2023 CONTINÚA BLOQUEADO."
)

# Test final independiente 2023

In [ ]:
# ============================================================
# PASO 50 - EVALUACIÓN FINAL EN TEST 2023
# ============================================================
#
# Entrenamiento definitivo: 2017-2022
# Test externo:             2023
#
# IMPORTANTE:
# - No se ajustan hiperparámetros.
# - No se seleccionan variables.
# - No se modifica el modelo según el resultado de 2023.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ------------------------------------------------------------
# 1. Separación FINAL train/test
# ------------------------------------------------------------

mask_train_final = df["ano"] <= 2022
mask_test_final  = df["ano"] == 2023


df_train_final = df.loc[mask_train_final].copy()
df_test_final  = df.loc[mask_test_final].copy()


print("=" * 80)
print("SEPARACIÓN FINAL TRAIN / TEST")
print("=" * 80)

print(
    "Años TRAIN:",
    sorted(df_train_final["ano"].unique())
)

print(
    "Años TEST:",
    sorted(df_test_final["ano"].unique())
)

print(
    "\nObservaciones TRAIN:",
    len(df_train_final)
)

print(
    "Observaciones TEST:",
    len(df_test_final)
)


# ------------------------------------------------------------
# 2. Variables predictoras
# ------------------------------------------------------------
#
# Usamos EXACTAMENTE las mismas variables del modelo completo.
# 47 numéricas + 2 categóricas = 49 predictores originales.
#
# Se utilizan las listas definidas previamente:
# variables_numericas
# variables_categoricas
# ------------------------------------------------------------

variables_modelo_final = (
    variables_numericas +
    variables_categoricas
)


X_train_final = df_train_final[
    variables_modelo_final
].copy()

y_train_final = df_train_final[
    "rendimiento_kg_ha"
].copy()


X_test_final = df_test_final[
    variables_modelo_final
].copy()

y_test_final = df_test_final[
    "rendimiento_kg_ha"
].copy()


print("\n" + "=" * 80)
print("VARIABLES")
print("=" * 80)

print(
    "Variables numéricas:",
    len(variables_numericas)
)

print(
    "Variables categóricas:",
    len(variables_categoricas)
)

print(
    "Total predictores:",
    len(variables_modelo_final)
)

print(
    "\nTarget incluido como predictor:",
    "rendimiento_kg_ha" in variables_modelo_final
)

print(
    "Año incluido como predictor:",
    "ano" in variables_modelo_final
)


# ------------------------------------------------------------
# 3. Clonar modelos ya seleccionados
# ------------------------------------------------------------
#
# No modificamos hiperparámetros.
# ------------------------------------------------------------

rf_final = clone(rf_tuned)
xgb_final = clone(xgb_tuned)


# ------------------------------------------------------------
# 4. Entrenamiento definitivo con 2017-2022
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ENTRENAMIENTO FINAL")
print("=" * 80)

rf_final.fit(
    X_train_final,
    y_train_final
)

xgb_final.fit(
    X_train_final,
    y_train_final
)

print("Random Forest entrenado.")
print("XGBoost entrenado.")


# ------------------------------------------------------------
# 5. Predicciones sobre 2023
# ------------------------------------------------------------

pred_rf_2023 = rf_final.predict(
    X_test_final
)

pred_xgb_2023 = xgb_final.predict(
    X_test_final
)


# Ensemble definido ANTES de abrir el test
pred_rf_xgb_2023 = (
    pred_rf_2023 +
    pred_xgb_2023
) / 2


# ------------------------------------------------------------
# 6. Función de métricas
# ------------------------------------------------------------

def calcular_metricas_finales(
    y_real,
    y_pred,
    nombre
):

    mae = mean_absolute_error(
        y_real,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_real,
            y_pred
        )
    )

    r2 = r2_score(
        y_real,
        y_pred
    )

    return {
        "Modelo": nombre,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


# ------------------------------------------------------------
# 7. Evaluación final
# ------------------------------------------------------------

resultados_test_2023 = pd.DataFrame([

    calcular_metricas_finales(
        y_test_final,
        pred_rf_2023,
        "Random Forest"
    ),

    calcular_metricas_finales(
        y_test_final,
        pred_rf_xgb_2023,
        "RF + XGBoost"
    ),

    # XGB se muestra como referencia secundaria,
    # aunque no era candidato final principal.
    calcular_metricas_finales(
        y_test_final,
        pred_xgb_2023,
        "XGBoost"
    )

])


resultados_test_2023 = (
    resultados_test_2023
    .sort_values("RMSE")
    .reset_index(drop=True)
)


print("\n" + "=" * 80)
print("RESULTADOS FINALES - TEST INDEPENDIENTE 2023")
print("=" * 80)

display(
    resultados_test_2023.round(3)
)


# ------------------------------------------------------------
# 8. Comparación con validación temporal previa
# ------------------------------------------------------------

comparacion_final = pd.DataFrame({

    "Modelo": [
        "Random Forest",
        "RF + XGBoost"
    ],

    "RMSE_validacion_2019_2022": [
        709.760,
        697.890
    ],

    "R2_validacion_2019_2022": [
        0.497,
        0.490
    ]

})


metricas_2023 = (
    resultados_test_2023
    .set_index("Modelo")
)


comparacion_final["RMSE_test_2023"] = [
    metricas_2023.loc[
        "Random Forest",
        "RMSE"
    ],

    metricas_2023.loc[
        "RF + XGBoost",
        "RMSE"
    ]
]


comparacion_final["R2_test_2023"] = [
    metricas_2023.loc[
        "Random Forest",
        "R2"
    ],

    metricas_2023.loc[
        "RF + XGBoost",
        "R2"
    ]
]


print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL vs TEST FINAL")
print("=" * 80)

display(
    comparacion_final.round(3)
)


# ------------------------------------------------------------
# 9. Guardar predicciones para análisis posterior
# ------------------------------------------------------------

predicciones_test_2023 = df_test_final[
    [
        "ano",
        "cultivo",
        "provincia_estandar",
        "rendimiento_kg_ha"
    ]
].copy()


predicciones_test_2023[
    "pred_RF"
] = pred_rf_2023

predicciones_test_2023[
    "pred_XGB"
] = pred_xgb_2023

predicciones_test_2023[
    "pred_RF_XGB"
] = pred_rf_xgb_2023


predicciones_test_2023[
    "error_RF"
] = (
    predicciones_test_2023["pred_RF"]
    -
    predicciones_test_2023["rendimiento_kg_ha"]
)


predicciones_test_2023[
    "error_RF_XGB"
] = (
    predicciones_test_2023["pred_RF_XGB"]
    -
    predicciones_test_2023["rendimiento_kg_ha"]
)


print("\n" + "=" * 80)
print("CONTROL FINAL")
print("=" * 80)

print(
    "TRAIN:",
    sorted(df_train_final["ano"].unique())
)

print(
    "TEST:",
    sorted(df_test_final["ano"].unique())
)

print(
    "n TRAIN:",
    len(df_train_final)
)

print(
    "n TEST:",
    len(df_test_final)
)

print(
    "\nTEST 2023 ABIERTO ÚNICAMENTE PARA EVALUACIÓN FINAL."
)

print(
    "No realizar tuning adicional utilizando estos resultados."
)

In [ ]:
# ============================================================
# PASO 51 - DIAGNÓSTICO DEL TEST FINAL 2023
# Random Forest definitivo
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. Añadir información de error
# ============================================================

diag_2023 = predicciones_test_2023.copy()

diag_2023["residuo_RF"] = (
    diag_2023["rendimiento_kg_ha"]
    - diag_2023["pred_RF"]
)

diag_2023["error_absoluto_RF"] = np.abs(
    diag_2023["residuo_RF"]
)


# ============================================================
# 2. Estadísticos observado vs predicho
# ============================================================

resumen_obs_pred = pd.DataFrame({

    "Observado_2023": [
        diag_2023["rendimiento_kg_ha"].mean(),
        diag_2023["rendimiento_kg_ha"].median(),
        diag_2023["rendimiento_kg_ha"].std(),
        diag_2023["rendimiento_kg_ha"].min(),
        diag_2023["rendimiento_kg_ha"].max()
    ],

    "Predicho_RF": [
        diag_2023["pred_RF"].mean(),
        diag_2023["pred_RF"].median(),
        diag_2023["pred_RF"].std(),
        diag_2023["pred_RF"].min(),
        diag_2023["pred_RF"].max()
    ]

}, index=[
    "Media",
    "Mediana",
    "Desviación estándar",
    "Mínimo",
    "Máximo"
])


print("=" * 80)
print("OBSERVADO vs PREDICHO - 2023")
print("=" * 80)

display(resumen_obs_pred.round(2))


# ============================================================
# 3. Sesgo medio
# ============================================================

sesgo = (
    diag_2023["pred_RF"]
    - diag_2023["rendimiento_kg_ha"]
).mean()

print("\n" + "=" * 80)
print("SESGO DEL MODELO")
print("=" * 80)

print(
    f"Sesgo medio (Predicho - Observado): "
    f"{sesgo:.2f} kg/ha"
)

if sesgo > 0:
    print("El modelo presenta SOBREESTIMACIÓN media.")
else:
    print("El modelo presenta SUBESTIMACIÓN media.")


# ============================================================
# 4. Resultados por cultivo
# ============================================================

resultados_cultivo = []

for cultivo, grupo in diag_2023.groupby("cultivo"):

    y_real = grupo["rendimiento_kg_ha"]
    y_pred = grupo["pred_RF"]

    resultados_cultivo.append({

        "cultivo": cultivo,

        "n": len(grupo),

        "rendimiento_observado_medio":
            y_real.mean(),

        "rendimiento_predicho_medio":
            y_pred.mean(),

        "sesgo_pred_menos_obs":
            (y_pred - y_real).mean(),

        "MAE":
            mean_absolute_error(
                y_real,
                y_pred
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y_real,
                    y_pred
                )
            ),

        "R2":
            r2_score(
                y_real,
                y_pred
            )
    })


resultados_cultivo = pd.DataFrame(
    resultados_cultivo
)


print("\n" + "=" * 80)
print("RESULTADOS 2023 POR CULTIVO")
print("=" * 80)

display(
    resultados_cultivo.round(2)
)


# ============================================================
# 5. Provincias con mayores errores
# ============================================================

errores_provincia = (
    diag_2023
    .groupby("provincia_estandar")
    .agg(
        n=("rendimiento_kg_ha", "size"),

        observado=(
            "rendimiento_kg_ha",
            "mean"
        ),

        predicho=(
            "pred_RF",
            "mean"
        ),

        MAE=(
            "error_absoluto_RF",
            "mean"
        ),

        sesgo=(
            "error_RF",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "MAE",
        ascending=False
    )
)


print("\n" + "=" * 80)
print("15 PROVINCIAS CON MAYOR ERROR ABSOLUTO")
print("=" * 80)

display(
    errores_provincia
    .head(15)
    .round(2)
)


# ============================================================
# 6. Gráfico observado vs predicho
# ============================================================

plt.figure(figsize=(7, 7))

plt.scatter(
    diag_2023["rendimiento_kg_ha"],
    diag_2023["pred_RF"],
    alpha=0.7
)

lim_min = min(
    diag_2023["rendimiento_kg_ha"].min(),
    diag_2023["pred_RF"].min()
)

lim_max = max(
    diag_2023["rendimiento_kg_ha"].max(),
    diag_2023["pred_RF"].max()
)

plt.plot(
    [lim_min, lim_max],
    [lim_min, lim_max],
    linestyle="--"
)

plt.xlabel(
    "Rendimiento observado (kg/ha)"
)

plt.ylabel(
    "Rendimiento predicho (kg/ha)"
)

plt.title(
    "Random Forest - Observado vs predicho\nTest independiente 2023"
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()


# ============================================================
# 7. Distribución observado vs predicho
# ============================================================

plt.figure(figsize=(9, 5))

plt.hist(
    diag_2023["rendimiento_kg_ha"],
    bins=15,
    alpha=0.6,
    label="Observado"
)

plt.hist(
    diag_2023["pred_RF"],
    bins=15,
    alpha=0.6,
    label="Predicho RF"
)

plt.xlabel(
    "Rendimiento (kg/ha)"
)

plt.ylabel(
    "Frecuencia"
)

plt.title(
    "Distribución del rendimiento observado y predicho - 2023"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 8. Magnitud global del error
# ============================================================

print("\n" + "=" * 80)
print("DIAGNÓSTICO GLOBAL 2023")
print("=" * 80)

print(
    f"Media observada: "
    f"{diag_2023['rendimiento_kg_ha'].mean():.2f} kg/ha"
)

print(
    f"Media predicha: "
    f"{diag_2023['pred_RF'].mean():.2f} kg/ha"
)

print(
    f"Sesgo medio: "
    f"{sesgo:.2f} kg/ha"
)

print(
    f"MAE: "
    f"{mean_absolute_error(diag_2023['rendimiento_kg_ha'], diag_2023['pred_RF']):.2f} kg/ha"
)

print(
    f"RMSE: "
    f"{np.sqrt(mean_squared_error(diag_2023['rendimiento_kg_ha'], diag_2023['pred_RF'])):.2f} kg/ha"
)

print(
    f"R²: "
    f"{r2_score(diag_2023['rendimiento_kg_ha'], diag_2023['pred_RF']):.3f}"
)

In [ ]:
# ============================================================
# PASO 52
# DISTRIBUTION SHIFT:
# TRAIN 2017-2022 vs TEST 2023
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Datos
# ------------------------------------------------------------

train_shift = df[df["ano"] <= 2022].copy()
test_shift  = df[df["ano"] == 2023].copy()


# ------------------------------------------------------------
# 2. Standardized Mean Difference (SMD)
# ------------------------------------------------------------

resultados_shift = []

for var in variables_numericas:

    train_var = train_shift[var].dropna()
    test_var  = test_shift[var].dropna()

    mean_train = train_var.mean()
    mean_test  = test_var.mean()

    std_train = train_var.std()
    std_test  = test_var.std()

    # SD combinada
    pooled_sd = np.sqrt(
        (
            std_train**2 +
            std_test**2
        ) / 2
    )

    if pooled_sd > 0:

        smd = (
            mean_test -
            mean_train
        ) / pooled_sd

    else:
        smd = np.nan


    resultados_shift.append({

        "variable": var,

        "media_train_2017_2022":
            mean_train,

        "media_test_2023":
            mean_test,

        "SMD":
            smd,

        "abs_SMD":
            abs(smd)
    })


shift_df = pd.DataFrame(
    resultados_shift
).sort_values(
    "abs_SMD",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 3. Clasificación descriptiva
# ------------------------------------------------------------

def clasificar_shift(x):

    if pd.isna(x):
        return "NA"

    if x < 0.2:
        return "Pequeño"

    elif x < 0.5:
        return "Moderado"

    elif x < 0.8:
        return "Importante"

    else:
        return "Muy importante"


shift_df["magnitud_shift"] = (
    shift_df["abs_SMD"]
    .apply(clasificar_shift)
)


# ------------------------------------------------------------
# 4. Resultados
# ------------------------------------------------------------

print("=" * 90)
print("DISTRIBUTION SHIFT: 2017-2022 vs 2023")
print("=" * 90)

display(
    shift_df.head(30).round(3)
)


print("\n" + "=" * 90)
print("RESUMEN")
print("=" * 90)

display(
    shift_df[
        "magnitud_shift"
    ].value_counts()
)


print(
    "\nVariables con |SMD| >= 0.5:",
    (shift_df["abs_SMD"] >= 0.5).sum()
)

print(
    "Variables con |SMD| >= 0.8:",
    (shift_df["abs_SMD"] >= 0.8).sum()
)


# ------------------------------------------------------------
# 5. TOP 20 cambios
# ------------------------------------------------------------

top_shift = (
    shift_df
    .head(20)
    .sort_values("abs_SMD")
)


plt.figure(figsize=(9, 7))

plt.barh(
    top_shift["variable"],
    top_shift["abs_SMD"]
)

plt.axvline(
    0.5,
    linestyle="--",
    linewidth=1
)

plt.axvline(
    0.8,
    linestyle="--",
    linewidth=1
)

plt.xlabel(
    "|Standardized Mean Difference|"
)

plt.ylabel(
    "Variable"
)

plt.title(
    "Cambio en la distribución de predictores\n"
    "2017–2022 vs 2023"
)

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 6. Cruzar SHIFT con SHAP
# ------------------------------------------------------------

# ranking_shap procede del Paso 48

shap_shift = ranking_shap.merge(
    shift_df[
        [
            "variable",
            "SMD",
            "abs_SMD",
            "magnitud_shift"
        ]
    ],
    on="variable",
    how="inner"
)


shap_shift = (
    shap_shift
    .sort_values(
        "mean_abs_SHAP",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 90)
print("IMPORTANCIA SHAP + DISTRIBUTION SHIFT")
print("=" * 90)

display(
    shap_shift
    .head(30)
    .round(3)
)

# Auditoria de coherencia temporal de ERA5

In [ ]:
# ============================================================
# PASO 53
# AUDITORÍA TEMPORAL ERA5
# Comprobación de coherencia 2017-2023
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Variables ERA5
# ------------------------------------------------------------

vars_radiacion = [
    "radiacion_mj_m2_ene_feb",
    "radiacion_mj_m2_mar_abr",
    "radiacion_mj_m2_may_jun",
    "radiacion_mj_m2_jul_ago",
    "radiacion_mj_m2_sep_dic",
    "radiacion_mj_m2_campana"
]

vars_precipitacion = [
    "precipitacion_mm_ene_feb",
    "precipitacion_mm_mar_abr",
    "precipitacion_mm_may_jun",
    "precipitacion_mm_jul_ago",
    "precipitacion_mm_sep_dic",
    "precipitacion_mm_campana"
]

vars_temperatura = [
    "temp_media_c_ene_feb",
    "temp_media_c_mar_abr",
    "temp_media_c_may_jun",
    "temp_media_c_jul_ago",
    "temp_media_c_sep_dic",
    "temp_media_c_campana"
]


# ------------------------------------------------------------
# 2. Medias anuales
# ------------------------------------------------------------

era5_anual = (
    df.groupby("ano")[
        vars_radiacion +
        vars_precipitacion +
        vars_temperatura
    ]
    .mean()
)


print("=" * 90)
print("MEDIAS ERA5 POR AÑO")
print("=" * 90)

display(
    era5_anual.round(2)
)


# ------------------------------------------------------------
# 3. Radiación por año
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("RADIACIÓN MEDIA POR AÑO")
print("=" * 90)

display(
    era5_anual[
        vars_radiacion
    ].round(2)
)


# ------------------------------------------------------------
# 4. Precipitación por año
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("PRECIPITACIÓN MEDIA POR AÑO")
print("=" * 90)

display(
    era5_anual[
        vars_precipitacion
    ].round(2)
)


# ------------------------------------------------------------
# 5. Temperatura por año
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TEMPERATURA MEDIA POR AÑO")
print("=" * 90)

display(
    era5_anual[
        vars_temperatura
    ].round(2)
)


# ------------------------------------------------------------
# 6. Cociente 2023 / media 2017-2022
# ------------------------------------------------------------

media_pre2023 = (
    era5_anual
    .loc[2017:2022]
    .mean()
)

ratio_2023 = (
    era5_anual.loc[2023]
    /
    media_pre2023
)


comparacion_ratio = pd.DataFrame({
    "media_2017_2022": media_pre2023,
    "valor_2023": era5_anual.loc[2023],
    "ratio_2023_vs_previo": ratio_2023
})


print("\n" + "=" * 90)
print("2023 / MEDIA 2017-2022")
print("=" * 90)

display(
    comparacion_ratio.round(3)
)

In [ ]:
# ============================================================
# PASO 54.1 - LOCALIZAR ARCHIVOS ERA5
# ============================================================

import os

RUTA_BASE = (
    "/content/TFM_Rendimiento_cultivos/"
    "TFM_Rendimiento_cultivos"
)

print("=" * 80)
print("ARCHIVOS RELACIONADOS CON ERA5")
print("=" * 80)

archivos_era5 = []

for raiz, carpetas, archivos in os.walk(RUTA_BASE):

    for archivo in archivos:

        if "era5" in archivo.lower():

            ruta = os.path.join(
                raiz,
                archivo
            )

            archivos_era5.append(ruta)

            print(ruta)

print(
    "\nNúmero de archivos encontrados:",
    len(archivos_era5)
)

In [ ]:
# ============================================================
# PASO 54.2 - LOCALIZAR CARPETAS ERA5
# ============================================================

print("=" * 80)
print("CARPETAS RELACIONADAS CON ERA5")
print("=" * 80)

carpetas_era5 = []

for raiz, carpetas, archivos in os.walk(RUTA_BASE):

    for carpeta in carpetas:

        if "era5" in carpeta.lower():

            ruta = os.path.join(
                raiz,
                carpeta
            )

            carpetas_era5.append(ruta)

            print(ruta)

print(
    "\nNúmero de carpetas encontradas:",
    len(carpetas_era5)
)

In [ ]:
# ============================================================
# PASO 54.3
# AUDITORÍA DIRECTA NETCDF ERA5-LAND: 2022 vs 2023
# ============================================================

import xarray as xr
import os
import numpy as np
import pandas as pd

RUTA_ERA5 = (
    "/content/TFM_Rendimiento_cultivos/"
    "01_Datos_originales/ERA5_Land"
)

archivos = {
    2022: os.path.join(RUTA_ERA5, "ERA5Land_mensual_2022_espana.nc"),
    2023: os.path.join(RUTA_ERA5, "ERA5Land_mensual_2023_espana.nc")
}

for ano, ruta in archivos.items():

    print("\n" + "=" * 100)
    print(f"ERA5-LAND {ano}")
    print("=" * 100)

    print("Archivo:")
    print(ruta)

    print("\nExiste:", os.path.exists(ruta))

    ds = xr.open_dataset(ruta)

    # --------------------------------------------------------
    # Estructura general
    # --------------------------------------------------------

    print("\nDIMENSIONES")
    print(ds.sizes)

    print("\nCOORDENADAS")
    print(list(ds.coords))

    print("\nVARIABLES")
    print(list(ds.data_vars))

    # --------------------------------------------------------
    # Coordenada temporal
    # --------------------------------------------------------

    posibles_tiempos = [
        "valid_time",
        "time",
        "date"
    ]

    coord_tiempo = None

    for candidato in posibles_tiempos:
        if candidato in ds.coords:
            coord_tiempo = candidato
            break

    print("\nCOORDENADA TEMPORAL:", coord_tiempo)

    if coord_tiempo is not None:

        fechas = pd.to_datetime(
            ds[coord_tiempo].values
        )

        print("Número de timestamps:", len(fechas))

        if len(fechas) > 0:
            print("Primera fecha:", fechas.min())
            print("Última fecha:", fechas.max())

            print("\nTimestamps:")
            print(fechas)

            print("\nMeses presentes:")
            print(sorted(pd.Series(fechas.month).unique()))

    # --------------------------------------------------------
    # Información de variables
    # --------------------------------------------------------

    print("\n" + "-" * 100)
    print("ATRIBUTOS DE LAS VARIABLES")
    print("-" * 100)

    for var in ds.data_vars:

        da = ds[var]

        print(f"\nVariable: {var}")
        print("Dimensiones:", da.dims)
        print("Shape:", da.shape)

        print(
            "Units:",
            da.attrs.get("units", "NO DEFINIDAS")
        )

        print(
            "Long name:",
            da.attrs.get("long_name", "NO DEFINIDO")
        )

        # Estadísticos básicos
        try:

            valores = da.values

            print(
                "Min:",
                float(np.nanmin(valores))
            )

            print(
                "Max:",
                float(np.nanmax(valores))
            )

            print(
                "Media:",
                float(np.nanmean(valores))
            )

            print(
                "NaN (%):",
                round(
                    np.isnan(valores).mean() * 100,
                    3
                )
            )

        except Exception as e:

            print(
                "No se pudieron calcular estadísticos:",
                e
            )

    ds.close()

In [ ]:
# ============================================================
# PASO 55
# AUDITORÍA DE TODOS LOS NETCDF ERA5-LAND DE 2023
# ============================================================

import os
import xarray as xr
import pandas as pd

RUTA_ERA5 = (
    "/content/TFM_Rendimiento_cultivos/"
    "01_Datos_originales/ERA5_Land"
)

resultados = []

for raiz, carpetas, archivos in os.walk(RUTA_ERA5):

    for archivo in archivos:

        # Solo NetCDF relacionados con 2023
        if (
            archivo.endswith(".nc")
            and "2023" in archivo
        ):

            ruta = os.path.join(raiz, archivo)

            try:
                ds = xr.open_dataset(ruta)

                # Detectar coordenada temporal
                coord_tiempo = None

                for candidato in ["valid_time", "time", "date"]:
                    if candidato in ds.coords:
                        coord_tiempo = candidato
                        break

                if coord_tiempo is not None:

                    fechas = pd.to_datetime(
                        ds[coord_tiempo].values
                    )

                    meses = sorted(
                        pd.Series(fechas.month)
                        .unique()
                        .tolist()
                    )

                    resultados.append({
                        "archivo": archivo,
                        "carpeta": raiz,
                        "n_timestamps": len(fechas),
                        "fecha_inicio": fechas.min(),
                        "fecha_fin": fechas.max(),
                        "meses": meses,
                        "variables": list(ds.data_vars)
                    })

                else:

                    resultados.append({
                        "archivo": archivo,
                        "carpeta": raiz,
                        "n_timestamps": None,
                        "fecha_inicio": None,
                        "fecha_fin": None,
                        "meses": None,
                        "variables": list(ds.data_vars)
                    })

                ds.close()

            except Exception as e:

                resultados.append({
                    "archivo": archivo,
                    "carpeta": raiz,
                    "n_timestamps": "ERROR",
                    "fecha_inicio": None,
                    "fecha_fin": None,
                    "meses": str(e),
                    "variables": None
                })


auditoria_2023 = pd.DataFrame(resultados)

print("=" * 100)
print("AUDITORÍA DE ARCHIVOS ERA5-LAND 2023")
print("=" * 100)

display(
    auditoria_2023[
        [
            "archivo",
            "n_timestamps",
            "fecha_inicio",
            "fecha_fin",
            "meses",
            "variables"
        ]
    ]
)

In [ ]:
# ============================================================
# PASO 56 CORREGIDO
# AUDITORÍA TEMPORAL DEL ERA5 PROVINCIAL MENSUAL
# ============================================================

import pandas as pd
import os

RUTA_CSV = (
    "/content/TFM_Rendimiento_cultivos/"
    "02_Datos_procesados/"
    "ERA5Land_provincial_mensual_2017_2023.csv"
)

era5_mensual = pd.read_csv(RUTA_CSV)

print("=" * 90)
print("AUDITORÍA TEMPORAL ERA5-LAND PROCESADO")
print("=" * 90)

print("\nDimensiones:", era5_mensual.shape)

print("\nAños de campaña disponibles:")
print(sorted(era5_mensual["ano_campana"].unique()))

print("\nAños calendario disponibles:")
print(sorted(era5_mensual["ano_calendario"].unique()))


# ============================================================
# 1. COBERTURA TEMPORAL DE CADA CAMPAÑA
# ============================================================

cobertura = (
    era5_mensual
    .groupby("ano_campana")
    .agg(
        n_filas=("mes", "size"),
        n_provincias=("provincia_estandar", "nunique"),
        n_anos_calendario=("ano_calendario", "nunique")
    )
    .reset_index()
)

print("\n" + "=" * 90)
print("COBERTURA POR CAMPAÑA")
print("=" * 90)

display(cobertura)


# ============================================================
# 2. COMBINACIONES AÑO CALENDARIO + MES POR CAMPAÑA
# ============================================================

for campana in sorted(era5_mensual["ano_campana"].unique()):

    temp = (
        era5_mensual[
            era5_mensual["ano_campana"] == campana
        ][["ano_calendario", "mes"]]
        .drop_duplicates()
        .sort_values(["ano_calendario", "mes"])
    )

    combinaciones = [
        f"{int(a)}-{int(m):02d}"
        for a, m in zip(temp["ano_calendario"], temp["mes"])
    ]

    print(f"\nCampaña {campana}:")
    print(combinaciones)


# ============================================================
# 3. COMPROBACIÓN ESPECÍFICA 2022 Y 2023
# ============================================================

for campana in [2022, 2023]:

    print("\n" + "=" * 90)
    print(f"CAMPAÑA {campana}")
    print("=" * 90)

    temp = era5_mensual[
        era5_mensual["ano_campana"] == campana
    ].copy()

    cobertura_mensual = (
        temp
        .groupby(["ano_calendario", "mes"])
        .agg(
            n_provincias=("provincia_estandar", "nunique"),
            temp_media_c=("temp_media_c", "mean"),
            temp_rocio_c=("temp_rocio_c", "mean"),
            precipitacion_mm=("precipitacion_mm", "mean"),
            radiacion_mj_m2=("radiacion_mj_m2", "mean")
        )
        .reset_index()
        .round(3)
    )

    display(cobertura_mensual)


# ============================================================
# 4. CONTROL DE NÚMERO DE MESES POR PROVINCIA
# ============================================================

control_provincias = (
    era5_mensual[
        era5_mensual["ano_campana"].isin([2022, 2023])
    ]
    .groupby(["ano_campana", "provincia_estandar"])
    .size()
    .reset_index(name="n_meses")
)

print("\n" + "=" * 90)
print("NÚMERO DE MESES POR PROVINCIA")
print("=" * 90)

display(
    control_provincias
    .groupby("ano_campana")["n_meses"]
    .agg(["min", "max", "mean"])
)


# ============================================================
# 5. ¿HAY PROVINCIAS CON COBERTURA INCOMPLETA?
# ============================================================

print("\n" + "=" * 90)
print("DISTRIBUCIÓN DEL NÚMERO DE MESES")
print("=" * 90)

display(
    control_provincias
    .groupby(["ano_campana", "n_meses"])
    .size()
    .reset_index(name="n_provincias")
)


# ============================================================
# 6. CONTROL DE VALORES FALTANTES
# ============================================================

variables_era5 = [
    "temp_media_c",
    "temp_rocio_c",
    "precipitacion_mm",
    "radiacion_mj_m2"
]

faltantes = (
    era5_mensual[
        era5_mensual["ano_campana"].isin([2022, 2023])
    ]
    .groupby("ano_campana")[variables_era5]
    .agg(lambda x: x.isna().sum())
)

print("\n" + "=" * 90)
print("VALORES FALTANTES 2022 / 2023")
print("=" * 90)

display(faltantes)


print("\n" + "=" * 90)
print("AUDITORÍA FINALIZADA")
print("=" * 90)

In [ ]:
# ============================================================
# PASO 57
# AUDITORÍA DIRECTA DE ssrd EN LOS NETCDF ERA5-LAND
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np
import os

BASE = (
    "/content/TFM_Rendimiento_cultivos/"
    "01_Datos_originales/ERA5_Land"
)

archivos = {
    2021: f"{BASE}/ERA5Land_mensual_2021_espana.nc",
    2022: f"{BASE}/ERA5Land_mensual_2022_espana.nc",
    2023: f"{BASE}/ERA5Land_mensual_2023_espana.nc"
}

resultados = []

print("=" * 90)
print("AUDITORÍA DIRECTA ssrd - ARCHIVOS NETCDF")
print("=" * 90)

for ano, ruta in archivos.items():

    print(f"\nAÑO {ano}")
    print("-" * 90)
    print("Archivo:", os.path.basename(ruta))
    print("Existe:", os.path.exists(ruta))

    ds = xr.open_dataset(ruta)

    # Detectar coordenada temporal
    if "valid_time" in ds.coords:
        tiempo = "valid_time"
    elif "time" in ds.coords:
        tiempo = "time"
    else:
        raise ValueError(f"No encuentro coordenada temporal en {ruta}")

    fechas = pd.to_datetime(ds[tiempo].values)

    print("Número de meses:", len(fechas))
    print("Fechas:", list(fechas.strftime("%Y-%m")))

    # Atributos de ssrd
    print("\nAtributos ssrd:")
    print(ds["ssrd"].attrs)

    # Media espacial mensual directa del NetCDF
    for i, fecha in enumerate(fechas):

        arr = ds["ssrd"].isel({tiempo: i}).values

        media_j_m2 = np.nanmean(arr)
        mediana_j_m2 = np.nanmedian(arr)
        minimo = np.nanmin(arr)
        maximo = np.nanmax(arr)

        resultados.append({
            "ano": ano,
            "mes": fecha.month,
            "fecha": fecha,
            "ssrd_media_J_m2": media_j_m2,
            "ssrd_mediana_J_m2": mediana_j_m2,
            "ssrd_min_J_m2": minimo,
            "ssrd_max_J_m2": maximo,
            "ssrd_media_MJ_m2": media_j_m2 / 1e6
        })

    ds.close()


df_ssrd_nc = pd.DataFrame(resultados)

print("\n" + "=" * 90)
print("MEDIA ESPACIAL DE ssrd DIRECTAMENTE DEL NETCDF")
print("=" * 90)

display(
    df_ssrd_nc[
        ["ano", "mes", "ssrd_media_J_m2", "ssrd_media_MJ_m2"]
    ].round(3)
)


# ============================================================
# COMPARACIÓN 2022 vs 2023 PARA ENERO-AGOSTO
# ============================================================

comparacion = (
    df_ssrd_nc[
        df_ssrd_nc["ano"].isin([2022, 2023])
    ]
    .pivot(
        index="mes",
        columns="ano",
        values="ssrd_media_MJ_m2"
    )
)

comparacion = comparacion.loc[
    comparacion.index.intersection(range(1, 9))
].copy()

comparacion["ratio_2023_2022"] = (
    comparacion[2023] / comparacion[2022]
)

comparacion["diferencia_pct"] = (
    (comparacion[2023] / comparacion[2022]) - 1
) * 100

print("\n" + "=" * 90)
print("COMPARACIÓN DIRECTA ssrd 2022 vs 2023")
print("=" * 90)

display(comparacion.round(3))


# ============================================================
# COMPARACIÓN 2021 / 2022 / 2023
# ============================================================

pivot = (
    df_ssrd_nc
    .pivot(
        index="mes",
        columns="ano",
        values="ssrd_media_MJ_m2"
    )
)

print("\n" + "=" * 90)
print("ssrd DIRECTO POR MES Y AÑO")
print("=" * 90)

display(pivot.round(3))


# ============================================================
# CONTROL DE UNIDADES
# ============================================================

print("\n" + "=" * 90)
print("CONTROL FINAL")
print("=" * 90)

for ano, ruta in archivos.items():

    ds = xr.open_dataset(ruta)

    print(
        ano,
        "| units:",
        ds["ssrd"].attrs.get("units"),
        "| long_name:",
        ds["ssrd"].attrs.get("long_name")
    )

    ds.close()

In [ ]:
# ============================================================
# PASO 58
# COMPARAR ssrd ENTRE LAS DISTINTAS DESCARGAS ERA5-LAND
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np
import os
import glob

BASE = (
    "/content/TFM_Rendimiento_cultivos/"
    "01_Datos_originales/ERA5_Land"
)

# ------------------------------------------------------------
# 1. Localizar NetCDF relevantes
# ------------------------------------------------------------

archivos = glob.glob(f"{BASE}/*.nc")

# Excluir prueba horaria y Canarias
archivos = [
    f for f in archivos
    if "prueba" not in os.path.basename(f).lower()
    and "canarias" not in os.path.basename(f).lower()
]

print("=" * 100)
print("ARCHIVOS A AUDITAR")
print("=" * 100)

for f in archivos:
    print(os.path.basename(f))

print("\nNúmero de archivos:", len(archivos))


# ------------------------------------------------------------
# 2. Extraer ssrd mensual directamente de cada archivo
# ------------------------------------------------------------

resultados = []

for ruta in archivos:

    nombre = os.path.basename(ruta)

    try:
        ds = xr.open_dataset(ruta)

        # detectar coordenada temporal
        if "valid_time" in ds.coords:
            tiempo = "valid_time"
        elif "time" in ds.coords:
            tiempo = "time"
        else:
            ds.close()
            continue

        if "ssrd" not in ds.data_vars:
            ds.close()
            continue

        fechas = pd.to_datetime(ds[tiempo].values)

        for i, fecha in enumerate(fechas):

            arr = ds["ssrd"].isel({tiempo: i}).values

            resultados.append({
                "archivo": nombre,
                "fecha": fecha,
                "ano": fecha.year,
                "mes": fecha.month,
                "ssrd_media_J_m2": np.nanmean(arr),
                "ssrd_media_MJ_m2": np.nanmean(arr) / 1e6
            })

        ds.close()

    except Exception as e:

        print("\nERROR leyendo:", nombre)
        print(e)


df_auditoria_ssrd = pd.DataFrame(resultados)


# ------------------------------------------------------------
# 3. Nos interesan especialmente sep-2022 → ago-2023
# ------------------------------------------------------------

inicio = pd.Timestamp("2022-09-01")
fin = pd.Timestamp("2023-08-01")

periodo_critico = df_auditoria_ssrd[
    (df_auditoria_ssrd["fecha"] >= inicio) &
    (df_auditoria_ssrd["fecha"] <= fin)
].copy()

print("\n" + "=" * 100)
print("PERIODO CRÍTICO: SEPTIEMBRE 2022 - AGOSTO 2023")
print("=" * 100)

tabla = (
    periodo_critico
    .pivot_table(
        index="fecha",
        columns="archivo",
        values="ssrd_media_MJ_m2",
        aggfunc="mean"
    )
    .round(3)
)

display(tabla)


# ------------------------------------------------------------
# 4. Comparar específicamente meses duplicados
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("MESES CON MÁS DE UNA COPIA DISPONIBLE")
print("=" * 100)

conteo = (
    periodo_critico
    .groupby("fecha")["archivo"]
    .nunique()
    .reset_index(name="n_archivos")
)

duplicados = conteo[
    conteo["n_archivos"] > 1
]["fecha"]

comparacion_duplicados = (
    periodo_critico[
        periodo_critico["fecha"].isin(duplicados)
    ]
    .sort_values(["fecha", "archivo"])
)

display(
    comparacion_duplicados[
        ["fecha", "archivo", "ssrd_media_MJ_m2"]
    ].round(3)
)


# ------------------------------------------------------------
# 5. Variabilidad entre archivos para el mismo mes
# ------------------------------------------------------------

consistencia = (
    periodo_critico
    .groupby("fecha")["ssrd_media_MJ_m2"]
    .agg(["count", "min", "max", "mean", "std"])
    .reset_index()
)

consistencia["ratio_max_min"] = (
    consistencia["max"] / consistencia["min"]
)

print("\n" + "=" * 100)
print("CONSISTENCIA ENTRE DESCARGAS")
print("=" * 100)

display(consistencia.round(3))


# ------------------------------------------------------------
# 6. Detectar discrepancias grandes
# ------------------------------------------------------------

discrepancias = consistencia[
    (consistencia["count"] > 1) &
    (consistencia["ratio_max_min"] > 1.20)
].copy()

print("\n" + "=" * 100)
print("MESES CON >20 % DE DIFERENCIA ENTRE ARCHIVOS")
print("=" * 100)

if len(discrepancias) == 0:
    print("No se detectaron discrepancias >20 % entre copias.")
else:
    display(discrepancias.round(3))


print("\n" + "=" * 100)
print("AUDITORÍA FINALIZADA")
print("=" * 100)

In [ ]:
# ============================================================
# PASO 59 - RASTREAR EL ORIGEN DE LOS DATOS ERA5 PROCESADOS
# ============================================================

import os
import glob

BASE = "/content/TFM_Rendimiento_cultivos"

print("=" * 100)
print("BÚSQUEDA DE NOTEBOOKS Y SCRIPTS RELACIONADOS CON ERA5")
print("=" * 100)

archivos_codigo = []

for patron in ["**/*.ipynb", "**/*.py"]:
    archivos_codigo.extend(
        glob.glob(os.path.join(BASE, patron), recursive=True)
    )

print(f"\nNúmero de notebooks/scripts encontrados: {len(archivos_codigo)}")

# Términos cuya aparición nos interesa especialmente
terminos = [
    "ERA5Land_provincial_mensual_2017_2023.csv",
    "ERA5Land_features_provincia_campana_2017_2023.csv",
    "ERA5Land_mensual_2023",
    "ERA5Land_mensual_2023_espana",
    "ERA5Land_mensual_2022",
    "ERA5Land_mensual_2022_espana",
    "radiacion_mj_m2",
    "ssrd"
]

resultados = []

for archivo in archivos_codigo:

    try:
        with open(archivo, "r", encoding="utf-8", errors="ignore") as f:
            contenido = f.read()

        encontrados = [t for t in terminos if t in contenido]

        if encontrados:
            resultados.append((archivo, encontrados))

    except Exception:
        pass


print("\n" + "=" * 100)
print("ARCHIVOS QUE CONTIENEN REFERENCIAS RELEVANTES")
print("=" * 100)

if len(resultados) == 0:
    print("\nNo se encontraron referencias.")

else:
    for archivo, encontrados in resultados:

        print("\nARCHIVO:")
        print(archivo)

        print("\nTérminos encontrados:")
        for termino in encontrados:
            print("  -", termino)

        print("-" * 100)


# ------------------------------------------------------------
# Buscar además referencias concretas a NetCDF ERA5
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("REFERENCIAS A ARCHIVOS .nc EN NOTEBOOKS/SCRIPTS")
print("=" * 100)

for archivo, _ in resultados:

    try:
        with open(archivo, "r", encoding="utf-8", errors="ignore") as f:
            contenido = f.read()

        # Mostrar fragmentos que contengan ERA5Land_mensual
        lineas = contenido.splitlines()

        coincidencias = [
            linea.strip()
            for linea in lineas
            if "ERA5Land_mensual" in linea
        ]

        if coincidencias:

            print("\nARCHIVO:")
            print(os.path.basename(archivo))

            # Evitamos imprimir cientos de líneas
            for linea in coincidencias[:30]:
                print(linea[:500])

    except Exception:
        pass


print("\n" + "=" * 100)
print("RASTREO FINALIZADO")
print("=" * 100)

In [ ]:
# ============================================================
# PASO 60 - COMPROBACIÓN FINAL DE CONSISTENCIA ERA5
# Dataset realmente utilizado por el modelo
# ============================================================

import pandas as pd
import numpy as np
import os

ruta = (
    "/content/TFM_Rendimiento_cultivos/"
    "02_Datos_procesados/"
    "ERA5Land_features_provincia_campana_2017_2023.csv"
)

print("=" * 90)
print("PASO 60 - CONTROL MÍNIMO DE CONSISTENCIA ERA5")
print("=" * 90)

print("\nArchivo existe:", os.path.exists(ruta))

era = pd.read_csv(ruta)

print("Dimensiones:", era.shape)

# ------------------------------------------------------------
# Variables clave
# ------------------------------------------------------------

variables = [
    "radiacion_mj_m2_ene_feb",
    "radiacion_mj_m2_mar_abr",
    "radiacion_mj_m2_may_jun",
    "radiacion_mj_m2_jul_ago",
    "radiacion_mj_m2_sep_dic",
    "radiacion_mj_m2_campana",
    "precipitacion_mm_campana",
    "temp_media_c_campana"
]

# Detectar columna de año
if "ano" in era.columns:
    col_ano = "ano"
elif "ano_campana" in era.columns:
    col_ano = "ano_campana"
else:
    raise ValueError(
        "No encuentro columna 'ano' ni 'ano_campana'. "
        f"Columnas disponibles: {era.columns.tolist()}"
    )

faltantes = [v for v in variables if v not in era.columns]

if faltantes:
    print("\nATENCIÓN. Variables no encontradas:")
    print(faltantes)

variables_ok = [v for v in variables if v in era.columns]

# ------------------------------------------------------------
# 1. Medias por campaña
# ------------------------------------------------------------

medias = (
    era.groupby(col_ano)[variables_ok]
       .mean()
       .round(3)
)

print("\n" + "=" * 90)
print("MEDIAS POR CAMPAÑA")
print("=" * 90)

display(medias)

# ------------------------------------------------------------
# 2. Comparación 2023 frente a 2017-2022
# ------------------------------------------------------------

previo = era[era[col_ano].between(2017, 2022)]
test = era[era[col_ano] == 2023]

comparacion = []

for v in variables_ok:

    media_previa = previo[v].mean()
    media_2023 = test[v].mean()

    comparacion.append({
        "variable": v,
        "media_2017_2022": media_previa,
        "media_2023": media_2023,
        "ratio_2023_previo": (
            media_2023 / media_previa
            if media_previa != 0 else np.nan
        )
    })

comparacion = pd.DataFrame(comparacion)

print("\n" + "=" * 90)
print("2023 / MEDIA 2017-2022")
print("=" * 90)

display(comparacion.round(3))

# ------------------------------------------------------------
# 3. ¿La caída de radiación ocurre en todas las provincias?
# ------------------------------------------------------------

if "provincia_estandar" in era.columns:

    rad = "radiacion_mj_m2_campana"

    if rad in era.columns:

        media_prov_previa = (
            previo.groupby("provincia_estandar")[rad]
                  .mean()
        )

        prov_2023 = (
            test.groupby("provincia_estandar")[rad]
                .mean()
        )

        comp_prov = pd.concat(
            [media_prov_previa, prov_2023],
            axis=1
        )

        comp_prov.columns = [
            "media_2017_2022",
            "valor_2023"
        ]

        comp_prov["ratio_2023_previo"] = (
            comp_prov["valor_2023"] /
            comp_prov["media_2017_2022"]
        )

        print("\n" + "=" * 90)
        print("RADIACIÓN DE CAMPAÑA: DISTRIBUCIÓN DEL RATIO POR PROVINCIA")
        print("=" * 90)

        print(
            comp_prov["ratio_2023_previo"]
            .describe()
            .round(3)
        )

        print("\nProvincias con ratio < 0.70:",
              (comp_prov["ratio_2023_previo"] < 0.70).sum())

        print("Provincias analizadas:",
              comp_prov["ratio_2023_previo"].notna().sum())

        print("\n5 ratios menores:")
        display(comp_prov.sort_values("ratio_2023_previo").head())

        print("\n5 ratios mayores:")
        display(comp_prov.sort_values("ratio_2023_previo").tail())

print("\n" + "=" * 90)
print("PASO 60 FINALIZADO")
print("=" * 90)

In [ ]:
# ============================================================
# PASO 61 - CONTROL FINAL DE LA SERIE HOMOGÉNEA *_espana.nc
# ============================================================

import os
import xarray as xr
import pandas as pd
import numpy as np

BASE = (
    "/content/TFM_Rendimiento_cultivos/"
    "01_Datos_originales/ERA5_Land"
)

resultados = []

for ano in range(2016, 2024):

    archivo = os.path.join(
        BASE,
        f"ERA5Land_mensual_{ano}_espana.nc"
    )

    if not os.path.exists(archivo):
        print("FALTA:", archivo)
        continue

    ds = xr.open_dataset(archivo)

    tiempo = "valid_time" if "valid_time" in ds.coords else "time"

    for i in range(ds.sizes[tiempo]):

        fecha = pd.to_datetime(ds[tiempo].values[i])

        ssrd = ds["ssrd"].isel({tiempo: i})

        media_J = float(ssrd.mean(skipna=True).values)

        resultados.append({
            "ano": fecha.year,
            "mes": fecha.month,
            "ssrd_media_MJ_m2": media_J / 1e6
        })

    ds.close()


serie = pd.DataFrame(resultados)

print("=" * 90)
print("COBERTURA DE *_espana.nc")
print("=" * 90)

cobertura = serie.groupby("ano").agg(
    n_meses=("mes", "nunique"),
    mes_min=("mes", "min"),
    mes_max=("mes", "max")
)

display(cobertura)


print("\n" + "=" * 90)
print("MEDIA MENSUAL DE SSRD POR AÑO")
print("=" * 90)

resumen = serie.groupby("ano")["ssrd_media_MJ_m2"].agg(
    ["mean", "min", "max"]
).round(3)

display(resumen)


print("\n" + "=" * 90)
print("ENERO-AGOSTO: COMPARACIÓN HOMOGÉNEA ENTRE AÑOS")
print("=" * 90)

ene_ago = (
    serie[serie["mes"].between(1, 8)]
    .groupby("ano")["ssrd_media_MJ_m2"]
    .mean()
    .to_frame("media_ene_ago")
)

ene_ago["ratio_vs_media_2017_2022"] = (
    ene_ago["media_ene_ago"] /
    ene_ago.loc[2017:2022, "media_ene_ago"].mean()
)

display(ene_ago.round(3))


print("\n" + "=" * 90)
print("COMPARACIÓN MES A MES 2022 vs 2023")
print("=" * 90)

tabla = (
    serie[serie["ano"].isin([2022, 2023])]
    .pivot(index="mes",
           columns="ano",
           values="ssrd_media_MJ_m2")
)

if 2022 in tabla.columns and 2023 in tabla.columns:

    tabla["ratio_2023_2022"] = tabla[2023] / tabla[2022]

display(tabla.round(3))


print("\n" + "=" * 90)
print("PASO 61 FINALIZADO")
print("=" * 90)

In [ ]:
# ============================================================
# PASO 62 - COMPROBAR CONFIGURACIÓN CDS API
# ============================================================

import os

print("Archivo ~/.cdsapirc:",
      os.path.exists(os.path.expanduser("~/.cdsapirc")))

print("Archivo en Drive:",
      os.path.exists(
          "~/.cdsapirc"
      ))

try:
    import cdsapi
    print("Librería cdsapi instalada: SÍ")
except ImportError:
    print("Librería cdsapi instalada: NO")

In [ ]:
!pip -q install cdsapi

#Descargar ERA5-Land homogéneo 2016–2023

In [ ]:
# ============================================================
# PASO 64.1 - CARPETA PARA NUEVA DESCARGA ERA5-LAND
# ============================================================

import os

DIR_NUEVO = (
    "/content/TFM_Rendimiento_cultivos/"
    "01_Datos_originales/ERA5_Land/ERA5Land_descarga_homogenea"
)

os.makedirs(DIR_NUEVO, exist_ok=True)

print("Carpeta creada:")
print(DIR_NUEVO)

In [ ]:
# ============================================================
# PASO 64.4 - CONECTAR CDS API DESDE SECRETO DE COLAB
# ============================================================

from google.colab import userdata
import cdsapi

# Recuperar token guardado en Secretos de Colab
CDS_KEY = userdata.get("API_ERA5")

# Crear cliente directamente con el token
client = cdsapi.Client(
    url="https://cds.climate.copernicus.eu/api",
    key=CDS_KEY
)

print("CDS API configurada correctamente.")
print("Token recuperado:", CDS_KEY is not None)

In [ ]:
# ============================================================
# PASO 64.5 - REDESCARGA HOMOGÉNEA ERA5-LAND 2023
# ============================================================

import os

# Carpeta ERA5-Land
ERA5_DIR = "/content/TFM_Rendimiento_cultivos/01_Datos_originales/ERA5_Land"

# Archivo NUEVO: no sobrescribimos nada existente
archivo_salida = os.path.join(
    ERA5_DIR,
    "ERA5Land_mensual_2023_CORREGIDO.nc"
)

print("Archivo de salida:")
print(archivo_salida)

# ------------------------------------------------------------
# Descarga ERA5-Land Monthly Means
# Enero-agosto 2023
# ------------------------------------------------------------

dataset = "reanalysis-era5-land-monthly-means"

request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "variable": [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "total_precipitation",
        "surface_solar_radiation_downwards"
    ],
    "year": ["2023"],
    "month": [
        "01", "02", "03", "04",
        "05", "06", "07", "08"
    ],
    "time": ["00:00"],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [
        44.5,   # North
        -19.0,  # West
        27.0,   # South
        4.5     # East
    ]
}

print("\nIniciando descarga...")
print("Periodo: enero-agosto 2023")
print("Variables: t2m, d2m, tp, ssrd")

client.retrieve(
    dataset,
    request,
    archivo_salida
)

print("\n" + "=" * 80)
print("DESCARGA FINALIZADA")
print("=" * 80)
print("Existe:", os.path.exists(archivo_salida))

if os.path.exists(archivo_salida):
    print(
        "Tamaño:",
        round(os.path.getsize(archivo_salida) / 1024**2, 2),
        "MB"
    )

In [ ]:
# ============================================================
# PASO 64.6 - VALIDACIÓN DE LA REDESCARGA ERA5-LAND 2023
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np
import os

ERA5_DIR = "/content/TFM_Rendimiento_cultivos/01_Datos_originales/ERA5_Land"

archivos = {
    "2022_espana": os.path.join(
        ERA5_DIR, "ERA5Land_mensual_2022_espana.nc"
    ),
    "2023_anterior": os.path.join(
        ERA5_DIR, "ERA5Land_mensual_2023_espana.nc"
    ),
    "2023_corregido": os.path.join(
        ERA5_DIR, "ERA5Land_mensual_2023_CORREGIDO.nc"
    )
}

resultados = []

for nombre, ruta in archivos.items():

    print("\n" + "=" * 80)
    print(nombre)
    print("=" * 80)
    print("Existe:", os.path.exists(ruta))

    ds = xr.open_dataset(ruta)

    # Detectar coordenada temporal
    if "valid_time" in ds.coords:
        tiempo = "valid_time"
    elif "time" in ds.coords:
        tiempo = "time"
    else:
        raise ValueError(f"No encuentro coordenada temporal en {nombre}")

    print("Dimensiones:", dict(ds.sizes))
    print("Variables:", list(ds.data_vars))
    print("Unidades ssrd:", ds["ssrd"].attrs.get("units"))
    print("Long name:", ds["ssrd"].attrs.get("long_name"))

    fechas = pd.to_datetime(ds[tiempo].values)

    for i, fecha in enumerate(fechas):

        # Para 2022 comparamos únicamente enero-agosto
        if fecha.month > 8:
            continue

        valor = float(ds["ssrd"].isel({tiempo: i}).mean(skipna=True).values)

        resultados.append({
            "archivo": nombre,
            "ano": fecha.year,
            "mes": fecha.month,
            "ssrd_media_J_m2": valor,
            "ssrd_media_MJ_m2": valor / 1e6
        })

    ds.close()


df_control = pd.DataFrame(resultados)

# ------------------------------------------------------------
# Tabla comparativa
# ------------------------------------------------------------

tabla = df_control.pivot(
    index="mes",
    columns="archivo",
    values="ssrd_media_MJ_m2"
)

if "2022_espana" in tabla.columns and "2023_corregido" in tabla.columns:
    tabla["ratio_2023corregido_2022"] = (
        tabla["2023_corregido"] / tabla["2022_espana"]
    )

if "2023_anterior" in tabla.columns and "2023_corregido" in tabla.columns:
    tabla["ratio_corregido_anterior"] = (
        tabla["2023_corregido"] / tabla["2023_anterior"]
    )

print("\n" + "=" * 80)
print("COMPARACIÓN ENERO-AGOSTO")
print("=" * 80)

display(tabla.round(3))


# ------------------------------------------------------------
# Resumen
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MEDIAS ENERO-AGOSTO")
print("=" * 80)

for col in ["2022_espana", "2023_anterior", "2023_corregido"]:
    if col in tabla.columns:
        print(f"{col:20s}: {tabla[col].mean():.3f} MJ/m²")

if "ratio_2023corregido_2022" in tabla.columns:
    print(
        "\nRatio medio 2023 corregido / 2022:",
        round(tabla["ratio_2023corregido_2022"].mean(), 3)
    )

if "ratio_corregido_anterior" in tabla.columns:
    print(
        "Ratio medio 2023 corregido / 2023 anterior:",
        round(tabla["ratio_corregido_anterior"].mean(), 3)
    )

print("\nPASO 64.6 FINALIZADO")

In [ ]:
# ============================================================
# PASO 64.7 - METADATOS SSRD 2022 vs 2023
# ============================================================

import xarray as xr
import os

ERA5_DIR = "/content/TFM_Rendimiento_cultivos/01_Datos_originales/ERA5_Land"

archivos = {
    "2022": os.path.join(
        ERA5_DIR, "ERA5Land_mensual_2022_espana.nc"
    ),
    "2023_nueva_descarga": os.path.join(
        ERA5_DIR, "ERA5Land_mensual_2023_CORREGIDO.nc"
    )
}

for nombre, ruta in archivos.items():

    print("\n" + "=" * 90)
    print(nombre)
    print("=" * 90)

    ds = xr.open_dataset(ruta)

    print("\nATRIBUTOS GLOBALES:")
    for k, v in ds.attrs.items():
        print(f"{k}: {v}")

    print("\nATRIBUTOS SSRD:")
    for k, v in ds["ssrd"].attrs.items():
        print(f"{k}: {v}")

    print("\nENCODING SSRD:")
    for k, v in ds["ssrd"].encoding.items():
        print(f"{k}: {v}")

    # Temporal
    tiempo = "valid_time" if "valid_time" in ds.coords else "time"

    print("\nATRIBUTOS COORDENADA TEMPORAL:")
    for k, v in ds[tiempo].attrs.items():
        print(f"{k}: {v}")

    print("\nENCODING COORDENADA TEMPORAL:")
    for k, v in ds[tiempo].encoding.items():
        print(f"{k}: {v}")

    ds.close()

print("\n" + "=" * 90)
print("PASO 64.7 FINALIZADO")
print("=" * 90)

In [ ]:
# ============================================================
# PASO 64.8 - REDESCARGA HOMOGÉNEA DE CONTROL: ERA5-LAND 2022
# ============================================================

import os

ERA5_DIR = "/content/TFM_Rendimiento_cultivos/01_Datos_originales/ERA5_Land"

archivo_2022_control = os.path.join(
    ERA5_DIR,
    "ERA5Land_mensual_2022_CONTROL.nc"
)

request_2022 = {
    "product_type": ["monthly_averaged_reanalysis"],
    "variable": [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "total_precipitation",
        "surface_solar_radiation_downwards"
    ],
    "year": ["2022"],
    "month": [
        "01", "02", "03", "04",
        "05", "06", "07", "08",
        "09", "10", "11", "12"
    ],
    "time": ["00:00"],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [
        44.5,
        -19.0,
        27.0,
        4.5
    ]
}

print("Descargando control homogéneo 2022...")

client.retrieve(
    "reanalysis-era5-land-monthly-means",
    request_2022,
    archivo_2022_control
)

print("\nDescarga finalizada.")
print("Existe:", os.path.exists(archivo_2022_control))
print(
    "Tamaño:",
    round(os.path.getsize(archivo_2022_control) / 1024**2, 2),
    "MB"
)

In [ ]:
# ============================================================
# PASO 64.8b - COMPARAR 2022 ANTIGUO vs CONTROL ACTUAL
# ============================================================

import xarray as xr
import pandas as pd
import os

archivo_antiguo = os.path.join(
    ERA5_DIR,
    "ERA5Land_mensual_2022_espana.nc"
)

archivo_control = os.path.join(
    ERA5_DIR,
    "ERA5Land_mensual_2022_CONTROL.nc"
)

def medias_ssrd(ruta):

    ds = xr.open_dataset(ruta)

    tiempo = "valid_time" if "valid_time" in ds.coords else "time"
    fechas = pd.to_datetime(ds[tiempo].values)

    datos = []

    for i, fecha in enumerate(fechas):

        valor = float(
            ds["ssrd"]
            .isel({tiempo: i})
            .mean(skipna=True)
            .values
        ) / 1e6

        datos.append({
            "mes": fecha.month,
            "ssrd_MJ_m2": valor
        })

    ds.close()

    return pd.DataFrame(datos).set_index("mes")


antiguo = medias_ssrd(archivo_antiguo)
control = medias_ssrd(archivo_control)

comparacion = antiguo.rename(
    columns={"ssrd_MJ_m2": "2022_antiguo"}
).join(
    control.rename(
        columns={"ssrd_MJ_m2": "2022_control_actual"}
    )
)

comparacion["ratio_control_antiguo"] = (
    comparacion["2022_control_actual"] /
    comparacion["2022_antiguo"]
)

comparacion["diferencia_pct"] = (
    (comparacion["ratio_control_antiguo"] - 1) * 100
)

print("=" * 80)
print("2022 ANTIGUO vs REDESCARGA ACTUAL")
print("=" * 80)

display(comparacion.round(3))

print("\nMáxima diferencia absoluta (%):",
      round(comparacion["diferencia_pct"].abs().max(), 2))

print("\nPASO 64.8 FINALIZADO")

In [ ]:
# ============================================================
# PASO 64.9 - CONTROL DE CONVERSIÓN SSRD
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np
import calendar
import os

ERA5_DIR = "/content/TFM_Rendimiento_cultivos/01_Datos_originales/ERA5_Land"

archivo = os.path.join(
    ERA5_DIR,
    "ERA5Land_mensual_2022_CONTROL.nc"
)

ds = xr.open_dataset(archivo)

tiempo = "valid_time" if "valid_time" in ds.coords else "time"

# ------------------------------------------------------------
# Seleccionar enero de 2022
# ------------------------------------------------------------

fechas = pd.to_datetime(ds[tiempo].values)

idx = np.where(
    (fechas.year == 2022) &
    (fechas.month == 1)
)[0][0]

ssrd = ds["ssrd"].isel({tiempo: idx})

valor_j_m2 = float(ssrd.mean(skipna=True).values)

dias_mes = calendar.monthrange(2022, 1)[1]

# Valor almacenado expresado en MJ/m²
valor_mj_m2_dia = valor_j_m2 / 1e6

# Acumulado mensual
valor_mj_m2_mes = valor_mj_m2_dia * dias_mes

print("=" * 80)
print("PASO 64.9 - INTERPRETACIÓN SSRD")
print("=" * 80)

print(f"\nFecha: enero 2022")
print(f"Días del mes: {dias_mes}")

print("\nValor medio almacenado:")
print(f"{valor_j_m2:,.2f} J/m²")
print(f"{valor_mj_m2_dia:.3f} MJ/m²")

print("\nSi el producto representa la media diaria del mes:")
print(
    f"Acumulado mensual = "
    f"{valor_mj_m2_dia:.3f} × {dias_mes} "
    f"= {valor_mj_m2_mes:.3f} MJ/m²"
)

print("\nAtributos:")
print("units:", ds["ssrd"].attrs.get("units"))
print("GRIB_dataType:", ds["ssrd"].attrs.get("GRIB_dataType"))
print("GRIB_paramId:", ds["ssrd"].attrs.get("GRIB_paramId"))

ds.close()

print("\n" + "=" * 80)
print("PASO 64.9 FINALIZADO")
print("=" * 80)

In [ ]:
# ============================================================
# PASO 65 - CONTROL FINAL DE VARIABLES ACUMULADAS ERA5
# ============================================================

import pandas as pd
import xarray as xr
import numpy as np
import calendar
import os

ERA5_DIR = "/content/TFM_Rendimiento_cultivos/01_Datos_originales/ERA5_Land"
PROC_DIR = "/content/TFM_Rendimiento_cultivos/02_Datos_procesados"

nc = os.path.join(
    ERA5_DIR,
    "ERA5Land_mensual_2022_CONTROL.nc"
)

csv = os.path.join(
    PROC_DIR,
    "ERA5Land_provincial_mensual_2017_2023.csv"
)

# ============================================================
# 1. Valores originales ERA5 - enero 2022
# ============================================================

ds = xr.open_dataset(nc)

tiempo = "valid_time" if "valid_time" in ds.coords else "time"
fechas = pd.to_datetime(ds[tiempo].values)

idx = np.where(
    (fechas.year == 2022) &
    (fechas.month == 1)
)[0][0]

dias = calendar.monthrange(2022, 1)[1]

tp_raw = float(
    ds["tp"].isel({tiempo: idx}).mean(skipna=True).values
)

ssrd_raw = float(
    ds["ssrd"].isel({tiempo: idx}).mean(skipna=True).values
)

ds.close()

# Conversiones correctas
tp_mm_dia = tp_raw * 1000
tp_mm_mes = tp_raw * 1000 * dias

ssrd_mj_dia = ssrd_raw / 1e6
ssrd_mj_mes = ssrd_raw / 1e6 * dias

# ============================================================
# 2. Valores del CSV procesado
# ============================================================

df = pd.read_csv(csv)

enero22 = df[
    (df["ano_calendario"] == 2022) &
    (df["mes"] == 1)
]

# ============================================================
# RESULTADOS
# ============================================================

print("=" * 80)
print("PASO 65 - CONTROL VARIABLES ACUMULADAS")
print("=" * 80)

print("\nERA5 ORIGINAL - ENERO 2022")

print("\nPRECIPITACIÓN")
print(f"Raw:                     {tp_raw:.6f} m")
print(f"Conversión diaria:        {tp_mm_dia:.3f} mm/día")
print(f"Acumulado mensual:        {tp_mm_mes:.3f} mm/mes")

print("\nRADIACIÓN")
print(f"Raw:                     {ssrd_raw:,.2f} J/m²")
print(f"Conversión diaria:        {ssrd_mj_dia:.3f} MJ/m²/día")
print(f"Acumulado mensual:        {ssrd_mj_mes:.3f} MJ/m²/mes")

print("\n" + "=" * 80)
print("CSV PROCESADO - MEDIA 50 PROVINCIAS")
print("=" * 80)

print(
    "precipitacion_mm:",
    round(enero22["precipitacion_mm"].mean(), 3)
)

print(
    "radiacion_mj_m2:",
    round(enero22["radiacion_mj_m2"].mean(), 3)
)

print("\n" + "=" * 80)
print("RATIOS CSV / VALOR DIARIO")
print("=" * 80)

print(
    "Precipitación:",
    round(
        enero22["precipitacion_mm"].mean() /
        tp_mm_dia,
        3
    )
)

print(
    "Radiación:",
    round(
        enero22["radiacion_mj_m2"].mean() /
        ssrd_mj_dia,
        3
    )
)

print("\nDías enero:", dias)

print("\nPASO 65 FINALIZADO")

In [ ]:
# ============================================================
# PASO 66 - CONTROL DE PONDERACIÓN POR NÚMERO DE DÍAS
# ============================================================

import pandas as pd
import numpy as np

PROC_DIR = "/content/TFM_Rendimiento_cultivos/02_Datos_procesados"

archivo = f"{PROC_DIR}/ERA5Land_provincial_mensual_2017_2023.csv"

df = pd.read_csv(archivo)

# Nos centramos en 2022, año completo y ya validado
d = df[df["ano_calendario"] == 2022].copy()

# Media provincial mensual
mensual = (
    d.groupby("mes")
    .agg(
        precipitacion_mm=("precipitacion_mm", "mean"),
        radiacion_mj_m2=("radiacion_mj_m2", "mean")
    )
)

print("=" * 80)
print("PASO 66 - MEDIAS PROVINCIALES MENSUALES 2022")
print("=" * 80)

print(mensual.round(3))

print("\n" + "=" * 80)
print("CONTROL DE MAGNITUD")
print("=" * 80)

print("\nRadiación:")
print("Media mensual:", round(mensual["radiacion_mj_m2"].mean(), 3))
print("Mínimo:", round(mensual["radiacion_mj_m2"].min(), 3))
print("Máximo:", round(mensual["radiacion_mj_m2"].max(), 3))

print("\nPrecipitación:")
print("Total anual medio:", round(mensual["precipitacion_mm"].sum(), 3), "mm")
print("Media mensual:", round(mensual["precipitacion_mm"].mean(), 3), "mm")

print("\nPASO 66 FINALIZADO")

In [ ]:
# ============================================================
# PASO 67 - AUDITORÍA DE FEATURES ERA5 DE CAMPAÑA
# ============================================================

import pandas as pd
import numpy as np
import os

PROC_DIR = "/content/TFM_Rendimiento_cultivos/02_Datos_procesados"

f_mensual = f"{PROC_DIR}/ERA5Land_provincial_mensual_2017_2023.csv"
f_features = f"{PROC_DIR}/ERA5Land_features_provincia_campana_2017_2023.csv"

mensual = pd.read_csv(f_mensual)
features = pd.read_csv(f_features)

print("=" * 90)
print("PASO 67 - AUDITORÍA FEATURES ERA5")
print("=" * 90)

print("\nMensual:", mensual.shape)
print("Features:", features.shape)

# ------------------------------------------------------------
# 1. Reconstruir features directamente desde el CSV mensual
# ------------------------------------------------------------

def agregar_periodo(df, meses, columna):
    x = df[df["mes"].isin(meses)]

    if columna in ["precipitacion_mm", "radiacion_mj_m2"]:
        return x.groupby(
            ["provincia_estandar", "ano_campana"]
        )[columna].sum()
    else:
        return x.groupby(
            ["provincia_estandar", "ano_campana"]
        )[columna].mean()


base = (
    mensual[["provincia_estandar", "ano_campana"]]
    .drop_duplicates()
    .copy()
)

periodos = {
    "ene_feb": [1, 2],
    "mar_abr": [3, 4],
    "may_jun": [5, 6],
    "jul_ago": [7, 8],
    "sep_dic": [9, 10, 11, 12]
}

# Radiación
for nombre, meses in periodos.items():
    s = agregar_periodo(
        mensual, meses, "radiacion_mj_m2"
    ).rename(f"radiacion_recalc_{nombre}")

    base = base.merge(
        s.reset_index(),
        on=["provincia_estandar", "ano_campana"],
        how="left"
    )

# Precipitación
for nombre, meses in periodos.items():
    s = agregar_periodo(
        mensual, meses, "precipitacion_mm"
    ).rename(f"precipitacion_recalc_{nombre}")

    base = base.merge(
        s.reset_index(),
        on=["provincia_estandar", "ano_campana"],
        how="left"
    )

# Campaña completa
rad_camp = (
    mensual.groupby(["provincia_estandar", "ano_campana"])
    ["radiacion_mj_m2"]
    .sum()
    .rename("radiacion_recalc_campana")
    .reset_index()
)

prec_camp = (
    mensual.groupby(["provincia_estandar", "ano_campana"])
    ["precipitacion_mm"]
    .sum()
    .rename("precipitacion_recalc_campana")
    .reset_index()
)

base = base.merge(
    rad_camp,
    on=["provincia_estandar", "ano_campana"],
    how="left"
)

base = base.merge(
    prec_camp,
    on=["provincia_estandar", "ano_campana"],
    how="left"
)

# ------------------------------------------------------------
# 2. Comparar con features almacenadas
# ------------------------------------------------------------

comparacion = base.merge(
    features,
    on=["provincia_estandar", "ano_campana"],
    how="left"
)

print("\n" + "=" * 90)
print("RADIACIÓN DE CAMPAÑA: RECALCULADA vs FEATURE")
print("=" * 90)

res_rad = (
    comparacion.groupby("ano_campana")
    .agg(
        recalculada=("radiacion_recalc_campana", "mean"),
        almacenada=("radiacion_mj_m2_campana", "mean")
    )
)

res_rad["ratio_almacenada_recalc"] = (
    res_rad["almacenada"] / res_rad["recalculada"]
)

print(res_rad.round(3))


print("\n" + "=" * 90)
print("PRECIPITACIÓN DE CAMPAÑA: RECALCULADA vs FEATURE")
print("=" * 90)

res_prec = (
    comparacion.groupby("ano_campana")
    .agg(
        recalculada=("precipitacion_recalc_campana", "mean"),
        almacenada=("precipitacion_mm_campana", "mean")
    )
)

res_prec["ratio_almacenada_recalc"] = (
    res_prec["almacenada"] / res_prec["recalculada"]
)

print(res_prec.round(3))


# ------------------------------------------------------------
# 3. Comparación de radiación por periodos
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("RADIACIÓN POR PERIODOS - 2022 Y 2023")
print("=" * 90)

for ano in [2022, 2023]:

    d = comparacion[comparacion["ano_campana"] == ano]

    print(f"\nAÑO {ano}")

    for periodo in periodos.keys():

        rec = d[f"radiacion_recalc_{periodo}"].mean()
        alm = d[f"radiacion_mj_m2_{periodo}"].mean()
        ratio = alm / rec if rec != 0 else np.nan

        print(
            f"{periodo:10s} | "
            f"recalc={rec:9.3f} | "
            f"feature={alm:9.3f} | "
            f"ratio={ratio:.3f}"
        )

print("\n" + "=" * 90)
print("PASO 67 FINALIZADO")
print("=" * 90)

In [ ]:
# ============================================================
# PASO 68 - CORRECCIÓN ACUMULADOS MENSUALES ERA5-LAND
# ============================================================

import pandas as pd
import numpy as np
import os
import calendar

PROC_DIR = "/content/TFM_Rendimiento_cultivos/02_Datos_procesados"

archivo_in = f"{PROC_DIR}/ERA5Land_provincial_mensual_2017_2023.csv"

# IMPORTANTE:
# No sobrescribimos todavía el archivo original.
archivo_out = f"{PROC_DIR}/ERA5Land_provincial_mensual_2017_2023_CORREGIDO.csv"

df = pd.read_csv(archivo_in)

print("=" * 90)
print("PASO 68 - CORRECCIÓN ACUMULADOS MENSUALES ERA5-LAND")
print("=" * 90)

print("\nDimensiones:", df.shape)

# ------------------------------------------------------------
# 1. Número de días de cada mes calendario
# ------------------------------------------------------------

df["dias_mes"] = df.apply(
    lambda x: calendar.monthrange(
        int(x["ano_calendario"]),
        int(x["mes"])
    )[1],
    axis=1
)

# ------------------------------------------------------------
# 2. Guardar valores anteriores para trazabilidad
# ------------------------------------------------------------

df["precipitacion_mm_diaria_original"] = df["precipitacion_mm"]
df["radiacion_mj_m2_diaria_original"] = df["radiacion_mj_m2"]

# ------------------------------------------------------------
# 3. Conversión a acumulados mensuales
# ------------------------------------------------------------

df["precipitacion_mm"] = (
    df["precipitacion_mm"] * df["dias_mes"]
)

df["radiacion_mj_m2"] = (
    df["radiacion_mj_m2"] * df["dias_mes"]
)

# ------------------------------------------------------------
# 4. Controles
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CONTROL DE MAGNITUDES POR CAMPAÑA")
print("=" * 90)

control = (
    df.groupby("ano_campana")
    .agg(
        precipitacion_campana_mm=("precipitacion_mm", "sum"),
        radiacion_campana_mj_m2=("radiacion_mj_m2", "sum")
    )
)

# Como el groupby anterior suma las 50 provincias,
# dividimos por nº provincias para obtener media provincial.
nprov = df.groupby("ano_campana")["provincia_estandar"].nunique()

control["precipitacion_media_provincia_mm"] = (
    control["precipitacion_campana_mm"] / nprov
)

control["radiacion_media_provincia_mj_m2"] = (
    control["radiacion_campana_mj_m2"] / nprov
)

print(
    control[
        [
            "precipitacion_media_provincia_mm",
            "radiacion_media_provincia_mj_m2"
        ]
    ].round(2)
)

# ------------------------------------------------------------
# 5. Control específico 2022 vs 2023
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CONTROL 2022 vs 2023")
print("=" * 90)

c = control[
    [
        "precipitacion_media_provincia_mm",
        "radiacion_media_provincia_mj_m2"
    ]
].copy()

if 2022 in c.index and 2023 in c.index:

    print("\nRatio 2023 / 2022:")

    print(
        (
            c.loc[2023] /
            c.loc[2022]
        ).round(3)
    )

# ------------------------------------------------------------
# 6. Guardar SIN sobrescribir original
# ------------------------------------------------------------

df.to_csv(archivo_out, index=False)

print("\n" + "=" * 90)
print("ARCHIVO CORREGIDO GUARDADO")
print("=" * 90)

print(archivo_out)
print("Existe:", os.path.exists(archivo_out))
print("Filas:", len(df))

print("\nPASO 68 FINALIZADO")

In [ ]:
# ============================================================
# PASO 69 - CONTROL ESPACIAL DIRECTO SSRD 2022 vs 2023
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np

BASE = "/content/TFM_Rendimiento_cultivos/01_Datos_originales/ERA5_Land"

f22 = f"{BASE}/ERA5Land_mensual_2022_espana.nc"
f23 = f"{BASE}/ERA5Land_mensual_2023_CORREGIDO.nc"

ds22 = xr.open_dataset(f22)
ds23 = xr.open_dataset(f23)

# Enero-agosto únicamente
a22 = ds22["ssrd"].isel(valid_time=slice(0,8))
a23 = ds23["ssrd"].isel(valid_time=slice(0,8))

print("=" * 85)
print("PASO 69 - CONTROL ESPACIAL DIRECTO SSRD")
print("=" * 85)

# ------------------------------------------------------------
# 1. Comprobar que las rejillas son exactamente iguales
# ------------------------------------------------------------

lat_equal = np.array_equal(
    ds22.latitude.values,
    ds23.latitude.values
)

lon_equal = np.array_equal(
    ds22.longitude.values,
    ds23.longitude.values
)

print("\nREJILLA")
print("Latitudes iguales :", lat_equal)
print("Longitudes iguales:", lon_equal)

# ------------------------------------------------------------
# 2. Ratio celda a celda por mes
# ------------------------------------------------------------

resultados = []

for i in range(8):

    x22 = a22.isel(valid_time=i).values.astype(float)
    x23 = a23.isel(valid_time=i).values.astype(float)

    mask = (
        np.isfinite(x22) &
        np.isfinite(x23) &
        (x22 > 0)
    )

    ratio = x23[mask] / x22[mask]

    resultados.append({
        "mes": i + 1,
        "n_celdas": mask.sum(),
        "ratio_mediana": np.nanmedian(ratio),
        "ratio_media": np.nanmean(ratio),
        "ratio_p05": np.nanpercentile(ratio, 5),
        "ratio_p95": np.nanpercentile(ratio, 95)
    })

res = pd.DataFrame(resultados)

print("\n" + "=" * 85)
print("RATIO SSRD 2023 / 2022 CELDA A CELDA")
print("=" * 85)

print(res.round(3).to_string(index=False))

# ------------------------------------------------------------
# 3. Control de tres puntos interiores de España
# ------------------------------------------------------------

puntos = {
    "Madrid_aprox": (40.4, -3.7),
    "Valladolid_aprox": (41.65, -4.72),
    "Cordoba_aprox": (37.88, -4.78)
}

print("\n" + "=" * 85)
print("CONTROL EN PUNTOS FIJOS")
print("=" * 85)

for nombre, (lat, lon) in puntos.items():

    s22 = (
        a22
        .sel(latitude=lat, longitude=lon, method="nearest")
        .values / 1e6
    )

    s23 = (
        a23
        .sel(latitude=lat, longitude=lon, method="nearest")
        .values / 1e6
    )

    print(f"\n{nombre}")

    tabla = pd.DataFrame({
        "mes": range(1,9),
        "2022_MJ_m2_dia": s22,
        "2023_MJ_m2_dia": s23,
        "ratio": s23 / s22
    })

    print(tabla.round(3).to_string(index=False))

ds22.close()
ds23.close()

print("\n" + "=" * 85)
print("PASO 69 FINALIZADO")
print("=" * 85)

In [ ]:
# ============================================================
# PASO 70 - CONTROL DEFINITIVO SSRD CON ERA5-LAND HORARIO
# ============================================================

import cdsapi
import xarray as xr
import numpy as np
import os
from google.colab import userdata

BASE = "/content/TFM_Rendimiento_cultivos/01_Datos_originales/ERA5_Land"
salida = f"{BASE}/CONTROL_ERA5Land_horario_enero_2023.nc"

# API ya configurada en Secrets
CDS_KEY = userdata.get("API_ERA5")

client = cdsapi.Client(
    url="https://cds.climate.copernicus.eu/api",
    key=CDS_KEY
)

print("Descargando control horario ERA5-Land...")

dataset = "reanalysis-era5-land"

request = {
    "variable": [
        "surface_solar_radiation_downwards"
    ],
    "year": "2023",
    "month": "01",
    "day": [
        "01", "02", "03", "04", "05",
        "06", "07", "08", "09", "10"
    ],
    "time": [
        "00:00", "01:00", "02:00", "03:00",
        "04:00", "05:00", "06:00", "07:00",
        "08:00", "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00", "15:00",
        "16:00", "17:00", "18:00", "19:00",
        "20:00", "21:00", "22:00", "23:00"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived",

    # Zona pequeña alrededor del centro peninsular.
    # No necesitamos descargar toda España.
    "area": [41, -5, 39, -2]
}

client.retrieve(dataset, request).download(salida)

print("\nDescarga finalizada.")
print("Existe:", os.path.exists(salida))
print("Tamaño MB:", round(os.path.getsize(salida)/1024**2, 2))

In [ ]:
# ============================================================
# PASO 70.1 - COMPARACIÓN HORARIO vs MENSUAL SSRD
# ============================================================

import xarray as xr
import numpy as np
import pandas as pd

BASE = "/content/TFM_Rendimiento_cultivos/01_Datos_originales/ERA5_Land"

f_hourly = f"{BASE}/CONTROL_ERA5Land_horario_enero_2023.nc"
f_monthly = f"{BASE}/ERA5Land_mensual_2023_CORREGIDO.nc"

dh = xr.open_dataset(f_hourly)
dm = xr.open_dataset(f_monthly)

print("=" * 80)
print("PASO 70.1 - HORARIO vs MENSUAL ERA5-LAND")
print("=" * 80)

print("\nVariables horario:", list(dh.data_vars))
print("Variables mensual:", list(dm.data_vars))

# ------------------------------------------------------------
# Localizar dimensión temporal
# ------------------------------------------------------------

time_h = "valid_time" if "valid_time" in dh.dims else "time"
time_m = "valid_time" if "valid_time" in dm.dims else "time"

# ------------------------------------------------------------
# Seleccionar exactamente la misma zona en el mensual
# que descargamos en el horario: 41N–39N, 5W–2W
# ------------------------------------------------------------

lat_h = dh.latitude
lon_h = dh.longitude

lat_min = float(lat_h.min())
lat_max = float(lat_h.max())
lon_min = float(lon_h.min())
lon_max = float(lon_h.max())

# ERA5 suele tener latitud descendente
if dm.latitude.values[0] > dm.latitude.values[-1]:
    dm_zona = dm.sel(
        latitude=slice(lat_max, lat_min),
        longitude=slice(lon_min, lon_max)
    )
else:
    dm_zona = dm.sel(
        latitude=slice(lat_min, lat_max),
        longitude=slice(lon_min, lon_max)
    )

# ------------------------------------------------------------
# HORARIO
#
# En ERA5-Land horario, ssrd es la acumulación correspondiente
# al intervalo horario.
#
# Sumamos los 24 valores horarios para obtener energía diaria.
# Después calculamos la media de los 10 días.
# ------------------------------------------------------------

ssrd_h = dh["ssrd"]

# Media espacial primero
serie_h = ssrd_h.mean(
    dim=["latitude", "longitude"],
    skipna=True
)

# Crear DataFrame temporal
df_h = pd.DataFrame({
    "fecha": pd.to_datetime(dh[time_h].values),
    "ssrd_J_m2": np.asarray(serie_h.values).reshape(-1)
})

df_h["dia"] = df_h["fecha"].dt.date

# suma de las 24 acumulaciones horarias
diario = (
    df_h.groupby("dia")["ssrd_J_m2"]
    .sum()
    / 1e6
)

media_diaria_horaria = diario.mean()

# ------------------------------------------------------------
# MENSUAL
#
# El producto monthly averaged almacena la media diaria
# correspondiente al mes.
# ------------------------------------------------------------

ssrd_m = dm_zona["ssrd"].isel({time_m: 0})

media_diaria_mensual = float(
    ssrd_m.mean(
        dim=["latitude", "longitude"],
        skipna=True
    ).values
) / 1e6

# ------------------------------------------------------------
# RESULTADOS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CONTROL HORARIO - 10 PRIMEROS DÍAS DE ENERO 2023")
print("=" * 80)

print(diario.round(3))

print("\nMedia diaria calculada desde horario:")
print(round(media_diaria_horaria, 3), "MJ/m²/día")

print("\nValor mensual ERA5-Land enero 2023:")
print(round(media_diaria_mensual, 3), "MJ/m²/día")

ratio = media_diaria_mensual / media_diaria_horaria

print("\nRatio mensual / horario:")
print(round(ratio, 3))

print("\n" + "=" * 80)

if 0.75 <= ratio <= 1.25:
    print("RESULTADO: magnitudes compatibles.")
    print("La radiación baja de 2023 está respaldada por el producto horario.")
else:
    print("RESULTADO: discrepancia importante.")
    print("El producto mensual y el horario no presentan magnitudes compatibles.")

print("=" * 80)

dh.close()
dm.close()

print("\nPASO 70.1 FINALIZADO")

In [ ]:
# ============================================================
# PASO 70.2 - ESTRUCTURA REAL DE SSRD HORARIO ERA5-LAND
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np

BASE = "/content/TFM_Rendimiento_cultivos/01_Datos_originales/ERA5_Land"
f = f"{BASE}/CONTROL_ERA5Land_horario_enero_2023.nc"

ds = xr.open_dataset(f)

print("=" * 80)
print("PASO 70.2 - ESTRUCTURA SSRD HORARIO")
print("=" * 80)

print("\nDIMENSIONES:")
print(dict(ds.sizes))

print("\nCOORDENADAS:")
print(list(ds.coords))

print("\nVARIABLES:")
print(list(ds.data_vars))

print("\nATRIBUTOS SSRD:")
for k, v in ds["ssrd"].attrs.items():
    print(f"{k}: {v}")

# Punto aproximadamente Madrid
x = ds["ssrd"].sel(
    latitude=40.4,
    longitude=-3.7,
    method="nearest"
)

print("\n" + "=" * 80)
print("PRIMEROS VALORES - MADRID APROX.")
print("=" * 80)

# Detectar coordenada temporal
time_name = "valid_time" if "valid_time" in ds.coords else "time"

tabla = pd.DataFrame({
    "fecha": pd.to_datetime(ds[time_name].values),
    "ssrd_J_m2": np.asarray(x.values).reshape(-1)
})

tabla["ssrd_MJ_m2"] = tabla["ssrd_J_m2"] / 1e6

print(tabla.head(48).to_string(index=False))

# Coordenadas especiales si existen
for coord in ["time", "valid_time", "step", "forecast_reference_time"]:
    if coord in ds.coords:
        print(f"\nCOORDENADA {coord}:")
        print(ds[coord])

print("\n" + "=" * 80)
print("PASO 70.2 FINALIZADO")
print("=" * 80)

ds.close()

In [ ]:
# ============================================================
# PASO 70.3 - VALIDACIÓN DEFINITIVA SSRD HORARIO VS MENSUAL
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np

BASE = "/content/TFM_Rendimiento_cultivos/01_Datos_originales/ERA5_Land"

f_horario = f"{BASE}/CONTROL_ERA5Land_horario_enero_2023.nc"
f_mensual = f"{BASE}/ERA5Land_mensual_2023_CORREGIDO.nc"

# ------------------------------------------------------------
# 1. Abrir archivos
# ------------------------------------------------------------

ds_h = xr.open_dataset(f_horario)
ds_m = xr.open_dataset(f_mensual)

# Punto aproximado de Madrid
h = ds_h["ssrd"].sel(
    latitude=40.4,
    longitude=-3.7,
    method="nearest"
)

m = ds_m["ssrd"].sel(
    latitude=40.4,
    longitude=-3.7,
    method="nearest"
)

fechas = pd.to_datetime(ds_h["valid_time"].values)

df = pd.DataFrame({
    "fecha": fechas,
    "ssrd_MJ_m2": np.asarray(h.values).reshape(-1) / 1e6
})

# ------------------------------------------------------------
# 2. Valor de las 00:00
#
# En ERA5-Land horario, el valor de las 00:00 representa
# el acumulado correspondiente al día anterior.
# ------------------------------------------------------------

cierres = df[df["fecha"].dt.hour == 0].copy()

cierres["dia_representado"] = (
    cierres["fecha"] - pd.Timedelta(days=1)
).dt.date

# Excluir el primer registro:
# 2023-01-01 00:00 corresponde al 31-12-2022
cierres = cierres[
    cierres["fecha"] > pd.Timestamp("2023-01-01 00:00:00")
].copy()

print("=" * 80)
print("PASO 70.3 - VALIDACIÓN DEFINITIVA SSRD")
print("=" * 80)

print("\nRADIACIÓN DIARIA DERIVADA DEL HORARIO:")
print(
    cierres[
        ["dia_representado", "ssrd_MJ_m2"]
    ].to_string(index=False)
)

media_horaria = cierres["ssrd_MJ_m2"].mean()

# ------------------------------------------------------------
# 3. Producto mensual ERA5-Land
# ------------------------------------------------------------

time_name = "valid_time" if "valid_time" in ds_m.coords else "time"

enero = m.sel({time_name: "2023-01-01"})

valor_mensual = float(enero.values) / 1e6

# ------------------------------------------------------------
# 4. Comparación
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPARACIÓN")
print("=" * 80)

print(f"\nMedia diaria desde horario (días disponibles): "
      f"{media_horaria:.3f} MJ/m²/día")

print(f"ERA5-Land monthly means enero 2023: "
      f"{valor_mensual:.3f} MJ/m²/día")

ratio = valor_mensual / media_horaria

print(f"\nRatio mensual / horario: {ratio:.3f}")

print("\nInterpretación:")
if 0.85 <= ratio <= 1.15:
    print("CONSISTENTE: ambas fuentes presentan magnitudes compatibles.")
else:
    print("NO CONSISTENTE: revisar interpretación antes de continuar.")

ds_h.close()
ds_m.close()

print("\n" + "=" * 80)
print("PASO 70.3 FINALIZADO")
print("=" * 80)

In [ ]:
# ============================================================
# PASO 71 - CORRECCIÓN DEFINITIVA DE ACUMULADOS ERA5-LAND
# ============================================================

import pandas as pd
import numpy as np
import os

BASE = "/content/TFM_Rendimiento_cultivos/02_Datos_procesados"

archivo_entrada = (
    f"{BASE}/ERA5Land_provincial_mensual_2017_2023.csv"
)

archivo_salida = (
    f"{BASE}/ERA5Land_provincial_mensual_2017_2023_CORREGIDO.csv"
)

# ------------------------------------------------------------
# 1. Cargar dataset mensual original
# ------------------------------------------------------------

df = pd.read_csv(archivo_entrada)

print("=" * 85)
print("PASO 71 - CORRECCIÓN DEFINITIVA ACUMULADOS ERA5-LAND")
print("=" * 85)

print("\nDimensiones originales:", df.shape)

# ------------------------------------------------------------
# 2. Crear fecha real año-mes
# ------------------------------------------------------------

df["fecha"] = pd.to_datetime(
    dict(
        year=df["ano_calendario"],
        month=df["mes"],
        day=1
    )
)

# Número real de días de cada mes
df["dias_mes"] = df["fecha"].dt.days_in_month

# ------------------------------------------------------------
# 3. Guardar valores anteriores SOLO para control
# ------------------------------------------------------------

df["precipitacion_mm_diaria_ERA5"] = df["precipitacion_mm"]
df["radiacion_mj_m2_diaria_ERA5"] = df["radiacion_mj_m2"]

# ------------------------------------------------------------
# 4. Convertir media diaria mensual de ERA5-Land
#    a acumulado mensual
# ------------------------------------------------------------

df["precipitacion_mm"] = (
    df["precipitacion_mm_diaria_ERA5"] * df["dias_mes"]
)

df["radiacion_mj_m2"] = (
    df["radiacion_mj_m2_diaria_ERA5"] * df["dias_mes"]
)

# ------------------------------------------------------------
# 5. Controles mínimos
# ------------------------------------------------------------

print("\nNúmero de campañas:", df["ano_campana"].nunique())
print("Provincias:", df["provincia_estandar"].nunique())
print("Filas:", len(df))

print("\nMeses por provincia/campaña:")

control_meses = (
    df.groupby(["ano_campana", "provincia_estandar"])
      .size()
)

print("Mínimo:", control_meses.min())
print("Máximo:", control_meses.max())

# ------------------------------------------------------------
# 6. Medias nacionales mensuales 2022 y 2023
#    Solo como control de magnitud
# ------------------------------------------------------------

control = (
    df[df["ano_calendario"].isin([2022, 2023])]
    .groupby(["ano_calendario", "mes"])
    [["precipitacion_mm", "radiacion_mj_m2"]]
    .mean()
    .round(3)
)

print("\n" + "=" * 85)
print("CONTROL DE MAGNITUD - MEDIAS PROVINCIALES")
print("=" * 85)

print(control)

# ------------------------------------------------------------
# 7. Acumulados medios por campaña
# ------------------------------------------------------------

campanas = (
    df.groupby(["ano_campana", "provincia_estandar"])
      .agg(
          precipitacion_campana_mm=("precipitacion_mm", "sum"),
          radiacion_campana_mj_m2=("radiacion_mj_m2", "sum")
      )
      .reset_index()
)

resumen = (
    campanas.groupby("ano_campana")
    [["precipitacion_campana_mm", "radiacion_campana_mj_m2"]]
    .mean()
    .round(2)
)

print("\n" + "=" * 85)
print("MAGNITUDES MEDIAS POR CAMPAÑA CORREGIDAS")
print("=" * 85)

print(resumen)

# ------------------------------------------------------------
# 8. Valores faltantes
# ------------------------------------------------------------

print("\nValores faltantes:")

print(
    df[
        [
            "temp_media_c",
            "temp_rocio_c",
            "precipitacion_mm",
            "radiacion_mj_m2"
        ]
    ].isna().sum()
)

# ------------------------------------------------------------
# 9. Guardar
# ------------------------------------------------------------

df.to_csv(archivo_salida, index=False)

print("\n" + "=" * 85)
print("ARCHIVO GUARDADO")
print("=" * 85)

print(archivo_salida)
print("Existe:", os.path.exists(archivo_salida))
print("Dimensiones:", df.shape)

print("\nPASO 71 FINALIZADO")

In [ ]:
# ============================================================
# PASO 72 - LOCALIZAR ARCHIVOS GEOGRÁFICOS PROVINCIALES
# ============================================================

import os

ROOT = "/content/TFM_Rendimiento_cultivos"

extensiones = (
    ".shp", ".gpkg", ".geojson",
    ".json", ".parquet"
)

encontrados = []

for ruta, carpetas, archivos in os.walk(ROOT):
    for archivo in archivos:
        if archivo.lower().endswith(extensiones):
            encontrados.append(os.path.join(ruta, archivo))

print("=" * 80)
print("PASO 72 - ARCHIVOS GEOGRÁFICOS DISPONIBLES")
print("=" * 80)

for f in encontrados:
    print(f)

print("\nNúmero de archivos encontrados:", len(encontrados))

In [ ]:
# ============================================================
# PASO 73.1 - ABRIR PROVINCIAS CNIG
# ============================================================

import geopandas as gpd

ruta_provincias = (
    "/content/TFM_Rendimiento_cultivos/"
    "02_Datos_procesados/provincias_CNIG_50.gpkg"
)

provincias = gpd.read_file(ruta_provincias)

print("Dimensiones:", provincias.shape)
print("CRS:", provincias.crs)
print("\nColumnas:")
print(provincias.columns.tolist())

print("\nPrimeras filas:")
display(provincias.head())

In [ ]:
# ============================================================
# PASO 73.2 - RECONSTRUCCIÓN LIMPIA ERA5-LAND PROVINCIAL
# ============================================================

import os
import calendar
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import Point

# ------------------------------------------------------------
# RUTAS
# ------------------------------------------------------------

ROOT = "/content/TFM_Rendimiento_cultivos"

DIR_ERA5 = f"{ROOT}/01_Datos_originales/ERA5_Land"

RUTA_PROVINCIAS = (
    f"{ROOT}/02_Datos_procesados/provincias_CNIG_50.gpkg"
)

SALIDA = (
    f"{ROOT}/02_Datos_procesados/"
    "ERA5Land_provincial_mensual_2017_2023_RECONSTRUIDO.csv"
)

# ------------------------------------------------------------
# 1. PROVINCIAS
# ------------------------------------------------------------

provincias = gpd.read_file(RUTA_PROVINCIAS)

if provincias.crs.to_epsg() != 4326:
    provincias = provincias.to_crs(4326)

print("=" * 85)
print("PASO 73.2 - RECONSTRUCCIÓN ERA5-LAND PROVINCIAL")
print("=" * 85)

print("\nProvincias:", len(provincias))
print("CRS:", provincias.crs)

# ------------------------------------------------------------
# 2. ARCHIVOS QUE UTILIZAREMOS
# ------------------------------------------------------------

archivos = {
    ano: f"{DIR_ERA5}/ERA5Land_mensual_{ano}_espana.nc"
    for ano in range(2016, 2024)
}

print("\nArchivos:")

for ano, ruta in archivos.items():
    print(ano, "->", os.path.exists(ruta))

assert all(os.path.exists(x) for x in archivos.values()), \
    "Falta algún NetCDF necesario."

# ------------------------------------------------------------
# 3. CREAR ASIGNACIÓN CELDA ERA5 -> PROVINCIA
#
# La rejilla es la misma en todos los años, por lo que
# hacemos esta operación espacial UNA SOLA VEZ.
# ------------------------------------------------------------

ds_ref = xr.open_dataset(archivos[2022])

lats = ds_ref["latitude"].values
lons = ds_ref["longitude"].values

lon_grid, lat_grid = np.meshgrid(lons, lats)

grid = gpd.GeoDataFrame(
    {
        "ilat": np.repeat(np.arange(len(lats)), len(lons)),
        "ilon": np.tile(np.arange(len(lons)), len(lats))
    },
    geometry=[
        Point(lon, lat)
        for lat, lon in zip(lat_grid.ravel(), lon_grid.ravel())
    ],
    crs="EPSG:4326"
)

# Spatial join
grid_prov = gpd.sjoin(
    grid,
    provincias[["provincia_estandar", "geometry"]],
    how="inner",
    predicate="within"
)

print("\nCeldas ERA5 asignadas a provincias:", len(grid_prov))
print(
    "Provincias con al menos una celda:",
    grid_prov["provincia_estandar"].nunique()
)

ds_ref.close()

# ------------------------------------------------------------
# 4. EXTRAER MEDIAS PROVINCIALES
# ------------------------------------------------------------

resultados = []

for ano in range(2016, 2024):

    print(f"\nProcesando {ano}...")

    ds = xr.open_dataset(archivos[ano])

    time_name = (
        "valid_time"
        if "valid_time" in ds.coords
        else "time"
    )

    fechas = pd.to_datetime(ds[time_name].values)

    for it, fecha in enumerate(fechas):

        # Solo periodo necesario para campañas 2017-2023
        if fecha < pd.Timestamp("2016-09-01"):
            continue

        if fecha > pd.Timestamp("2023-08-01"):
            continue

        dias_mes = calendar.monthrange(
            fecha.year,
            fecha.month
        )[1]

        # ----------------------------------------------------
        # Extraer matrices
        # ----------------------------------------------------

        t2m = np.asarray(
            ds["t2m"].isel({time_name: it}).values
        )

        d2m = np.asarray(
            ds["d2m"].isel({time_name: it}).values
        )

        tp = np.asarray(
            ds["tp"].isel({time_name: it}).values
        )

        ssrd = np.asarray(
            ds["ssrd"].isel({time_name: it}).values
        )

        # ----------------------------------------------------
        # Valores de cada celda asignada
        # ----------------------------------------------------

        temp = (
            t2m[
                grid_prov["ilat"].values,
                grid_prov["ilon"].values
            ] - 273.15
        )

        rocio = (
            d2m[
                grid_prov["ilat"].values,
                grid_prov["ilon"].values
            ] - 273.15
        )

        # ERA5-Land monthly averaged accumulated variables:
        # media diaria -> acumulado mensual

        precipitacion = (
            tp[
                grid_prov["ilat"].values,
                grid_prov["ilon"].values
            ]
            * 1000
            * dias_mes
        )

        radiacion = (
            ssrd[
                grid_prov["ilat"].values,
                grid_prov["ilon"].values
            ]
            / 1e6
            * dias_mes
        )

        tmp = pd.DataFrame({
            "provincia_estandar":
                grid_prov["provincia_estandar"].values,

            "temp_media_c": temp,
            "temp_rocio_c": rocio,
            "precipitacion_mm": precipitacion,
            "radiacion_mj_m2": radiacion
        })

        # Media de las celdas de cada provincia
        tmp = (
            tmp.groupby("provincia_estandar", as_index=False)
            .mean()
        )

        tmp["ano_calendario"] = fecha.year
        tmp["mes"] = fecha.month

        # Año de campaña:
        # sep-dic pertenecen a la campaña siguiente
        if fecha.month >= 9:
            tmp["ano_campana"] = fecha.year + 1
        else:
            tmp["ano_campana"] = fecha.year

        resultados.append(tmp)

    ds.close()

# ------------------------------------------------------------
# 5. UNIR TODO
# ------------------------------------------------------------

era5 = pd.concat(
    resultados,
    ignore_index=True
)

columnas = [
    "provincia_estandar",
    "ano_calendario",
    "mes",
    "temp_media_c",
    "temp_rocio_c",
    "precipitacion_mm",
    "radiacion_mj_m2",
    "ano_campana"
]

era5 = era5[columnas]

era5 = era5.sort_values(
    ["ano_campana", "provincia_estandar",
     "ano_calendario", "mes"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 6. CONTROL FINAL MÍNIMO
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("CONTROL FINAL")
print("=" * 85)

print("\nDimensiones:", era5.shape)

print(
    "Campañas:",
    sorted(era5["ano_campana"].unique())
)

print(
    "Provincias:",
    era5["provincia_estandar"].nunique()
)

control = (
    era5.groupby(
        ["ano_campana", "provincia_estandar"]
    )
    .size()
)

print("\nMeses por provincia/campaña:")
print("Mínimo:", control.min())
print("Máximo:", control.max())

print("\nValores faltantes:")
print(
    era5[
        [
            "temp_media_c",
            "temp_rocio_c",
            "precipitacion_mm",
            "radiacion_mj_m2"
        ]
    ].isna().sum()
)

# ------------------------------------------------------------
# 7. MAGNITUDES POR CAMPAÑA
# ------------------------------------------------------------

resumen = (
    era5
    .groupby(
        ["ano_campana", "provincia_estandar"]
    )
    .agg(
        precipitacion_campana_mm=(
            "precipitacion_mm", "sum"
        ),
        radiacion_campana_mj_m2=(
            "radiacion_mj_m2", "sum"
        ),
        temp_media_c=(
            "temp_media_c", "mean"
        )
    )
    .groupby("ano_campana")
    .mean()
    .round(2)
)

print("\n" + "=" * 85)
print("MEDIAS POR CAMPAÑA")
print("=" * 85)

print(resumen)

# ------------------------------------------------------------
# 8. GUARDAR
# ------------------------------------------------------------

era5.to_csv(SALIDA, index=False)

print("\n" + "=" * 85)
print("ARCHIVO RECONSTRUIDO GUARDADO")
print("=" * 85)

print(SALIDA)
print("Existe:", os.path.exists(SALIDA))
print("Filas:", len(era5))

print("\nPASO 73.2 FINALIZADO")

# Error ERA5 - descarga correcta segun forum de la página

In [ ]:
# ============================================================
# PASO 74
# DESCARGA CORRECTA ERA5-LAND PARA CAMPAÑA 2023
# Solución oficial ECMWF al known issue Sep-2022 / Feb-2024
# ============================================================

import os
import cdsapi
from google.colab import userdata

ROOT = "/content/TFM_Rendimiento_cultivos"

DIR_ERA5 = (
    f"{ROOT}/01_Datos_originales/ERA5_Land"
)

DIR_CORREGIDO = (
    f"{DIR_ERA5}/ERA5Land_FIX_oficial"
)

os.makedirs(
    DIR_CORREGIDO,
    exist_ok=True
)

# API
CDS_KEY = userdata.get("API_ERA5")

client = cdsapi.Client(
    url="https://cds.climate.copernicus.eu/api",
    key=CDS_KEY
)

dataset = "reanalysis-era5-land-monthly-means"

# ============================================================
# SEPTIEMBRE-DICIEMBRE 2022
# ============================================================

salida_2022 = os.path.join(
    DIR_CORREGIDO,
    "ERA5Land_FIX_2022_sep_dic.nc"
)

request_2022 = {

    # IMPORTANTE:
    # solución oficial para el periodo afectado
    "product_type": [
        "monthly_averaged_reanalysis_by_hour_of_day"
    ],

    "variable": [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "total_precipitation",
        "surface_solar_radiation_downwards"
    ],

    "year": ["2022"],

    "month": [
        "09", "10", "11", "12"
    ],

    "time": ["00:00"],

    "data_format": "netcdf",

    "download_format": "unarchived",

    "area": [
        44.5,
        -19.0,
        27.0,
        4.5
    ]
}

print("=" * 80)
print("DESCARGANDO SEPTIEMBRE-DICIEMBRE 2022")
print("=" * 80)

client.retrieve(
    dataset,
    request_2022,
    salida_2022
)

print("OK:", salida_2022)


# ============================================================
# ENERO-AGOSTO 2023
# ============================================================

salida_2023 = os.path.join(
    DIR_CORREGIDO,
    "ERA5Land_FIX_2023_ene_ago.nc"
)

request_2023 = {

    "product_type": [
        "monthly_averaged_reanalysis_by_hour_of_day"
    ],

    "variable": [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "total_precipitation",
        "surface_solar_radiation_downwards"
    ],

    "year": ["2023"],

    "month": [
        "01", "02", "03", "04",
        "05", "06", "07", "08"
    ],

    "time": ["00:00"],

    "data_format": "netcdf",

    "download_format": "unarchived",

    "area": [
        44.5,
        -19.0,
        27.0,
        4.5
    ]
}

print("\n" + "=" * 80)
print("DESCARGANDO ENERO-AGOSTO 2023")
print("=" * 80)

client.retrieve(
    dataset,
    request_2023,
    salida_2023
)

print("OK:", salida_2023)


print("\n" + "=" * 80)
print("PASO 74 FINALIZADO")
print("=" * 80)

print(
    "\n2022:",
    os.path.exists(salida_2022),
    round(os.path.getsize(salida_2022) / 1024**2, 2),
    "MB"
)

print(
    "2023:",
    os.path.exists(salida_2023),
    round(os.path.getsize(salida_2023) / 1024**2, 2),
    "MB"
)

In [ ]:
# ============================================================
# PASO 75
# INTEGRAR FIX OFICIAL ERA5-LAND + REGENERAR FEATURES
# CAMPAÑA 2023
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import Point

ROOT = "/content/TFM_Rendimiento_cultivos"

DIR_ERA5 = f"{ROOT}/01_Datos_originales/ERA5_Land"
DIR_FIX = f"{DIR_ERA5}/ERA5Land_FIX_oficial"
DIR_PROC = f"{ROOT}/02_Datos_procesados"

F_PROVINCIAS = f"{DIR_PROC}/provincias_CNIG_50.gpkg"

F_FIX_2022 = f"{DIR_FIX}/ERA5Land_FIX_2022_sep_dic.nc"
F_FIX_2023 = f"{DIR_FIX}/ERA5Land_FIX_2023_ene_ago.nc"

# Dataset mensual limpio que reconstruimos en el Paso 73.2
F_MENSUAL_BASE = (
    f"{DIR_PROC}/"
    "ERA5Land_provincial_mensual_2017_2023_RECONSTRUIDO.csv"
)

F_MENSUAL_FINAL = (
    f"{DIR_PROC}/"
    "ERA5Land_provincial_mensual_2017_2023_FINAL.csv"
)

F_FEATURES_FINAL = (
    f"{DIR_PROC}/"
    "ERA5Land_features_provincia_campana_2017_2023_FINAL.csv"
)


# ============================================================
# 1. CONTROLES DE ARCHIVOS
# ============================================================

print("=" * 90)
print("PASO 75 - INTEGRACIÓN FIX OFICIAL ERA5-LAND")
print("=" * 90)

for f in [
    F_PROVINCIAS,
    F_FIX_2022,
    F_FIX_2023,
    F_MENSUAL_BASE
]:
    print(os.path.basename(f), "->", os.path.exists(f))

assert os.path.exists(F_FIX_2022), "Falta FIX 2022."
assert os.path.exists(F_FIX_2023), "Falta FIX 2023."


# ============================================================
# 2. PROVINCIAS
# ============================================================

provincias = gpd.read_file(F_PROVINCIAS)

if provincias.crs.to_epsg() != 4326:
    provincias = provincias.to_crs(4326)


# ============================================================
# 3. CREAR MAPA CELDA ERA5 -> PROVINCIA
# ============================================================

ds_ref = xr.open_dataset(F_FIX_2023)

lats = ds_ref["latitude"].values
lons = ds_ref["longitude"].values

lon_grid, lat_grid = np.meshgrid(lons, lats)

grid = gpd.GeoDataFrame(
    {
        "ilat": np.repeat(np.arange(len(lats)), len(lons)),
        "ilon": np.tile(np.arange(len(lons)), len(lats))
    },
    geometry=[
        Point(lon, lat)
        for lat, lon in zip(
            lat_grid.ravel(),
            lon_grid.ravel()
        )
    ],
    crs="EPSG:4326"
)

grid_prov = gpd.sjoin(
    grid,
    provincias[
        ["provincia_estandar", "geometry"]
    ],
    how="inner",
    predicate="within"
)

ds_ref.close()

print(
    "\nCeldas asignadas:",
    len(grid_prov)
)

print(
    "Provincias:",
    grid_prov["provincia_estandar"].nunique()
)


# ============================================================
# 4. FUNCIÓN PARA EXTRAER EL FIX A PROVINCIA
# ============================================================

def procesar_fix(ruta):

    ds = xr.open_dataset(ruta)

    time_name = (
        "valid_time"
        if "valid_time" in ds.coords
        else "time"
    )

    fechas = pd.to_datetime(
        ds[time_name].values
    )

    resultados = []

    for i, fecha in enumerate(fechas):

        t2m = np.asarray(
            ds["t2m"].isel({time_name: i}).values
        )

        d2m = np.asarray(
            ds["d2m"].isel({time_name: i}).values
        )

        tp = np.asarray(
            ds["tp"].isel({time_name: i}).values
        )

        ssrd = np.asarray(
            ds["ssrd"].isel({time_name: i}).values
        )

        ilat = grid_prov["ilat"].values
        ilon = grid_prov["ilon"].values

        # Temperaturas
        temp = t2m[ilat, ilon] - 273.15
        rocio = d2m[ilat, ilon] - 273.15

        # IMPORTANTE:
        # En el FIX "monthly averaged reanalysis by hour of day"
        # a 00:00, usamos directamente los acumulados mensuales
        # recuperados según la solución oficial ECMWF.
        precipitacion = tp[ilat, ilon] * 1000
        radiacion = ssrd[ilat, ilon] / 1e6

        tmp = pd.DataFrame({
            "provincia_estandar":
                grid_prov["provincia_estandar"].values,
            "temp_media_c": temp,
            "temp_rocio_c": rocio,
            "precipitacion_mm": precipitacion,
            "radiacion_mj_m2": radiacion
        })

        tmp = (
            tmp.groupby(
                "provincia_estandar",
                as_index=False
            )
            .mean()
        )

        tmp["ano_calendario"] = fecha.year
        tmp["mes"] = fecha.month

        if fecha.month >= 9:
            tmp["ano_campana"] = fecha.year + 1
        else:
            tmp["ano_campana"] = fecha.year

        resultados.append(tmp)

    ds.close()

    return pd.concat(
        resultados,
        ignore_index=True
    )


# ============================================================
# 5. EXTRAER FIX 2022 + 2023
# ============================================================

fix_2022 = procesar_fix(F_FIX_2022)
fix_2023 = procesar_fix(F_FIX_2023)

fix = pd.concat(
    [fix_2022, fix_2023],
    ignore_index=True
)

print("\nFIX obtenido:")
print("Filas:", len(fix))
print(
    "Meses:",
    sorted(
        fix[
            ["ano_calendario", "mes"]
        ]
        .drop_duplicates()
        .itertuples(
            index=False,
            name=None
        )
    )
)


# ============================================================
# 6. SUSTITUIR SOLO CAMPAÑA 2023
# ============================================================

mensual = pd.read_csv(
    F_MENSUAL_BASE
)

# Conservamos campañas 2017-2022
mensual_sin_2023 = mensual[
    mensual["ano_campana"] != 2023
].copy()

# Incorporamos campaña 2023 corregida
mensual_final = pd.concat(
    [
        mensual_sin_2023,
        fix
    ],
    ignore_index=True
)

mensual_final = mensual_final[
    [
        "provincia_estandar",
        "ano_calendario",
        "mes",
        "temp_media_c",
        "temp_rocio_c",
        "precipitacion_mm",
        "radiacion_mj_m2",
        "ano_campana"
    ]
]

mensual_final = mensual_final.sort_values(
    [
        "ano_campana",
        "provincia_estandar",
        "ano_calendario",
        "mes"
    ]
).reset_index(drop=True)


# ============================================================
# 7. CONTROL ESTRUCTURAL
# ============================================================

print("\n" + "=" * 90)
print("CONTROL DATASET MENSUAL FINAL")
print("=" * 90)

print(
    "Dimensiones:",
    mensual_final.shape
)

print(
    "Provincias:",
    mensual_final["provincia_estandar"].nunique()
)

control = (
    mensual_final
    .groupby(
        [
            "ano_campana",
            "provincia_estandar"
        ]
    )
    .size()
)

print(
    "Meses/provincia-campaña:",
    control.min(),
    "-",
    control.max()
)

print(
    "NaN:",
    mensual_final[
        [
            "temp_media_c",
            "temp_rocio_c",
            "precipitacion_mm",
            "radiacion_mj_m2"
        ]
    ]
    .isna()
    .sum()
    .sum()
)


# ============================================================
# 8. REGENERAR FEATURES ERA5
# ============================================================

periodos = {
    "ene_feb": [1, 2],
    "mar_abr": [3, 4],
    "may_jun": [5, 6],
    "jul_ago": [7, 8],
    "sep_dic": [9, 10, 11, 12]
}

keys = [
    "provincia_estandar",
    "ano_campana"
]

features = (
    mensual_final[keys]
    .drop_duplicates()
    .copy()
)


# ------------------------------------------------------------
# Variables medias
# ------------------------------------------------------------

for variable, prefijo in [
    ("temp_media_c", "temp_media_c"),
    ("temp_rocio_c", "temp_rocio_c")
]:

    for nombre, meses in periodos.items():

        x = (
            mensual_final[
                mensual_final["mes"].isin(meses)
            ]
            .groupby(keys)[variable]
            .mean()
            .rename(
                f"{prefijo}_{nombre}"
            )
            .reset_index()
        )

        features = features.merge(
            x,
            on=keys,
            how="left"
        )


# ------------------------------------------------------------
# Variables acumuladas
# ------------------------------------------------------------

for variable, prefijo in [
    (
        "precipitacion_mm",
        "precipitacion_mm"
    ),
    (
        "radiacion_mj_m2",
        "radiacion_mj_m2"
    )
]:

    for nombre, meses in periodos.items():

        x = (
            mensual_final[
                mensual_final["mes"].isin(meses)
            ]
            .groupby(keys)[variable]
            .sum()
            .rename(
                f"{prefijo}_{nombre}"
            )
            .reset_index()
        )

        features = features.merge(
            x,
            on=keys,
            how="left"
        )


# ------------------------------------------------------------
# Campaña completa
# ------------------------------------------------------------

for variable, nombre in [
    (
        "temp_media_c",
        "temp_media_c_campana"
    ),
    (
        "temp_rocio_c",
        "temp_rocio_c_campana"
    )
]:

    x = (
        mensual_final
        .groupby(keys)[variable]
        .mean()
        .rename(nombre)
        .reset_index()
    )

    features = features.merge(
        x,
        on=keys,
        how="left"
    )


for variable, nombre in [
    (
        "precipitacion_mm",
        "precipitacion_mm_campana"
    ),
    (
        "radiacion_mj_m2",
        "radiacion_mj_m2_campana"
    )
]:

    x = (
        mensual_final
        .groupby(keys)[variable]
        .sum()
        .rename(nombre)
        .reset_index()
    )

    features = features.merge(
        x,
        on=keys,
        how="left"
    )


# ============================================================
# 9. CONTROL DE MAGNITUDES POR CAMPAÑA
# ============================================================

print("\n" + "=" * 90)
print("ERA5 FINAL - MEDIAS POR CAMPAÑA")
print("=" * 90)

resumen = (
    features
    .groupby("ano_campana")
    [
        [
            "precipitacion_mm_campana",
            "radiacion_mj_m2_campana",
            "temp_media_c_campana"
        ]
    ]
    .mean()
    .round(2)
)

display(resumen)


# ============================================================
# 10. RATIO 2023 / 2022
# ============================================================

print("\n" + "=" * 90)
print("RATIO 2023 / 2022")
print("=" * 90)

ratio = (
    resumen.loc[2023] /
    resumen.loc[2022]
)

display(
    ratio.to_frame(
        "ratio_2023_2022"
    ).round(3)
)


# ============================================================
# 11. GUARDAR
# ============================================================

mensual_final.to_csv(
    F_MENSUAL_FINAL,
    index=False
)

features.to_csv(
    F_FEATURES_FINAL,
    index=False
)

print("\n" + "=" * 90)
print("ARCHIVOS FINALES GUARDADOS")
print("=" * 90)

print(F_MENSUAL_FINAL)
print(F_FEATURES_FINAL)

print("\nDimensiones mensual:", mensual_final.shape)
print("Dimensiones features:", features.shape)

print("\nPASO 75 FINALIZADO")

In [ ]:
# ============================================================
# PASO 76
# FIX DEFINITIVO ERA5-LAND CAMPAÑA 2023
# - Corrige SOLO tp y ssrd
# - Mantiene t2m y d2m originales
# ============================================================

import os
import calendar
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import Point

ROOT = "/content/TFM_Rendimiento_cultivos"

DIR_ERA5 = f"{ROOT}/01_Datos_originales/ERA5_Land"
DIR_FIX = f"{DIR_ERA5}/ERA5Land_FIX_oficial"
DIR_PROC = f"{ROOT}/02_Datos_procesados"

F_FIX_2022 = f"{DIR_FIX}/ERA5Land_FIX_2022_sep_dic.nc"
F_FIX_2023 = f"{DIR_FIX}/ERA5Land_FIX_2023_ene_ago.nc"

F_PROVINCIAS = f"{DIR_PROC}/provincias_CNIG_50.gpkg"

# Dataset reconstruido correcto para estructura y variables no acumuladas
F_BASE = (
    f"{DIR_PROC}/"
    "ERA5Land_provincial_mensual_2017_2023_RECONSTRUIDO.csv"
)

F_MENSUAL_FINAL = (
    f"{DIR_PROC}/"
    "ERA5Land_provincial_mensual_2017_2023_DEFINITIVO.csv"
)

F_FEATURES_FINAL = (
    f"{DIR_PROC}/"
    "ERA5Land_features_provincia_campana_2017_2023_DEFINITIVO.csv"
)


print("=" * 90)
print("PASO 76 - FIX DEFINITIVO ERA5-LAND")
print("=" * 90)


# ============================================================
# 1. PROVINCIAS Y REJILLA
# ============================================================

provincias = gpd.read_file(F_PROVINCIAS)

if provincias.crs.to_epsg() != 4326:
    provincias = provincias.to_crs(4326)

ds_ref = xr.open_dataset(F_FIX_2023)

lats = ds_ref["latitude"].values
lons = ds_ref["longitude"].values

lon_grid, lat_grid = np.meshgrid(lons, lats)

grid = gpd.GeoDataFrame(
    {
        "ilat": np.repeat(np.arange(len(lats)), len(lons)),
        "ilon": np.tile(np.arange(len(lons)), len(lats))
    },
    geometry=[
        Point(lon, lat)
        for lat, lon in zip(lat_grid.ravel(), lon_grid.ravel())
    ],
    crs="EPSG:4326"
)

grid_prov = gpd.sjoin(
    grid,
    provincias[["provincia_estandar", "geometry"]],
    how="inner",
    predicate="within"
)

ds_ref.close()


# ============================================================
# 2. FUNCIÓN: EXTRAER SOLO VARIABLES ACUMULADAS DEL FIX
# ============================================================

def extraer_fix_acumulados(ruta):

    ds = xr.open_dataset(ruta)

    time_name = (
        "valid_time"
        if "valid_time" in ds.coords
        else "time"
    )

    fechas = pd.to_datetime(ds[time_name].values)

    resultados = []

    ilat = grid_prov["ilat"].values
    ilon = grid_prov["ilon"].values

    for i, fecha in enumerate(fechas):

        dias_mes = calendar.monthrange(
            fecha.year,
            fecha.month
        )[1]

        tp = np.asarray(
            ds["tp"].isel({time_name: i}).values
        )

        ssrd = np.asarray(
            ds["ssrd"].isel({time_name: i}).values
        )

        # -----------------------------------------------
        # Monthly averaged by hour of day, 00:00
        #
        # 00:00 = acumulación media de las 24 h previas.
        # Multiplicamos por días para acumulado mensual.
        # -----------------------------------------------

        precipitacion = (
            tp[ilat, ilon]
            * 1000       # m -> mm
            * dias_mes
        )

        radiacion = (
            ssrd[ilat, ilon]
            / 1e6        # J/m² -> MJ/m²
            * dias_mes
        )

        tmp = pd.DataFrame({
            "provincia_estandar":
                grid_prov["provincia_estandar"].values,
            "precipitacion_mm_FIX": precipitacion,
            "radiacion_mj_m2_FIX": radiacion
        })

        tmp = (
            tmp.groupby(
                "provincia_estandar",
                as_index=False
            )
            .mean()
        )

        tmp["ano_calendario"] = fecha.year
        tmp["mes"] = fecha.month

        if fecha.month >= 9:
            tmp["ano_campana"] = fecha.year + 1
        else:
            tmp["ano_campana"] = fecha.year

        resultados.append(tmp)

    ds.close()

    return pd.concat(resultados, ignore_index=True)


# ============================================================
# 3. EXTRAER LOS 12 MESES AFECTADOS
# ============================================================

fix = pd.concat(
    [
        extraer_fix_acumulados(F_FIX_2022),
        extraer_fix_acumulados(F_FIX_2023)
    ],
    ignore_index=True
)

print("\nFilas FIX:", len(fix))
print(
    "Provincias FIX:",
    fix["provincia_estandar"].nunique()
)

print(
    "Meses FIX:",
    fix[
        ["ano_calendario", "mes"]
    ]
    .drop_duplicates()
    .shape[0]
)


# ============================================================
# 4. DATASET BASE
# ============================================================

base = pd.read_csv(F_BASE)

print("\nBase:", base.shape)


# ============================================================
# 5. SUSTITUIR ÚNICAMENTE tp Y ssrd DE CAMPAÑA 2023
# ============================================================

keys = [
    "provincia_estandar",
    "ano_calendario",
    "mes",
    "ano_campana"
]

df_final = base.merge(
    fix,
    on=keys,
    how="left"
)

mask_2023 = (
    df_final["ano_campana"] == 2023
)

# Comprobar que hay FIX para las 600 filas
print(
    "\nFilas campaña 2023:",
    mask_2023.sum()
)

print(
    "Filas FIX disponibles campaña 2023:",
    df_final.loc[
        mask_2023,
        "radiacion_mj_m2_FIX"
    ].notna().sum()
)

# Sustituimos SOLO variables acumuladas
df_final.loc[
    mask_2023,
    "precipitacion_mm"
] = df_final.loc[
    mask_2023,
    "precipitacion_mm_FIX"
]

df_final.loc[
    mask_2023,
    "radiacion_mj_m2"
] = df_final.loc[
    mask_2023,
    "radiacion_mj_m2_FIX"
]

# Eliminar columnas auxiliares
df_final = df_final.drop(
    columns=[
        "precipitacion_mm_FIX",
        "radiacion_mj_m2_FIX"
    ]
)


# ============================================================
# 6. REGENERAR FEATURES
# ============================================================

periodos = {
    "ene_feb": [1, 2],
    "mar_abr": [3, 4],
    "may_jun": [5, 6],
    "jul_ago": [7, 8],
    "sep_dic": [9, 10, 11, 12]
}

keys_feat = [
    "provincia_estandar",
    "ano_campana"
]

features = (
    df_final[keys_feat]
    .drop_duplicates()
    .copy()
)


# Variables medias: temperatura
for variable in [
    "temp_media_c",
    "temp_rocio_c"
]:

    for periodo, meses in periodos.items():

        x = (
            df_final[
                df_final["mes"].isin(meses)
            ]
            .groupby(keys_feat)[variable]
            .mean()
            .rename(f"{variable}_{periodo}")
            .reset_index()
        )

        features = features.merge(
            x,
            on=keys_feat,
            how="left"
        )


# Variables acumuladas
for variable in [
    "precipitacion_mm",
    "radiacion_mj_m2"
]:

    for periodo, meses in periodos.items():

        x = (
            df_final[
                df_final["mes"].isin(meses)
            ]
            .groupby(keys_feat)[variable]
            .sum()
            .rename(f"{variable}_{periodo}")
            .reset_index()
        )

        features = features.merge(
            x,
            on=keys_feat,
            how="left"
        )


# Campaña completa: medias
for variable in [
    "temp_media_c",
    "temp_rocio_c"
]:

    x = (
        df_final
        .groupby(keys_feat)[variable]
        .mean()
        .rename(f"{variable}_campana")
        .reset_index()
    )

    features = features.merge(
        x,
        on=keys_feat,
        how="left"
    )


# Campaña completa: acumulados
for variable in [
    "precipitacion_mm",
    "radiacion_mj_m2"
]:

    x = (
        df_final
        .groupby(keys_feat)[variable]
        .sum()
        .rename(f"{variable}_campana")
        .reset_index()
    )

    features = features.merge(
        x,
        on=keys_feat,
        how="left"
    )


# ============================================================
# 7. CONTROL FINAL — EL ÚNICO QUE NECESITAMOS
# ============================================================

resumen = (
    features
    .groupby("ano_campana")[
        [
            "precipitacion_mm_campana",
            "radiacion_mj_m2_campana",
            "temp_media_c_campana"
        ]
    ]
    .mean()
    .round(2)
)

print("\n" + "=" * 90)
print("ERA5 DEFINITIVO - MEDIAS POR CAMPAÑA")
print("=" * 90)

display(resumen)


print("\n" + "=" * 90)
print("RATIO 2023 / 2022")
print("=" * 90)

ratio = (
    resumen.loc[2023]
    /
    resumen.loc[2022]
)

display(
    ratio
    .to_frame("ratio_2023_2022")
    .round(3)
)


# ============================================================
# 8. CONTROLES ESTRUCTURALES
# ============================================================

print("\nDimensiones mensual:", df_final.shape)
print("Dimensiones features:", features.shape)

print(
    "NaN mensual:",
    df_final[
        [
            "temp_media_c",
            "temp_rocio_c",
            "precipitacion_mm",
            "radiacion_mj_m2"
        ]
    ].isna().sum().sum()
)

control = (
    df_final
    .groupby(
        ["ano_campana", "provincia_estandar"]
    )
    .size()
)

print(
    "Meses/provincia-campaña:",
    control.min(),
    "-",
    control.max()
)


# ============================================================
# 9. GUARDAR
# ============================================================

df_final.to_csv(
    F_MENSUAL_FINAL,
    index=False
)

features.to_csv(
    F_FEATURES_FINAL,
    index=False
)

print("\nArchivos guardados:")
print(F_MENSUAL_FINAL)
print(F_FEATURES_FINAL)

print("\nPASO 76 FINALIZADO")

# Actualizar dataset maestro

In [ ]:
# ============================================================
# PASO 77 - ACTUALIZAR DATASET INTEGRADO CON ERA5 DEFINITIVO
# ============================================================

import pandas as pd
import os

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"

# Dataset integrado utilizado hasta ahora
F_DATASET = f"{PROC}/dataset_final_modelado.csv"

# ERA5 definitivo
F_ERA5 = (
    f"{PROC}/"
    "ERA5Land_features_provincia_campana_2017_2023_DEFINITIVO.csv"
)

# Salida nueva
F_SALIDA = (
    f"{PROC}/"
    "dataset_final_modelado_ERA5_DEFINITIVO.csv"
)

# ------------------------------------------------------------
# 1. Cargar
# ------------------------------------------------------------

df_base = pd.read_csv(F_DATASET)
era5 = pd.read_csv(F_ERA5)

print("=" * 90)
print("PASO 77 - ACTUALIZACIÓN DATASET CON ERA5 DEFINITIVO")
print("=" * 90)

print("\nDataset base:", df_base.shape)
print("ERA5 definitivo:", era5.shape)

print("\nColumnas ERA5:")
print(era5.columns.tolist())

# ------------------------------------------------------------
# 2. Detectar clave temporal
# ------------------------------------------------------------

if "ano" in df_base.columns:
    col_ano_base = "ano"
elif "ano_campana" in df_base.columns:
    col_ano_base = "ano_campana"
else:
    raise ValueError("No encuentro año en dataset base.")

# Normalizar nombre en ERA5
era5_merge = era5.rename(
    columns={"ano_campana": col_ano_base}
)

keys = [
    "provincia_estandar",
    col_ano_base
]

# ------------------------------------------------------------
# 3. Identificar columnas ERA5 antiguas
# ------------------------------------------------------------

cols_era5 = [
    c for c in era5_merge.columns
    if c not in keys
]

print("\nVariables ERA5 a sustituir:", len(cols_era5))

# ------------------------------------------------------------
# 4. Eliminar ERA5 antiguo del dataset base
# ------------------------------------------------------------

cols_eliminar = [
    c for c in cols_era5
    if c in df_base.columns
]

print("Variables ERA5 antiguas encontradas:", len(cols_eliminar))

df_sin_era5 = df_base.drop(
    columns=cols_eliminar
)

# ------------------------------------------------------------
# 5. Merge con ERA5 definitivo
# ------------------------------------------------------------

df_final = df_sin_era5.merge(
    era5_merge,
    on=keys,
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 6. Controles mínimos
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CONTROL FINAL")
print("=" * 90)

print("Dimensiones:", df_final.shape)

print(
    "Duplicados:",
    df_final.duplicated().sum()
)

print(
    "NaN totales en ERA5:",
    df_final[cols_era5].isna().sum().sum()
)

print(
    "Años:",
    sorted(df_final[col_ano_base].unique())
)

print(
    "Provincias:",
    df_final["provincia_estandar"].nunique()
)

if "cultivo" in df_final.columns:
    print("\nDistribución por cultivo:")
    print(df_final["cultivo"].value_counts())

# ------------------------------------------------------------
# 7. Guardar
# ------------------------------------------------------------

df_final.to_csv(
    F_SALIDA,
    index=False
)

print("\nArchivo guardado:")
print(F_SALIDA)
print("Existe:", os.path.exists(F_SALIDA))

print("\nPASO 77 FINALIZADO")

In [ ]:
# ============================================================
# PASO 77.1 - LOCALIZAR DATASET INTEGRADO EXISTENTE
# ============================================================

import os
import pandas as pd

ROOT = "/content/TFM_Rendimiento_cultivos"

carpetas = [
    f"{ROOT}/02_Datos_procesados",
    f"{ROOT}/05_Resultados"
]

print("=" * 90)
print("PASO 77.1 - CSV DISPONIBLES")
print("=" * 90)

archivos = []

for carpeta in carpetas:
    if not os.path.exists(carpeta):
        continue

    for root, dirs, files in os.walk(carpeta):
        for f in files:
            if f.lower().endswith(".csv"):
                ruta = os.path.join(root, f)

                try:
                    df_tmp = pd.read_csv(ruta, nrows=5)

                    archivos.append({
                        "archivo": f,
                        "carpeta": root,
                        "n_columnas": len(df_tmp.columns),
                        "columnas": ", ".join(df_tmp.columns)
                    })

                except Exception:
                    pass

resultado = pd.DataFrame(archivos)

print("\nNúmero de CSV encontrados:", len(resultado))

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

display(
    resultado[
        ["archivo", "carpeta", "n_columnas", "columnas"]
    ]
)

print("\nPASO 77.1 FINALIZADO")

In [ ]:
# ============================================================
# PASO 78 - ACTUALIZAR DATASET MAESTRO CON ERA5 DEFINITIVO
# ============================================================

import pandas as pd
import os

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"

F_MAESTRO = f"{PROC}/dataset_maestro_TFM_2017_2023.csv"

F_ERA5 = (
    f"{PROC}/"
    "ERA5Land_features_provincia_campana_2017_2023_DEFINITIVO.csv"
)

F_SALIDA = (
    f"{PROC}/"
    "dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

# ============================================================
# 1. CARGAR
# ============================================================

maestro = pd.read_csv(F_MAESTRO)
era5 = pd.read_csv(F_ERA5)

print("=" * 90)
print("PASO 78 - ACTUALIZACIÓN DEL DATASET MAESTRO")
print("=" * 90)

print("\nMaestro original:", maestro.shape)
print("ERA5 definitivo:", era5.shape)

# ============================================================
# 2. NORMALIZAR CLAVE TEMPORAL
# ============================================================

era5 = era5.rename(
    columns={"ano_campana": "ano"}
)

keys = [
    "provincia_estandar",
    "ano"
]

# ============================================================
# 3. VARIABLES ERA5 DEFINITIVAS
# ============================================================

cols_era5 = [
    c for c in era5.columns
    if c not in keys
]

print("\nVariables ERA5 definitivas:", len(cols_era5))

for c in cols_era5:
    print(" -", c)

# ============================================================
# 4. COMPROBAR QUE LAS CLAVES SON ÚNICAS EN ERA5
# ============================================================

duplicados_era5 = era5.duplicated(
    subset=keys
).sum()

print("\nDuplicados ERA5 por provincia+año:", duplicados_era5)

assert duplicados_era5 == 0, \
    "ERA5 tiene duplicados en provincia+año."

# ============================================================
# 5. ELIMINAR VARIABLES ERA5 ANTIGUAS DEL MAESTRO
# ============================================================

cols_era5_antiguas = [
    c for c in cols_era5
    if c in maestro.columns
]

print(
    "Variables ERA5 antiguas encontradas en maestro:",
    len(cols_era5_antiguas)
)

maestro_sin_era5 = maestro.drop(
    columns=cols_era5_antiguas
)

# ============================================================
# 6. MERGE
# ============================================================

maestro_final = maestro_sin_era5.merge(
    era5,
    on=keys,
    how="left",
    validate="many_to_one"
)

# ============================================================
# 7. CONTROLES MÍNIMOS
# ============================================================

print("\n" + "=" * 90)
print("CONTROL FINAL")
print("=" * 90)

print("Dimensiones original:", maestro.shape)
print("Dimensiones definitivo:", maestro_final.shape)

print(
    "NaN en variables ERA5:",
    maestro_final[cols_era5].isna().sum().sum()
)

print(
    "Duplicados filas completas:",
    maestro_final.duplicated().sum()
)

print(
    "Años:",
    sorted(maestro_final["ano"].unique())
)

print(
    "Provincias:",
    maestro_final["provincia_estandar"].nunique()
)

print(
    "Cultivos:",
    maestro_final["cultivo"].value_counts().to_dict()
)

print(
    "Target NaN:",
    maestro_final["rendimiento_kg_ha"].isna().sum()
)

# ============================================================
# 8. COMPROBAR 2023
# ============================================================

control_2023 = (
    maestro_final[
        maestro_final["ano"] == 2023
    ][
        [
            "precipitacion_mm_campana",
            "radiacion_mj_m2_campana",
            "temp_media_c_campana"
        ]
    ]
    .mean()
)

print("\nMedias ERA5 2023 en dataset maestro:")
print(control_2023.round(3))

# ============================================================
# 9. GUARDAR
# ============================================================

maestro_final.to_csv(
    F_SALIDA,
    index=False
)

print("\n" + "=" * 90)
print("DATASET MAESTRO DEFINITIVO GUARDADO")
print("=" * 90)

print(F_SALIDA)
print("Existe:", os.path.exists(F_SALIDA))

print("\nPASO 78 FINALIZADO")

# Modelado definitivo rendimiento actualizado (ERA)

In [ ]:
# ============================================================
# PASO 79 - PREPARACIÓN DEL DATASET DEFINITIVO PARA MODELADO
# ============================================================

import pandas as pd
import numpy as np
import os

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"

F_DATASET = (
    f"{PROC}/"
    "dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

# ------------------------------------------------------------
# 1. CARGAR DATASET
# ------------------------------------------------------------

df = pd.read_csv(F_DATASET)

print("=" * 90)
print("PASO 79 - DATASET DEFINITIVO PARA MODELADO")
print("=" * 90)

print("\nArchivo:", F_DATASET)
print("Dimensiones:", df.shape)

# ------------------------------------------------------------
# 2. ESTRUCTURA GENERAL
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("COLUMNAS")
print("=" * 90)

for i, c in enumerate(df.columns, 1):
    print(f"{i:02d}. {c}")

# ------------------------------------------------------------
# 3. VARIABLES BÁSICAS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("ESTRUCTURA DEL DATASET")
print("=" * 90)

print("Años:", sorted(df["ano"].unique()))
print("Cultivos:", df["cultivo"].value_counts().to_dict())
print("Provincias:", df["provincia_estandar"].nunique())

print("\nObservaciones por año:")
print(df.groupby("ano").size())

print("\nObservaciones por cultivo y año:")
print(
    df.groupby(["ano", "cultivo"])
      .size()
      .unstack(fill_value=0)
)

# ------------------------------------------------------------
# 4. TARGET
# ------------------------------------------------------------

TARGET = "rendimiento_kg_ha"

print("\n" + "=" * 90)
print("TARGET")
print("=" * 90)

print(
    df.groupby("cultivo")[TARGET]
      .agg(["count", "mean", "std", "min", "median", "max"])
      .round(1)
)

# ------------------------------------------------------------
# 5. VALORES FALTANTES
# ------------------------------------------------------------

nan = df.isna().sum()
nan = nan[nan > 0].sort_values(ascending=False)

print("\n" + "=" * 90)
print("VALORES FALTANTES")
print("=" * 90)

if len(nan) == 0:
    print("No hay valores faltantes.")
else:
    print(nan)

# ------------------------------------------------------------
# 6. DUPLICADOS DE LA UNIDAD DE ANÁLISIS
# ------------------------------------------------------------

claves = [
    "ano",
    "cultivo",
    "provincia_estandar"
]

n_dup = df.duplicated(subset=claves).sum()

print("\nDuplicados año + cultivo + provincia:", n_dup)

# ------------------------------------------------------------
# 7. IDENTIFICAR PREDICTORES
# ------------------------------------------------------------

NO_PREDICTORES = [
    "rendimiento_kg_ha",
    "ano",
    "provincia_estandar"
]

predictores = [
    c for c in df.columns
    if c not in NO_PREDICTORES
]

numericos = [
    c for c in predictores
    if pd.api.types.is_numeric_dtype(df[c])
]

categoricos = [
    c for c in predictores
    if not pd.api.types.is_numeric_dtype(df[c])
]

print("\n" + "=" * 90)
print("PREDICTORES")
print("=" * 90)

print("Total:", len(predictores))
print("Numéricos:", len(numericos))
print("Categóricos:", len(categoricos))

print("\nVariables categóricas:")
print(categoricos)

# ------------------------------------------------------------
# 8. PARTICIÓN TEMPORAL
# ------------------------------------------------------------

train = df[df["ano"] <= 2022].copy()
test = df[df["ano"] == 2023].copy()

print("\n" + "=" * 90)
print("PARTICIÓN TEMPORAL")
print("=" * 90)

print("TRAIN 2017-2022:", train.shape)
print("TEST  2023     :", test.shape)

print("\nTrain por cultivo:")
print(train["cultivo"].value_counts())

print("\nTest 2023 por cultivo:")
print(test["cultivo"].value_counts())

# ------------------------------------------------------------
# 9. CONTROL DE SOLAPAMIENTO TEMPORAL
# ------------------------------------------------------------

print("\nAños train:", sorted(train["ano"].unique()))
print("Años test :", sorted(test["ano"].unique()))

assert train["ano"].max() < test["ano"].min()

print("\nSeparación temporal correcta: TRUE")

print("\nPASO 79 FINALIZADO")

# Selección de modelos (nuevo ERA5)

In [ ]:
# ============================================================
# PASO 80 - COMPARACIÓN TEMPORAL DE 5 MODELOS
# Dataset definitivo con ERA5 corregido
# ============================================================

!pip -q install xgboost

import os
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# 1. CARGAR DATASET DEFINITIVO
# ============================================================

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"
RESULTADOS = f"{ROOT}/05_Resultados"

F_DATASET = (
    f"{PROC}/dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

df = pd.read_csv(F_DATASET)

TARGET = "rendimiento_kg_ha"

# Año se usa solo para validación temporal.
# Provincia queda fuera del modelo en esta comparación.
EXCLUIR = [
    TARGET,
    "ano",
    "provincia_estandar"
]

features = [
    c for c in df.columns
    if c not in EXCLUIR
]

num_cols = [
    c for c in features
    if c != "cultivo"
]

cat_cols = ["cultivo"]

print("=" * 90)
print("PASO 80 - COMPARACIÓN TEMPORAL DE MODELOS")
print("=" * 90)

print("\nDataset:", df.shape)
print("Predictores:", len(features))
print("Numéricos:", len(num_cols))
print("Categóricos:", len(cat_cols))

print("\nAños disponibles:")
print(sorted(df["ano"].unique()))

print("\n2023 reservado para test final: SÍ")

# ============================================================
# 2. PREPROCESADO
# ============================================================

# Para Ridge necesitamos estandarización.
preprocessor_lineal = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(strategy="median")
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    )
                ]
            ),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            cat_cols
        )
    ]
)

# Para árboles no hace falta escalar.
preprocessor_arboles = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            cat_cols
        )
    ]
)

# ============================================================
# 3. MODELOS
# ============================================================

modelos = {

    "Ridge": {
        "preprocessor": preprocessor_lineal,
        "modelo": Ridge(
            alpha=10.0
        )
    },

    "RandomForest": {
        "preprocessor": preprocessor_arboles,
        "modelo": RandomForestRegressor(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        )
    },

    "ExtraTrees": {
        "preprocessor": preprocessor_arboles,
        "modelo": ExtraTreesRegressor(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            bootstrap=False,
            random_state=42,
            n_jobs=-1
        )
    },

    "GradientBoosting": {
        "preprocessor": preprocessor_arboles,
        "modelo": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=2,
            min_samples_split=2,
            min_samples_leaf=3,
            subsample=0.9,
            random_state=42
        )
    },

    "XGBoost": {
        "preprocessor": preprocessor_arboles,
        "modelo": XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=4,
            min_child_weight=5,
            subsample=0.9,
            colsample_bytree=0.7,
            reg_alpha=0.01,
            reg_lambda=1.0,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    }
}

# ============================================================
# 4. VALIDACIÓN TEMPORAL
# ============================================================

validation_years = [2020, 2021, 2022]

resultados = []

for ano_val in validation_years:

    train = df[df["ano"] < ano_val].copy()
    val = df[df["ano"] == ano_val].copy()

    # Seguridad: nunca usar 2023
    train = train[train["ano"] <= 2022]
    val = val[val["ano"] <= 2022]

    X_train = train[features]
    y_train = train[TARGET]

    X_val = val[features]
    y_val = val[TARGET]

    print("\n" + "=" * 90)
    print(
        f"VALIDACIÓN {ano_val} | "
        f"TRAIN {train['ano'].min()}-{train['ano'].max()} | "
        f"n_train={len(train)} | "
        f"n_val={len(val)}"
    )
    print("=" * 90)

    for nombre, config in modelos.items():

        pipe = Pipeline(
            steps=[
                (
                    "preprocessor",
                    config["preprocessor"]
                ),
                (
                    "modelo",
                    config["modelo"]
                )
            ]
        )

        pipe.fit(
            X_train,
            y_train
        )

        pred = pipe.predict(
            X_val
        )

        mae = mean_absolute_error(
            y_val,
            pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                pred
            )
        )

        r2 = r2_score(
            y_val,
            pred
        )

        resultados.append({
            "modelo": nombre,
            "ano_validacion": ano_val,
            "train_desde": int(train["ano"].min()),
            "train_hasta": int(train["ano"].max()),
            "n_train": len(train),
            "n_validacion": len(val),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

        print(
            f"{nombre:18s} | "
            f"MAE={mae:8.2f} | "
            f"RMSE={rmse:8.2f} | "
            f"R²={r2:7.3f}"
        )

# ============================================================
# 5. TABLA DE RESULTADOS POR FOLD
# ============================================================

res = pd.DataFrame(
    resultados
)

print("\n" + "=" * 90)
print("RESULTADOS POR FOLD")
print("=" * 90)

display(
    res.round(3)
)

# ============================================================
# 6. RESUMEN GLOBAL POR MODELO
# ============================================================

resumen = (
    res
    .groupby("modelo")
    .agg(
        MAE_medio=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        RMSE_medio=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),

        R2_medio=("R2", "mean"),
        R2_std=("R2", "std"),

        R2_min=("R2", "min"),
        R2_max=("R2", "max")
    )
    .sort_values(
        ["RMSE_medio", "MAE_medio"]
    )
    .round(3)
)

print("\n" + "=" * 90)
print("COMPARACIÓN GLOBAL")
print("=" * 90)

display(resumen)

# ============================================================
# 7. RANKING SIMPLE
# ============================================================

ranking = resumen.reset_index().copy()

ranking["rank_RMSE"] = (
    ranking["RMSE_medio"]
    .rank(method="min")
)

ranking["rank_MAE"] = (
    ranking["MAE_medio"]
    .rank(method="min")
)

ranking["rank_R2"] = (
    ranking["R2_medio"]
    .rank(
        method="min",
        ascending=False
    )
)

# Ranking combinado solo como apoyo, no como criterio único.
ranking["rank_total"] = (
    ranking["rank_RMSE"] +
    ranking["rank_MAE"] +
    ranking["rank_R2"]
)

ranking = ranking.sort_values(
    [
        "rank_total",
        "RMSE_medio"
    ]
)

print("\n" + "=" * 90)
print("RANKING DE APOYO")
print("=" * 90)

display(
    ranking[
        [
            "modelo",
            "MAE_medio",
            "RMSE_medio",
            "R2_medio",
            "R2_std",
            "R2_min",
            "rank_RMSE",
            "rank_MAE",
            "rank_R2",
            "rank_total"
        ]
    ]
)

# ============================================================
# 8. RESULTADOS POR AÑO EN FORMATO COMPARABLE
# ============================================================

print("\n" + "=" * 90)
print("RMSE POR AÑO")
print("=" * 90)

display(
    res.pivot(
        index="ano_validacion",
        columns="modelo",
        values="RMSE"
    ).round(2)
)

print("\n" + "=" * 90)
print("R² POR AÑO")
print("=" * 90)

display(
    res.pivot(
        index="ano_validacion",
        columns="modelo",
        values="R2"
    ).round(3)
)

# ============================================================
# 9. CONTROL METODOLÓGICO
# ============================================================

print("\n" + "=" * 90)
print("CONTROL METODOLÓGICO")
print("=" * 90)

print(
    "Años de validación:",
    validation_years
)

print(
    "2023 utilizado:",
    2023 in res["ano_validacion"].values
)

print(
    "2023 permanece reservado para test final."
)

# ============================================================
# 10. GUARDAR
# ============================================================

os.makedirs(
    RESULTADOS,
    exist_ok=True
)

F_RES_FOLDS = (
    f"{RESULTADOS}/"
    "paso80_modelos_folds_ERA5_DEFINITIVO.csv"
)

F_RES_RESUMEN = (
    f"{RESULTADOS}/"
    "paso80_modelos_resumen_ERA5_DEFINITIVO.csv"
)

res.to_csv(
    F_RES_FOLDS,
    index=False
)

ranking.to_csv(
    F_RES_RESUMEN,
    index=False
)

print("\nResultados guardados:")
print(F_RES_FOLDS)
print(F_RES_RESUMEN)

print("\nPASO 80 FINALIZADO")

# Comparación de ensembles

In [ ]:
# ============================================================
# PASO 81
# COMPARACIÓN DE MODELOS FINALISTAS Y ENSEMBLES
# 2023 PERMANECE BLOQUEADO
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import clone

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# 1. CARGAR DATASET DEFINITIVO
# ============================================================

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"
RESULTADOS = f"{ROOT}/05_Resultados"

F_DATASET = (
    f"{PROC}/dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

df = pd.read_csv(F_DATASET)

TARGET = "rendimiento_kg_ha"

EXCLUIR = [
    TARGET,
    "ano",
    "provincia_estandar"
]

features = [
    c for c in df.columns
    if c not in EXCLUIR
]

num_cols = [
    c for c in features
    if c != "cultivo"
]

cat_cols = ["cultivo"]


print("=" * 90)
print("PASO 81 - MODELOS FINALISTAS Y ENSEMBLES")
print("=" * 90)

print("Dataset:", df.shape)
print("Predictores:", len(features))
print("2023 reservado para test final: SÍ")


# ============================================================
# 2. PREPROCESADO
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            cat_cols
        )
    ]
)


# ============================================================
# 3. MODELOS FINALISTAS
# ============================================================

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

et = ExtraTreesRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

xgb = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.7,
    reg_alpha=0.01,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


# ============================================================
# 4. VALIDACIÓN TEMPORAL
# ============================================================

validation_years = [
    2020,
    2021,
    2022
]

resultados = []

predicciones_folds = []


for ano_val in validation_years:

    train = df[
        df["ano"] < ano_val
    ].copy()

    val = df[
        df["ano"] == ano_val
    ].copy()

    # Seguridad metodológica
    assert train["ano"].max() <= 2021
    assert ano_val <= 2022

    X_train = train[features]
    y_train = train[TARGET]

    X_val = val[features]
    y_val = val[TARGET]


    print("\n" + "=" * 90)
    print(
        f"VALIDACIÓN {ano_val} | "
        f"TRAIN {train['ano'].min()}-{train['ano'].max()} | "
        f"n_train={len(train)} | "
        f"n_val={len(val)}"
    )
    print("=" * 90)


    # ========================================================
    # 5. ENTRENAR MODELOS INDIVIDUALES
    # ========================================================

    pipes = {}

    for nombre, modelo in {
        "RandomForest": rf,
        "XGBoost": xgb,
        "ExtraTrees": et
    }.items():

        pipe = Pipeline(
            steps=[
                (
                    "preprocessor",
                    clone(preprocessor)
                ),
                (
                    "modelo",
                    clone(modelo)
                )
            ]
        )

        pipe.fit(
            X_train,
            y_train
        )

        pipes[nombre] = pipe


    # ========================================================
    # 6. PREDICCIONES INDIVIDUALES
    # ========================================================

    pred_rf = pipes["RandomForest"].predict(X_val)
    pred_xgb = pipes["XGBoost"].predict(X_val)
    pred_et = pipes["ExtraTrees"].predict(X_val)


    # ========================================================
    # 7. ENSEMBLES
    # ========================================================

    predicciones = {

        "RandomForest":
            pred_rf,

        "XGBoost":
            pred_xgb,

        "ExtraTrees":
            pred_et,

        # Promedio simple
        "RF_XGB":
            (pred_rf + pred_xgb) / 2,

        "RF_ET":
            (pred_rf + pred_et) / 2,

        "RF_XGB_ET":
            (pred_rf + pred_xgb + pred_et) / 3,

        # Ensemble ponderado:
        # mayor peso a RF por su estabilidad temporal
        "RF70_XGB30":
            0.70 * pred_rf +
            0.30 * pred_xgb
    }


    # ========================================================
    # 8. MÉTRICAS
    # ========================================================

    for nombre, pred in predicciones.items():

        mae = mean_absolute_error(
            y_val,
            pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                pred
            )
        )

        r2 = r2_score(
            y_val,
            pred
        )

        resultados.append({
            "modelo": nombre,
            "ano_validacion": ano_val,
            "n_train": len(train),
            "n_validacion": len(val),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

        print(
            f"{nombre:16s} | "
            f"MAE={mae:8.2f} | "
            f"RMSE={rmse:8.2f} | "
            f"R²={r2:7.3f}"
        )


    # ========================================================
    # 9. GUARDAR PREDICCIONES OUT-OF-FOLD
    # ========================================================

    fold_preds = pd.DataFrame({

        "ano_validacion":
            ano_val,

        "provincia_estandar":
            val["provincia_estandar"].values,

        "cultivo":
            val["cultivo"].values,

        "observado":
            y_val.values,

        "pred_RF":
            pred_rf,

        "pred_XGB":
            pred_xgb,

        "pred_ET":
            pred_et,

        "pred_RF_XGB":
            predicciones["RF_XGB"],

        "pred_RF_ET":
            predicciones["RF_ET"],

        "pred_RF_XGB_ET":
            predicciones["RF_XGB_ET"],

        "pred_RF70_XGB30":
            predicciones["RF70_XGB30"]
    })

    predicciones_folds.append(
        fold_preds
    )


# ============================================================
# 10. RESULTADOS POR FOLD
# ============================================================

res = pd.DataFrame(
    resultados
)

print("\n" + "=" * 90)
print("RESULTADOS POR FOLD")
print("=" * 90)

display(
    res.round(3)
)


# ============================================================
# 11. COMPARACIÓN GLOBAL
# ============================================================

resumen = (
    res
    .groupby("modelo")
    .agg(

        MAE_medio=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        RMSE_medio=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),

        R2_medio=("R2", "mean"),
        R2_std=("R2", "std"),

        R2_min=("R2", "min"),
        R2_max=("R2", "max")
    )
    .sort_values(
        ["RMSE_medio", "MAE_medio"]
    )
    .round(3)
)

print("\n" + "=" * 90)
print("COMPARACIÓN GLOBAL")
print("=" * 90)

display(resumen)


# ============================================================
# 12. RMSE POR AÑO
# ============================================================

print("\n" + "=" * 90)
print("RMSE POR AÑO")
print("=" * 90)

tabla_rmse = (
    res
    .pivot(
        index="ano_validacion",
        columns="modelo",
        values="RMSE"
    )
)

display(
    tabla_rmse.round(2)
)


# ============================================================
# 13. R² POR AÑO
# ============================================================

print("\n" + "=" * 90)
print("R² POR AÑO")
print("=" * 90)

tabla_r2 = (
    res
    .pivot(
        index="ano_validacion",
        columns="modelo",
        values="R2"
    )
)

display(
    tabla_r2.round(3)
)


# ============================================================
# 14. RANKING
# ============================================================

ranking = resumen.reset_index().copy()

ranking["rank_RMSE"] = (
    ranking["RMSE_medio"]
    .rank(method="min")
)

ranking["rank_MAE"] = (
    ranking["MAE_medio"]
    .rank(method="min")
)

ranking["rank_R2"] = (
    ranking["R2_medio"]
    .rank(
        method="min",
        ascending=False
    )
)

ranking["rank_estabilidad"] = (
    ranking["RMSE_std"]
    .rank(method="min")
)

ranking["rank_total"] = (
    ranking["rank_RMSE"] +
    ranking["rank_MAE"] +
    ranking["rank_R2"] +
    ranking["rank_estabilidad"]
)

ranking = ranking.sort_values(
    [
        "rank_total",
        "RMSE_medio"
    ]
)


print("\n" + "=" * 90)
print("RANKING DE APOYO")
print("=" * 90)

display(
    ranking[
        [
            "modelo",
            "MAE_medio",
            "RMSE_medio",
            "RMSE_std",
            "R2_medio",
            "R2_std",
            "R2_min",
            "rank_total"
        ]
    ]
)


# ============================================================
# 15. CONTROL METODOLÓGICO
# ============================================================

print("\n" + "=" * 90)
print("CONTROL METODOLÓGICO")
print("=" * 90)

print(
    "Años evaluados:",
    validation_years
)

print(
    "2023 utilizado:",
    2023 in res["ano_validacion"].values
)

print(
    "\nTEST 2023 CONTINÚA BLOQUEADO."
)


# ============================================================
# 16. GUARDAR
# ============================================================

os.makedirs(
    RESULTADOS,
    exist_ok=True
)

F_RES = (
    f"{RESULTADOS}/"
    "paso81_ensembles_validacion_ERA5_DEFINITIVO.csv"
)

F_PRED = (
    f"{RESULTADOS}/"
    "paso81_predicciones_OOF_ERA5_DEFINITIVO.csv"
)

res.to_csv(
    F_RES,
    index=False
)

pd.concat(
    predicciones_folds,
    ignore_index=True
).to_csv(
    F_PRED,
    index=False
)

print("\nArchivos guardados:")
print(F_RES)
print(F_PRED)

print("\nPASO 81 FINALIZADO")

# Tuning temporal final de RF y XGBoost

In [ ]:
# ============================================================
# PASO 82
# TUNING TEMPORAL FINAL - RANDOM FOREST Y XGBOOST
# 2023 NO SE UTILIZA
# ============================================================

import os
import itertools
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# 1. DATOS
# ============================================================

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"
RESULTADOS = f"{ROOT}/05_Resultados"

F_DATASET = (
    f"{PROC}/dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

df = pd.read_csv(F_DATASET)

TARGET = "rendimiento_kg_ha"

EXCLUIR = [
    TARGET,
    "ano",
    "provincia_estandar"
]

features = [
    c for c in df.columns
    if c not in EXCLUIR
]

num_cols = [
    c for c in features
    if c != "cultivo"
]

cat_cols = ["cultivo"]

# Solamente datos disponibles para desarrollo
dev = df[df["ano"] <= 2022].copy()

validation_years = [2020, 2021, 2022]

print("=" * 90)
print("PASO 82 - TUNING TEMPORAL FINAL")
print("=" * 90)

print("Dataset desarrollo 2017-2022:", dev.shape)
print("Test 2023 utilizado: NO")
print("Validaciones:", validation_years)

# ============================================================
# 2. PREPROCESADOR
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            cat_cols
        )
    ]
)

# ============================================================
# 3. FUNCIÓN DE EVALUACIÓN TEMPORAL
# ============================================================

def evaluar_configuracion(modelo, nombre_modelo, parametros):

    resultados_folds = []

    for ano_val in validation_years:

        train = dev[
            dev["ano"] < ano_val
        ].copy()

        val = dev[
            dev["ano"] == ano_val
        ].copy()

        X_train = train[features]
        y_train = train[TARGET]

        X_val = val[features]
        y_val = val[TARGET]

        pipe = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("modelo", modelo)
            ]
        )

        pipe.fit(X_train, y_train)

        pred = pipe.predict(X_val)

        mae = mean_absolute_error(
            y_val, pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val, pred
            )
        )

        r2 = r2_score(
            y_val, pred
        )

        resultados_folds.append({
            "ano": ano_val,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    folds = pd.DataFrame(resultados_folds)

    return {
        "modelo": nombre_modelo,
        **parametros,

        "MAE_medio":
            folds["MAE"].mean(),

        "RMSE_medio":
            folds["RMSE"].mean(),

        "RMSE_std":
            folds["RMSE"].std(),

        "R2_medio":
            folds["R2"].mean(),

        "R2_std":
            folds["R2"].std(),

        "R2_min":
            folds["R2"].min(),

        "R2_2020":
            folds.loc[
                folds["ano"] == 2020,
                "R2"
            ].iloc[0],

        "R2_2021":
            folds.loc[
                folds["ano"] == 2021,
                "R2"
            ].iloc[0],

        "R2_2022":
            folds.loc[
                folds["ano"] == 2022,
                "R2"
            ].iloc[0]
    }


# ============================================================
# 4. RANDOM FOREST
# ============================================================

print("\n" + "=" * 90)
print("TUNING RANDOM FOREST")
print("=" * 90)

grid_rf = {
    "n_estimators": [500, 800],
    "max_depth": [None, 15, 25],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5, 0.8],
    "bootstrap": [True, False]
}

# Para evitar una búsqueda excesiva,
# utilizamos una selección reproducible de configuraciones.

combinaciones_rf = list(
    itertools.product(
        grid_rf["n_estimators"],
        grid_rf["max_depth"],
        grid_rf["min_samples_leaf"],
        grid_rf["max_features"],
        grid_rf["bootstrap"]
    )
)

rng = np.random.default_rng(42)

idx_rf = rng.choice(
    len(combinaciones_rf),
    size=min(30, len(combinaciones_rf)),
    replace=False
)

resultados_rf = []

for i, idx in enumerate(idx_rf, 1):

    (
        n_estimators,
        max_depth,
        min_samples_leaf,
        max_features,
        bootstrap
    ) = combinaciones_rf[idx]

    parametros = {
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "max_features": max_features,
        "bootstrap": bootstrap
    }

    modelo = RandomForestRegressor(
        **parametros,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    )

    resultado = evaluar_configuracion(
        modelo,
        "RandomForest",
        parametros
    )

    resultados_rf.append(resultado)

    print(
        f"RF {i:02d}/{len(idx_rf)} | "
        f"RMSE={resultado['RMSE_medio']:.2f} | "
        f"R²={resultado['R2_medio']:.3f}"
    )


df_rf = (
    pd.DataFrame(resultados_rf)
    .sort_values(
        ["RMSE_medio", "MAE_medio"]
    )
    .reset_index(drop=True)
)

print("\nTOP 10 RANDOM FOREST")

display(
    df_rf.head(10).round(3)
)


# ============================================================
# 5. XGBOOST
# ============================================================

print("\n" + "=" * 90)
print("TUNING XGBOOST")
print("=" * 90)

grid_xgb = {
    "n_estimators": [200, 400, 600],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05, 0.08],
    "min_child_weight": [3, 5, 8],
    "subsample": [0.7, 0.9],
    "colsample_bytree": [0.5, 0.7, 0.9]
}

combinaciones_xgb = list(
    itertools.product(
        grid_xgb["n_estimators"],
        grid_xgb["max_depth"],
        grid_xgb["learning_rate"],
        grid_xgb["min_child_weight"],
        grid_xgb["subsample"],
        grid_xgb["colsample_bytree"]
    )
)

idx_xgb = rng.choice(
    len(combinaciones_xgb),
    size=min(35, len(combinaciones_xgb)),
    replace=False
)

resultados_xgb = []

for i, idx in enumerate(idx_xgb, 1):

    (
        n_estimators,
        max_depth,
        learning_rate,
        min_child_weight,
        subsample,
        colsample_bytree
    ) = combinaciones_xgb[idx]

    parametros = {
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "min_child_weight": min_child_weight,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree
    }

    modelo = XGBRegressor(
        **parametros,
        reg_alpha=0.01,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    resultado = evaluar_configuracion(
        modelo,
        "XGBoost",
        parametros
    )

    resultados_xgb.append(resultado)

    print(
        f"XGB {i:02d}/{len(idx_xgb)} | "
        f"RMSE={resultado['RMSE_medio']:.2f} | "
        f"R²={resultado['R2_medio']:.3f}"
    )


df_xgb = (
    pd.DataFrame(resultados_xgb)
    .sort_values(
        ["RMSE_medio", "MAE_medio"]
    )
    .reset_index(drop=True)
)

print("\nTOP 10 XGBOOST")

display(
    df_xgb.head(10).round(3)
)


# ============================================================
# 6. MEJORES CONFIGURACIONES
# ============================================================

best_rf = df_rf.iloc[0]

best_xgb = df_xgb.iloc[0]

print("\n" + "=" * 90)
print("MEJOR RANDOM FOREST")
print("=" * 90)

display(
    best_rf.to_frame("valor")
)

print("\n" + "=" * 90)
print("MEJOR XGBOOST")
print("=" * 90)

display(
    best_xgb.to_frame("valor")
)


# ============================================================
# 7. GUARDAR
# ============================================================

os.makedirs(
    RESULTADOS,
    exist_ok=True
)

F_RF = (
    f"{RESULTADOS}/"
    "paso82_tuning_RF_ERA5_DEFINITIVO.csv"
)

F_XGB = (
    f"{RESULTADOS}/"
    "paso82_tuning_XGB_ERA5_DEFINITIVO.csv"
)

df_rf.to_csv(
    F_RF,
    index=False
)

df_xgb.to_csv(
    F_XGB,
    index=False
)

print("\nArchivos guardados:")
print(F_RF)
print(F_XGB)

print("\n" + "=" * 90)
print("CONTROL METODOLÓGICO")
print("=" * 90)

print("Datos usados para tuning: 2017-2022")
print("2023 utilizado: NO")
print("TEST 2023 CONTINÚA BLOQUEADO.")

print("\nPASO 82 FINALIZADO")

# Revalidación final de RF, XGB y ensembles

In [ ]:
# ============================================================
# PASO 83
# REVALIDACIÓN FINAL DE MODELOS SELECCIONADOS
# 2023 SIGUE BLOQUEADO
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# 1. DATOS
# ============================================================

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"
RESULTADOS = f"{ROOT}/05_Resultados"

F_DATASET = (
    f"{PROC}/dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

df = pd.read_csv(F_DATASET)

TARGET = "rendimiento_kg_ha"

EXCLUIR = [
    TARGET,
    "ano",
    "provincia_estandar"
]

features = [
    c for c in df.columns
    if c not in EXCLUIR
]

num_cols = [
    c for c in features
    if c != "cultivo"
]

cat_cols = ["cultivo"]

validation_years = [2020, 2021, 2022]

print("=" * 90)
print("PASO 83 - REVALIDACIÓN FINAL")
print("=" * 90)

print("Dataset:", df.shape)
print("2023 utilizado: NO")


# ============================================================
# 2. PREPROCESADO
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            cat_cols
        )
    ]
)


# ============================================================
# 3. MODELOS OPTIMIZADOS
# ============================================================

rf = RandomForestRegressor(
    n_estimators=800,
    max_depth=15,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features="sqrt",
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

xgb = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.08,
    min_child_weight=3,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.01,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


# ============================================================
# 4. VALIDACIÓN TEMPORAL
# ============================================================

resultados = []

for ano_val in validation_years:

    train = df[df["ano"] < ano_val].copy()
    val = df[df["ano"] == ano_val].copy()

    X_train = train[features]
    y_train = train[TARGET]

    X_val = val[features]
    y_val = val[TARGET]

    print("\n" + "=" * 90)
    print(
        f"VALIDACIÓN {ano_val} | "
        f"TRAIN {train['ano'].min()}-{train['ano'].max()}"
    )
    print("=" * 90)

    pipe_rf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("modelo", rf)
        ]
    )

    pipe_xgb = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("modelo", xgb)
        ]
    )

    pipe_rf.fit(X_train, y_train)
    pipe_xgb.fit(X_train, y_train)

    pred_rf = pipe_rf.predict(X_val)
    pred_xgb = pipe_xgb.predict(X_val)

    predicciones = {
        "RF_tuned":
            pred_rf,

        "XGB_tuned":
            pred_xgb,

        "RF_XGB_50_50":
            0.50 * pred_rf +
            0.50 * pred_xgb,

        "RF70_XGB30":
            0.70 * pred_rf +
            0.30 * pred_xgb
    }

    for nombre, pred in predicciones.items():

        mae = mean_absolute_error(
            y_val,
            pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                pred
            )
        )

        r2 = r2_score(
            y_val,
            pred
        )

        resultados.append({
            "modelo": nombre,
            "ano_validacion": ano_val,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

        print(
            f"{nombre:16s} | "
            f"MAE={mae:8.2f} | "
            f"RMSE={rmse:8.2f} | "
            f"R²={r2:7.3f}"
        )


# ============================================================
# 5. RESUMEN
# ============================================================

res = pd.DataFrame(resultados)

print("\n" + "=" * 90)
print("RESULTADOS POR AÑO")
print("=" * 90)

display(
    res.round(3)
)

resumen = (
    res.groupby("modelo")
    .agg(
        MAE_medio=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_medio=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        R2_medio=("R2", "mean"),
        R2_std=("R2", "std"),
        R2_min=("R2", "min"),
        R2_max=("R2", "max")
    )
    .sort_values(
        ["RMSE_medio", "MAE_medio"]
    )
    .round(3)
)

print("\n" + "=" * 90)
print("COMPARACIÓN FINAL")
print("=" * 90)

display(resumen)


# ============================================================
# 6. R² POR AÑO
# ============================================================

print("\n" + "=" * 90)
print("R² POR AÑO")
print("=" * 90)

display(
    res.pivot(
        index="ano_validacion",
        columns="modelo",
        values="R2"
    ).round(3)
)


# ============================================================
# 7. CONTROL
# ============================================================

print("\n" + "=" * 90)
print("CONTROL METODOLÓGICO")
print("=" * 90)

print("Años utilizados:", validation_years)
print("2023 utilizado:", 2023 in res["ano_validacion"].values)
print("TEST 2023 CONTINÚA BLOQUEADO.")


# ============================================================
# 8. GUARDAR
# ============================================================

F_SALIDA = (
    f"{RESULTADOS}/"
    "paso83_revalidacion_final_ERA5_DEFINITIVO.csv"
)

res.to_csv(
    F_SALIDA,
    index=False
)

print("\nGuardado:")
print(F_SALIDA)

print("\nPASO 83 FINALIZADO")

# Test final independiente 2023

In [ ]:
# ============================================================
# PASO 84
# TEST FINAL INDEPENDIENTE 2023
#
# Modelos preseleccionados exclusivamente con 2017-2022:
#   1. RF_tuned
#   2. XGB_tuned
#   3. RF_XGB_50_50
#   4. RF70_XGB30
#
# IMPORTANTE:
# Tras abrir 2023 NO realizar tuning adicional.
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# 1. CARGAR DATASET DEFINITIVO
# ============================================================

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"
RESULTADOS = f"{ROOT}/05_Resultados"

F_DATASET = (
    f"{PROC}/dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

df = pd.read_csv(F_DATASET)

TARGET = "rendimiento_kg_ha"

EXCLUIR = [
    TARGET,
    "ano",
    "provincia_estandar"
]

features = [
    c for c in df.columns
    if c not in EXCLUIR
]

num_cols = [
    c for c in features
    if c != "cultivo"
]

cat_cols = ["cultivo"]


# ============================================================
# 2. SEPARACIÓN FINAL
# ============================================================

train = df[
    df["ano"] <= 2022
].copy()

test = df[
    df["ano"] == 2023
].copy()

print("=" * 90)
print("PASO 84 - TEST FINAL INDEPENDIENTE 2023")
print("=" * 90)

print("\nTRAIN:")
print("Años:", sorted(train["ano"].unique()))
print("n:", len(train))

print("\nTEST:")
print("Años:", sorted(test["ano"].unique()))
print("n:", len(test))

assert train["ano"].max() == 2022
assert test["ano"].nunique() == 1
assert test["ano"].iloc[0] == 2023


X_train = train[features]
y_train = train[TARGET]

X_test = test[features]
y_test = test[TARGET]


# ============================================================
# 3. PREPROCESADO
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            cat_cols
        )
    ]
)


# ============================================================
# 4. MODELOS FIJADOS EN PASO 82
# ============================================================

rf = RandomForestRegressor(
    n_estimators=800,
    max_depth=15,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features="sqrt",
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

xgb = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.08,
    min_child_weight=3,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.01,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


# ============================================================
# 5. ENTRENAR CON TODO 2017-2022
# ============================================================

pipe_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("modelo", rf)
    ]
)

pipe_xgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("modelo", xgb)
    ]
)

print("\nEntrenando modelos finales...")

pipe_rf.fit(
    X_train,
    y_train
)

pipe_xgb.fit(
    X_train,
    y_train
)

print("Random Forest entrenado.")
print("XGBoost entrenado.")


# ============================================================
# 6. PREDICCIONES 2023
# ============================================================

pred_rf = pipe_rf.predict(X_test)
pred_xgb = pipe_xgb.predict(X_test)

predicciones = {

    "RF_tuned":
        pred_rf,

    "XGB_tuned":
        pred_xgb,

    "RF_XGB_50_50":
        0.50 * pred_rf +
        0.50 * pred_xgb,

    "RF70_XGB30":
        0.70 * pred_rf +
        0.30 * pred_xgb
}


# ============================================================
# 7. MÉTRICAS TEST 2023
# ============================================================

resultados = []

for nombre, pred in predicciones.items():

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )

    r2 = r2_score(
        y_test,
        pred
    )

    sesgo = np.mean(
        pred - y_test.values
    )

    resultados.append({
        "modelo": nombre,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "sesgo_pred_menos_obs": sesgo
    })


resultados_test = (
    pd.DataFrame(resultados)
    .sort_values("RMSE")
    .reset_index(drop=True)
)


print("\n" + "=" * 90)
print("RESULTADOS FINALES - TEST 2023")
print("=" * 90)

display(
    resultados_test.round(3)
)


# ============================================================
# 8. RESULTADOS POR CULTIVO
# ============================================================

resultados_cultivo = []

for nombre, pred in predicciones.items():

    tmp = test[
        [
            "cultivo",
            "provincia_estandar",
            TARGET
        ]
    ].copy()

    tmp["predicho"] = pred

    for cultivo, d in tmp.groupby("cultivo"):

        obs = d[TARGET]
        pr = d["predicho"]

        resultados_cultivo.append({

            "modelo": nombre,
            "cultivo": cultivo,
            "n": len(d),

            "observado_medio":
                obs.mean(),

            "predicho_medio":
                pr.mean(),

            "sesgo":
                (pr - obs).mean(),

            "MAE":
                mean_absolute_error(
                    obs,
                    pr
                ),

            "RMSE":
                np.sqrt(
                    mean_squared_error(
                        obs,
                        pr
                    )
                ),

            "R2":
                r2_score(
                    obs,
                    pr
                )
        })


res_cultivo = pd.DataFrame(
    resultados_cultivo
)


print("\n" + "=" * 90)
print("RESULTADOS 2023 POR CULTIVO")
print("=" * 90)

display(
    res_cultivo.round(3)
)


# ============================================================
# 9. GUARDAR PREDICCIONES INDIVIDUALES
# ============================================================

pred_test = test[
    [
        "ano",
        "provincia_estandar",
        "cultivo",
        TARGET
    ]
].copy()

pred_test = pred_test.rename(
    columns={
        TARGET: "observado"
    }
)

pred_test["pred_RF_tuned"] = (
    pred_rf
)

pred_test["pred_XGB_tuned"] = (
    pred_xgb
)

pred_test["pred_RF_XGB_50_50"] = (
    predicciones["RF_XGB_50_50"]
)

pred_test["pred_RF70_XGB30"] = (
    predicciones["RF70_XGB30"]
)


# ============================================================
# 10. COMPARACIÓN CON VALIDACIÓN TEMPORAL
# ============================================================

validacion = pd.DataFrame({
    "modelo": [
        "RF_tuned",
        "XGB_tuned",
        "RF_XGB_50_50",
        "RF70_XGB30"
    ],

    "RMSE_validacion_2020_2022": [
        705.102,
        700.216,
        682.161,
        686.803
    ],

    "R2_validacion_2020_2022": [
        0.449,
        0.356,
        0.433,
        0.446
    ]
})

comparacion = validacion.merge(
    resultados_test[
        [
            "modelo",
            "RMSE",
            "R2"
        ]
    ],
    on="modelo",
    how="left"
)

comparacion = comparacion.rename(
    columns={
        "RMSE":
            "RMSE_test_2023",

        "R2":
            "R2_test_2023"
    }
)


print("\n" + "=" * 90)
print("VALIDACIÓN TEMPORAL vs TEST EXTERNO 2023")
print("=" * 90)

display(
    comparacion.round(3)
)


# ============================================================
# 11. DISTRIBUCIÓN OBSERVADO / PREDICHO
# ============================================================

print("\n" + "=" * 90)
print("DISTRIBUCIÓN OBSERVADA 2023")
print("=" * 90)

print(
    y_test.describe().round(2)
)

print("\nMedias predichas:")

for nombre, pred in predicciones.items():

    print(
        f"{nombre:16s}: "
        f"{np.mean(pred):.2f} kg/ha"
    )


# ============================================================
# 12. GUARDAR
# ============================================================

os.makedirs(
    RESULTADOS,
    exist_ok=True
)

F_METRICAS = (
    f"{RESULTADOS}/"
    "paso84_test_final_2023_metricas.csv"
)

F_CULTIVOS = (
    f"{RESULTADOS}/"
    "paso84_test_final_2023_por_cultivo.csv"
)

F_PREDICCIONES = (
    f"{RESULTADOS}/"
    "paso84_test_final_2023_predicciones.csv"
)

F_COMPARACION = (
    f"{RESULTADOS}/"
    "paso84_validacion_vs_test_2023.csv"
)

resultados_test.to_csv(
    F_METRICAS,
    index=False
)

res_cultivo.to_csv(
    F_CULTIVOS,
    index=False
)

pred_test.to_csv(
    F_PREDICCIONES,
    index=False
)

comparacion.to_csv(
    F_COMPARACION,
    index=False
)


# ============================================================
# 13. CONTROL METODOLÓGICO FINAL
# ============================================================

print("\n" + "=" * 90)
print("CONTROL METODOLÓGICO FINAL")
print("=" * 90)

print(
    "TRAIN:",
    sorted(train["ano"].unique())
)

print(
    "TEST:",
    sorted(test["ano"].unique())
)

print("n TRAIN:", len(train))
print("n TEST:", len(test))

print(
    "\n2023 se ha utilizado exclusivamente "
    "como TEST FINAL."
)

print(
    "A partir de este punto NO realizar "
    "tuning, selección de variables ni "
    "modificación de ensembles usando 2023."
)

print("\nPASO 84 FINALIZADO")

# Diagnóstico final del rendimiento

In [ ]:
# ============================================================
# PASO 85
# DIAGNÓSTICO FINAL DEL TARGET Y BASELINES
# NO SE MODIFICA NINGÚN MODELO
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"
RESULTADOS = f"{ROOT}/05_Resultados"

F_DATASET = (
    f"{PROC}/dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

df = pd.read_csv(F_DATASET)

TARGET = "rendimiento_kg_ha"

train = df[df["ano"] <= 2022].copy()
test = df[df["ano"] == 2023].copy()

# ============================================================
# 1. DISTRIBUCIÓN DEL RENDIMIENTO POR AÑO
# ============================================================

resumen_ano = (
    df.groupby("ano")[TARGET]
      .agg(
          n="count",
          media="mean",
          mediana="median",
          std="std",
          minimo="min",
          maximo="max"
      )
      .round(2)
)

print("=" * 90)
print("RENDIMIENTO POR AÑO")
print("=" * 90)

display(resumen_ano)


# ============================================================
# 2. RENDIMIENTO POR AÑO Y CULTIVO
# ============================================================

resumen_cultivo = (
    df.groupby(["ano", "cultivo"])[TARGET]
      .agg(
          n="count",
          media="mean",
          mediana="median",
          std="std"
      )
      .round(2)
)

print("\n" + "=" * 90)
print("RENDIMIENTO POR AÑO Y CULTIVO")
print("=" * 90)

display(resumen_cultivo)


# ============================================================
# 3. COMPARACIÓN 2023 vs 2017-2022
# ============================================================

media_train = train[TARGET].mean()
media_2023 = test[TARGET].mean()

mediana_train = train[TARGET].median()
mediana_2023 = test[TARGET].median()

print("\n" + "=" * 90)
print("2023 vs PERIODO DE ENTRENAMIENTO")
print("=" * 90)

print(f"Media 2017-2022:   {media_train:.2f} kg/ha")
print(f"Media 2023:        {media_2023:.2f} kg/ha")
print(
    f"Diferencia:        "
    f"{media_2023-media_train:.2f} kg/ha"
)

print(
    f"Cambio relativo:   "
    f"{(media_2023/media_train - 1)*100:.1f} %"
)

print()

print(f"Mediana 2017-2022: {mediana_train:.2f} kg/ha")
print(f"Mediana 2023:      {mediana_2023:.2f} kg/ha")


# ============================================================
# 4. BASELINE 1:
#    MEDIA GLOBAL HISTÓRICA 2017-2022
# ============================================================

y_test = test[TARGET].values

pred_media_global = np.full(
    len(test),
    media_train
)

# ============================================================
# 5. BASELINE 2:
#    MEDIA HISTÓRICA POR CULTIVO
# ============================================================

medias_cultivo_train = (
    train.groupby("cultivo")[TARGET]
         .mean()
)

pred_media_cultivo = (
    test["cultivo"]
    .map(medias_cultivo_train)
    .values
)


# ============================================================
# 6. FUNCIÓN MÉTRICAS
# ============================================================

def metricas(nombre, pred):

    return {
        "modelo": nombre,
        "MAE": mean_absolute_error(
            y_test,
            pred
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                y_test,
                pred
            )
        ),
        "R2": r2_score(
            y_test,
            pred
        ),
        "sesgo_pred_menos_obs":
            np.mean(pred - y_test)
    }


baselines = pd.DataFrame([
    metricas(
        "Baseline_media_global",
        pred_media_global
    ),
    metricas(
        "Baseline_media_por_cultivo",
        pred_media_cultivo
    )
])

print("\n" + "=" * 90)
print("BASELINES HISTÓRICOS - TEST 2023")
print("=" * 90)

display(
    baselines.round(3)
)


# ============================================================
# 7. COMPARACIÓN CON MODELOS FINALES
# ============================================================

F_MODELOS = (
    f"{RESULTADOS}/"
    "paso84_test_final_2023_metricas.csv"
)

modelos_finales = pd.read_csv(
    F_MODELOS
)

comparacion = pd.concat(
    [
        modelos_finales[
            [
                "modelo",
                "MAE",
                "RMSE",
                "R2",
                "sesgo_pred_menos_obs"
            ]
        ],
        baselines
    ],
    ignore_index=True
)

comparacion = comparacion.sort_values(
    "RMSE"
).reset_index(drop=True)

print("\n" + "=" * 90)
print("MODELOS vs BASELINES - TEST 2023")
print("=" * 90)

display(
    comparacion.round(3)
)


# ============================================================
# 8. GUARDAR
# ============================================================

F_SALIDA = (
    f"{RESULTADOS}/"
    "paso85_diagnostico_target_y_baselines.csv"
)

comparacion.to_csv(
    F_SALIDA,
    index=False
)

print("\nGuardado:")
print(F_SALIDA)

print("\nPASO 85 FINALIZADO")

# Diagnóstico final de errores 2023

In [ ]:
# ============================================================
# PASO 86
# DIAGNÓSTICO FINAL DE ERRORES - TEST 2023
# Análisis descriptivo. NO tuning.
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

ROOT = "/content/TFM_Rendimiento_cultivos"
RESULTADOS = f"{ROOT}/05_Resultados"

F_PRED = (
    f"{RESULTADOS}/"
    "paso84_test_final_2023_predicciones.csv"
)

pred = pd.read_csv(F_PRED)

print("=" * 90)
print("PASO 86 - DIAGNÓSTICO FINAL DE ERRORES 2023")
print("=" * 90)

print("\nDimensiones:", pred.shape)
print("Observaciones:", len(pred))

# ============================================================
# 1. COLUMNAS DE PREDICCIÓN
# ============================================================

modelos = {
    "RF_tuned": "pred_RF_tuned",
    "XGB_tuned": "pred_XGB_tuned",
    "RF_XGB_50_50": "pred_RF_XGB_50_50",
    "RF70_XGB30": "pred_RF70_XGB30"
}

# ============================================================
# 2. RESIDUOS
# ============================================================

for nombre, col in modelos.items():

    pred[f"residuo_{nombre}"] = (
        pred["observado"] - pred[col]
    )

    pred[f"error_abs_{nombre}"] = (
        pred[f"residuo_{nombre}"].abs()
    )


# ============================================================
# 3. RESUMEN GLOBAL DE RESIDUOS
# ============================================================

residuos_global = []

for nombre, col in modelos.items():

    resid = pred["observado"] - pred[col]

    residuos_global.append({
        "modelo": nombre,
        "residuo_medio_obs_menos_pred": resid.mean(),
        "residuo_mediana": resid.median(),
        "std_residuo": resid.std(),
        "error_abs_mediano": resid.abs().median(),
        "pct_sobreestimaciones":
            (pred[col] > pred["observado"]).mean() * 100,
        "pct_subestimaciones":
            (pred[col] < pred["observado"]).mean() * 100
    })

res_global = pd.DataFrame(residuos_global)

print("\n" + "=" * 90)
print("RESUMEN DE RESIDUOS")
print("=" * 90)

display(
    res_global.round(2)
)


# ============================================================
# 4. ERROR POR CULTIVO
# ============================================================

resultados_cultivo = []

for nombre, col in modelos.items():

    for cultivo, d in pred.groupby("cultivo"):

        obs = d["observado"]
        pr = d[col]

        resultados_cultivo.append({
            "modelo": nombre,
            "cultivo": cultivo,
            "n": len(d),

            "observado_medio":
                obs.mean(),

            "predicho_medio":
                pr.mean(),

            "MAE":
                mean_absolute_error(obs, pr),

            "RMSE":
                np.sqrt(
                    mean_squared_error(obs, pr)
                ),

            "R2":
                r2_score(obs, pr),

            "sesgo_pred_menos_obs":
                (pr - obs).mean()
        })

res_cultivo = pd.DataFrame(
    resultados_cultivo
)

print("\n" + "=" * 90)
print("ERROR POR CULTIVO")
print("=" * 90)

display(
    res_cultivo.round(2)
)


# ============================================================
# 5. ERROR SEGÚN NIVEL DE RENDIMIENTO
# ============================================================

pred["grupo_rendimiento"] = pd.qcut(
    pred["observado"],
    q=4,
    labels=[
        "Q1_muy_bajo",
        "Q2_bajo",
        "Q3_medio",
        "Q4_alto"
    ],
    duplicates="drop"
)

resultados_grupos = []

for nombre, col in modelos.items():

    for grupo, d in pred.groupby(
        "grupo_rendimiento",
        observed=True
    ):

        resultados_grupos.append({
            "modelo": nombre,
            "grupo_rendimiento": grupo,
            "n": len(d),

            "observado_medio":
                d["observado"].mean(),

            "predicho_medio":
                d[col].mean(),

            "MAE":
                mean_absolute_error(
                    d["observado"],
                    d[col]
                ),

            "sesgo_pred_menos_obs":
                (d[col] - d["observado"]).mean()
        })

res_grupos = pd.DataFrame(
    resultados_grupos
)

print("\n" + "=" * 90)
print("ERROR SEGÚN NIVEL DE RENDIMIENTO OBSERVADO")
print("=" * 90)

display(
    res_grupos.round(2)
)


# ============================================================
# 6. PROVINCIAS CON MAYOR ERROR - RF Y XGB
# ============================================================

for nombre in [
    "RF_tuned",
    "XGB_tuned"
]:

    col_error = f"error_abs_{nombre}"
    col_pred = modelos[nombre]

    top = (
        pred[
            [
                "provincia_estandar",
                "cultivo",
                "observado",
                col_pred,
                col_error
            ]
        ]
        .sort_values(
            col_error,
            ascending=False
        )
        .head(15)
    )

    print("\n" + "=" * 90)
    print(
        f"15 MAYORES ERRORES ABSOLUTOS - {nombre}"
    )
    print("=" * 90)

    display(
        top.round(2)
    )


# ============================================================
# 7. OBSERVADO vs PREDICHO
# ============================================================

for nombre in [
    "RF_tuned",
    "XGB_tuned"
]:

    col = modelos[nombre]

    plt.figure(figsize=(7, 6))

    plt.scatter(
        pred["observado"],
        pred[col],
        alpha=0.7
    )

    limite_min = min(
        pred["observado"].min(),
        pred[col].min()
    )

    limite_max = max(
        pred["observado"].max(),
        pred[col].max()
    )

    plt.plot(
        [limite_min, limite_max],
        [limite_min, limite_max],
        linestyle="--"
    )

    plt.xlabel(
        "Rendimiento observado (kg/ha)"
    )

    plt.ylabel(
        "Rendimiento predicho (kg/ha)"
    )

    plt.title(
        f"Observado vs predicho - {nombre} - 2023"
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# 8. RESIDUOS vs OBSERVADO
# ============================================================

for nombre in [
    "RF_tuned",
    "XGB_tuned"
]:

    resid = (
        pred["observado"] -
        pred[modelos[nombre]]
    )

    plt.figure(figsize=(7, 5))

    plt.scatter(
        pred["observado"],
        resid,
        alpha=0.7
    )

    plt.axhline(
        0,
        linestyle="--"
    )

    plt.xlabel(
        "Rendimiento observado (kg/ha)"
    )

    plt.ylabel(
        "Residuo: observado - predicho (kg/ha)"
    )

    plt.title(
        f"Residuos vs rendimiento observado - {nombre}"
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# 9. GUARDAR RESULTADOS
# ============================================================

F_RESIDUOS = (
    f"{RESULTADOS}/"
    "paso86_residuos_test_2023.csv"
)

F_CULTIVO = (
    f"{RESULTADOS}/"
    "paso86_residuos_por_cultivo.csv"
)

F_GRUPOS = (
    f"{RESULTADOS}/"
    "paso86_error_por_nivel_rendimiento.csv"
)

pred.to_csv(
    F_RESIDUOS,
    index=False
)

res_cultivo.to_csv(
    F_CULTIVO,
    index=False
)

res_grupos.to_csv(
    F_GRUPOS,
    index=False
)

print("\nArchivos guardados:")
print(F_RESIDUOS)
print(F_CULTIVO)
print(F_GRUPOS)

print("\nPASO 86 FINALIZADO")

# Permutation Importance temporal definitiva


In [ ]:
# ============================================================
# PASO 87
# PERMUTATION IMPORTANCE TEMPORAL DEFINITIVA
# RANDOM FOREST OPTIMIZADO
#
# Se calcula independientemente en:
# 2020, 2021 y 2022
#
# 2023 NO SE UTILIZA
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

# ============================================================
# 1. CARGAR DATASET DEFINITIVO
# ============================================================

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"
RESULTADOS = f"{ROOT}/05_Resultados"

F_DATASET = (
    f"{PROC}/dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

df = pd.read_csv(F_DATASET)

TARGET = "rendimiento_kg_ha"

EXCLUIR = [
    TARGET,
    "ano",
    "provincia_estandar"
]

features = [
    c for c in df.columns
    if c not in EXCLUIR
]

num_cols = [
    c for c in features
    if c != "cultivo"
]

cat_cols = ["cultivo"]

validation_years = [
    2020,
    2021,
    2022
]

print("=" * 90)
print("PASO 87 - PERMUTATION IMPORTANCE TEMPORAL")
print("=" * 90)

print("Dataset:", df.shape)
print("Predictores:", len(features))
print("Años evaluados:", validation_years)
print("2023 utilizado: NO")


# ============================================================
# 2. PREPROCESADO
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            cat_cols
        )
    ]
)


# ============================================================
# 3. RANDOM FOREST DEFINITIVO
# ============================================================

rf = RandomForestRegressor(
    n_estimators=800,
    max_depth=15,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features="sqrt",
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)


# ============================================================
# 4. PERMUTATION IMPORTANCE POR FOLD
# ============================================================

resultados = []

metricas_folds = []

for ano_val in validation_years:

    train = df[
        df["ano"] < ano_val
    ].copy()

    val = df[
        df["ano"] == ano_val
    ].copy()

    X_train = train[features]
    y_train = train[TARGET]

    X_val = val[features]
    y_val = val[TARGET]

    pipe = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "modelo",
                rf
            )
        ]
    )

    pipe.fit(
        X_train,
        y_train
    )

    pred = pipe.predict(X_val)

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            pred
        )
    )

    r2 = r2_score(
        y_val,
        pred
    )

    metricas_folds.append({
        "ano_validacion": ano_val,
        "RMSE": rmse,
        "R2": r2
    })

    print(
        f"\nCalculando permutation importance "
        f"para {ano_val}..."
    )

    perm = permutation_importance(
        pipe,
        X_val,
        y_val,

        # Queremos medir cuánto empeora el RMSE
        scoring="neg_root_mean_squared_error",

        n_repeats=20,
        random_state=42,
        n_jobs=-1
    )

    # permutation_importance devuelve reducción del score.
    # Como el score es -RMSE, valores positivos indican
    # que permutar esa variable empeora el modelo.
    for variable, imp, std in zip(
        features,
        perm.importances_mean,
        perm.importances_std
    ):

        resultados.append({
            "ano_validacion":
                ano_val,

            "variable":
                variable,

            "importancia":
                imp,

            "importancia_std_repeticiones":
                std
        })


# ============================================================
# 5. RESULTADO COMPLETO
# ============================================================

imp_folds = pd.DataFrame(
    resultados
)

print("\n" + "=" * 90)
print("MÉTRICAS DE LOS FOLDS")
print("=" * 90)

display(
    pd.DataFrame(
        metricas_folds
    ).round(3)
)


# ============================================================
# 6. AGREGAR IMPORTANCIA ENTRE AÑOS
# ============================================================

resumen = (
    imp_folds
    .groupby("variable")
    .agg(
        importancia_media=(
            "importancia",
            "mean"
        ),

        importancia_std_temporal=(
            "importancia",
            "std"
        ),

        importancia_min=(
            "importancia",
            "min"
        ),

        importancia_max=(
            "importancia",
            "max"
        ),

        n_anos_positiva=(
            "importancia",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
    .sort_values(
        "importancia_media",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# 7. TOP 30
# ============================================================

print("\n" + "=" * 90)
print("TOP 30 VARIABLES - PERMUTATION IMPORTANCE TEMPORAL")
print("=" * 90)

display(
    resumen.head(30).round(3)
)


# ============================================================
# 8. VARIABLES CONSISTENTES
# ============================================================

consistentes = resumen[
    resumen["n_anos_positiva"] == 3
].copy()

print("\n" + "=" * 90)
print("VARIABLES CON IMPORTANCIA POSITIVA EN LOS 3 AÑOS")
print("=" * 90)

display(
    consistentes.head(30).round(3)
)


# ============================================================
# 9. VARIABLES POR FAMILIA
# ============================================================

def familia(variable):

    if variable.startswith(
        ("NDVI", "EVI", "NDMI")
    ):
        return "Sentinel-2"

    elif variable.startswith(
        (
            "temp_",
            "precipitacion_",
            "radiacion_"
        )
    ):
        return "ERA5-Land"

    elif variable in [
        "clay_pct_0_30cm",
        "sand_pct_0_30cm",
        "soc_g_kg_0_30cm",
        "ph_0_30cm",
        "bdod_g_cm3_0_30cm"
    ]:
        return "SoilGrids"

    elif variable in [
        "n_recintos",
        "superficie_eurocrops_ha",
        "superficie_media_recinto_ha"
    ]:
        return "EuroCrops"

    elif variable == "cultivo":
        return "Cultivo"

    else:
        return "Otros"


resumen["familia"] = (
    resumen["variable"]
    .apply(familia)
)

familias = (
    resumen
    .groupby("familia")
    .agg(
        importancia_media_total=(
            "importancia_media",
            "sum"
        ),

        importancia_media_variable=(
            "importancia_media",
            "mean"
        ),

        n_variables=(
            "variable",
            "count"
        )
    )
    .sort_values(
        "importancia_media_total",
        ascending=False
    )
    .reset_index()
)


print("\n" + "=" * 90)
print("IMPORTANCIA AGREGADA POR FUENTE DE DATOS")
print("=" * 90)

display(
    familias.round(3)
)


# ============================================================
# 10. GUARDAR
# ============================================================

os.makedirs(
    RESULTADOS,
    exist_ok=True
)

F_FOLDS = (
    f"{RESULTADOS}/"
    "paso87_permutation_importance_folds.csv"
)

F_RESUMEN = (
    f"{RESULTADOS}/"
    "paso87_permutation_importance_resumen.csv"
)

F_FAMILIAS = (
    f"{RESULTADOS}/"
    "paso87_importancia_por_fuente.csv"
)

imp_folds.to_csv(
    F_FOLDS,
    index=False
)

resumen.to_csv(
    F_RESUMEN,
    index=False
)

familias.to_csv(
    F_FAMILIAS,
    index=False
)

print("\nArchivos guardados:")
print(F_FOLDS)
print(F_RESUMEN)
print(F_FAMILIAS)

print("\n" + "=" * 90)
print("CONTROL METODOLÓGICO")
print("=" * 90)

print("Importancia calculada sobre: 2020-2022")
print("Modelo: RF_tuned")
print("2023 utilizado: NO")

print("\nPASO 87 FINALIZADO")

# SHAP definitivo para Random Forest

In [ ]:
# ============================================================
# PASO 88
# SHAP DEFINITIVO - RANDOM FOREST
# Dataset ERA5 definitivo
#
# Objetivo:
# - Interpretar el modelo final
# - NO seleccionar variables
# - NO hacer tuning
# - 2023 NO se utiliza
# ============================================================

!pip -q install shap

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# ============================================================
# 1. DATOS
# ============================================================

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"
RESULTADOS = f"{ROOT}/05_Resultados"

F_DATASET = (
    f"{PROC}/dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

df = pd.read_csv(F_DATASET)

TARGET = "rendimiento_kg_ha"

# Solo desarrollo
dev = df[df["ano"] <= 2022].copy()

EXCLUIR = [
    TARGET,
    "ano",
    "provincia_estandar"
]

features = [
    c for c in df.columns
    if c not in EXCLUIR
]

num_cols = [
    c for c in features
    if c != "cultivo"
]

cat_cols = ["cultivo"]

X = dev[features]
y = dev[TARGET]

print("=" * 90)
print("PASO 88 - SHAP DEFINITIVO")
print("=" * 90)

print("Datos utilizados:", dev.shape)
print("Años:", sorted(dev["ano"].unique()))
print("Predictores originales:", len(features))
print("2023 utilizado: NO")


# ============================================================
# 2. PREPROCESADO
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
                sparse_output=False
            ),
            cat_cols
        )
    ],
    verbose_feature_names_out=False
)


# ============================================================
# 3. TRANSFORMAR DATOS
# ============================================================

X_trans = preprocessor.fit_transform(X)

feature_names = (
    preprocessor.get_feature_names_out()
)

X_trans_df = pd.DataFrame(
    X_trans,
    columns=feature_names,
    index=X.index
)

print("\nVariables después del preprocesado:", len(feature_names))


# ============================================================
# 4. RANDOM FOREST DEFINITIVO
# ============================================================

rf = RandomForestRegressor(
    n_estimators=800,
    max_depth=15,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features="sqrt",
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_trans_df,
    y
)

print("Random Forest entrenado.")


# ============================================================
# 5. SHAP
# ============================================================

print("\nCalculando SHAP...")

explainer = shap.TreeExplainer(rf)

shap_values = explainer.shap_values(
    X_trans_df
)

shap_array = np.asarray(shap_values)

print(
    "Shape SHAP:",
    shap_array.shape
)


# ============================================================
# 6. IMPORTANCIA GLOBAL SHAP
# ============================================================

mean_abs_shap = np.abs(
    shap_array
).mean(axis=0)

shap_importancia = pd.DataFrame({
    "variable": feature_names,
    "mean_abs_SHAP": mean_abs_shap
})

shap_importancia = (
    shap_importancia
    .sort_values(
        "mean_abs_SHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 90)
print("TOP 30 VARIABLES SEGÚN SHAP")
print("=" * 90)

display(
    shap_importancia.head(30).round(3)
)


# ============================================================
# 7. SUMMARY PLOT
# ============================================================

plt.figure()

shap.summary_plot(
    shap_array,
    X_trans_df,
    max_display=20,
    show=False
)

plt.title(
    "SHAP summary - Random Forest"
)

plt.tight_layout()
plt.show()


# ============================================================
# 8. BAR PLOT
# ============================================================

plt.figure()

shap.summary_plot(
    shap_array,
    X_trans_df,
    plot_type="bar",
    max_display=20,
    show=False
)

plt.title(
    "Importancia global SHAP - Random Forest"
)

plt.tight_layout()
plt.show()


# ============================================================
# 9. COMPARAR SHAP CON PERMUTATION IMPORTANCE
# ============================================================

F_PERM = (
    f"{RESULTADOS}/"
    "paso87_permutation_importance_resumen.csv"
)

perm = pd.read_csv(F_PERM)

# En este dataset cultivo se transforma en una dummy.
# Para la comparación principal nos centramos en variables
# que conservan exactamente el mismo nombre.

comparacion = shap_importancia.merge(
    perm[
        [
            "variable",
            "importancia_media",
            "n_anos_positiva"
        ]
    ],
    on="variable",
    how="inner"
)

comparacion["rank_SHAP"] = (
    comparacion["mean_abs_SHAP"]
    .rank(
        ascending=False,
        method="min"
    )
)

comparacion["rank_permutation"] = (
    comparacion["importancia_media"]
    .rank(
        ascending=False,
        method="min"
    )
)

comparacion = comparacion.sort_values(
    "rank_SHAP"
)

print("\n" + "=" * 90)
print("COMPARACIÓN SHAP vs PERMUTATION IMPORTANCE")
print("=" * 90)

display(
    comparacion.head(30).round(3)
)


# ============================================================
# 10. GUARDAR
# ============================================================

F_SHAP = (
    f"{RESULTADOS}/"
    "paso88_SHAP_importancia_RF.csv"
)

F_COMP = (
    f"{RESULTADOS}/"
    "paso88_SHAP_vs_permutation.csv"
)

shap_importancia.to_csv(
    F_SHAP,
    index=False
)

comparacion.to_csv(
    F_COMP,
    index=False
)

print("\nArchivos guardados:")
print(F_SHAP)
print(F_COMP)

print("\n" + "=" * 90)
print("CONTROL METODOLÓGICO")
print("=" * 90)

print("Modelo interpretado: RF_tuned")
print("Datos usados: 2017-2022")
print("2023 utilizado: NO")
print("SHAP se utiliza únicamente para interpretación.")

print("\nPASO 88 FINALIZADO")

# Figuras de rendimiento y diagnostico

In [ ]:
# ============================================================
# PASO 89
# FIGURAS FINALES - RENDIMIENTO Y DIAGNÓSTICO
#
# NO entrena modelos
# NO modifica modelos
# NO realiza selección de variables
# Utiliza exclusivamente resultados ya obtenidos
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# 1. RUTAS
# ============================================================

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"
RESULTADOS = f"{ROOT}/05_Resultados"

FIGURAS = f"{RESULTADOS}/Figuras_finales"
os.makedirs(FIGURAS, exist_ok=True)

F_DATASET = (
    f"{PROC}/dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

F_VALIDACION = (
    f"{RESULTADOS}/paso83_revalidacion_final_ERA5_DEFINITIVO.csv"
)

F_RESIDUOS = (
    f"{RESULTADOS}/paso86_residuos_test_2023.csv"
)

# ============================================================
# 2. CARGA
# ============================================================

df = pd.read_csv(F_DATASET)
val = pd.read_csv(F_VALIDACION)
res = pd.read_csv(F_RESIDUOS)

print("=" * 90)
print("PASO 89 - FIGURAS FINALES")
print("=" * 90)

print("\nDataset:", df.shape)
print("Validación:", val.shape)
print("Residuos 2023:", res.shape)

print("\nColumnas residuos 2023:")
print(res.columns.tolist())


# ============================================================
# 3. FIGURA 1
# EVOLUCIÓN TEMPORAL DEL RENDIMIENTO
# ============================================================

rend_anual = (
    df.groupby("ano")["rendimiento_kg_ha"]
    .agg(["mean", "median", "std"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    rend_anual["ano"],
    rend_anual["mean"],
    marker="o",
    linewidth=2,
    label="Media"
)

ax.plot(
    rend_anual["ano"],
    rend_anual["median"],
    marker="s",
    linewidth=2,
    label="Mediana"
)

ax.axvline(
    2022.5,
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel("Año")
ax.set_ylabel("Rendimiento (kg/ha)")
ax.set_title(
    "Evolución temporal del rendimiento observado"
)

ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()

f1 = f"{FIGURAS}/Figura_01_rendimiento_anual.png"

plt.savefig(
    f1,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 4. FIGURA 2
# RENDIMIENTO POR CULTIVO
# ============================================================

rend_cultivo = (
    df.groupby(["ano", "cultivo"])["rendimiento_kg_ha"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))

for cultivo, datos in rend_cultivo.groupby("cultivo"):

    etiqueta = (
        "Cebada"
        if cultivo == "barley"
        else "Trigo blando"
    )

    ax.plot(
        datos["ano"],
        datos["rendimiento_kg_ha"],
        marker="o",
        linewidth=2,
        label=etiqueta
    )

ax.axvline(
    2022.5,
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel("Año")
ax.set_ylabel("Rendimiento medio (kg/ha)")
ax.set_title(
    "Evolución del rendimiento por cultivo"
)

ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()

f2 = f"{FIGURAS}/Figura_02_rendimiento_por_cultivo.png"

plt.savefig(
    f2,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 5. FIGURA 3
# R² EN VALIDACIÓN TEMPORAL
# ============================================================

pivot_r2 = val.pivot(
    index="ano_validacion",
    columns="modelo",
    values="R2"
)

fig, ax = plt.subplots(figsize=(9, 5))

for modelo in pivot_r2.columns:

    ax.plot(
        pivot_r2.index,
        pivot_r2[modelo],
        marker="o",
        linewidth=2,
        label=modelo
    )

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("Año de validación")
ax.set_ylabel("R²")
ax.set_title(
    "Rendimiento de los modelos en validación temporal"
)

ax.set_xticks(
    pivot_r2.index
)

ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()

f3 = f"{FIGURAS}/Figura_03_R2_validacion_temporal.png"

plt.savefig(
    f3,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 6. FIGURA 4
# RMSE EN VALIDACIÓN TEMPORAL
# ============================================================

pivot_rmse = val.pivot(
    index="ano_validacion",
    columns="modelo",
    values="RMSE"
)

fig, ax = plt.subplots(figsize=(9, 5))

for modelo in pivot_rmse.columns:

    ax.plot(
        pivot_rmse.index,
        pivot_rmse[modelo],
        marker="o",
        linewidth=2,
        label=modelo
    )

ax.set_xlabel("Año de validación")
ax.set_ylabel("RMSE (kg/ha)")
ax.set_title(
    "RMSE en validación temporal"
)

ax.set_xticks(
    pivot_rmse.index
)

ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()

f4 = f"{FIGURAS}/Figura_04_RMSE_validacion_temporal.png"

plt.savefig(
    f4,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 7. IDENTIFICAR COLUMNAS DE PREDICCIÓN
# ============================================================

pred_cols = [
    c for c in res.columns
    if c.startswith("pred_")
]

print("\nColumnas de predicción encontradas:")
print(pred_cols)

# XGBoost fue el modelo con menor error en test 2023
pred_xgb_candidates = [
    c for c in pred_cols
    if "XGB_tuned" in c
]

if len(pred_xgb_candidates) == 0:
    raise ValueError(
        "No se ha encontrado la predicción XGB_tuned "
        "en paso86_residuos_test_2023.csv"
    )

PRED_XGB = pred_xgb_candidates[0]

print("\nPredicción XGBoost utilizada:")
print(PRED_XGB)


# ============================================================
# 8. FIGURA 5
# OBSERVADO vs PREDICHO 2023 - XGBOOST
# ============================================================

y_obs = res["observado"]
y_pred = res[PRED_XGB]

mae = mean_absolute_error(
    y_obs,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_obs,
        y_pred
    )
)

r2 = r2_score(
    y_obs,
    y_pred
)

fig, ax = plt.subplots(figsize=(6.5, 6.5))

ax.scatter(
    y_obs,
    y_pred,
    alpha=0.7,
    s=45
)

lim_min = min(
    y_obs.min(),
    y_pred.min()
)

lim_max = max(
    y_obs.max(),
    y_pred.max()
)

ax.plot(
    [lim_min, lim_max],
    [lim_min, lim_max],
    linestyle="--",
    linewidth=1.5
)

ax.set_xlim(lim_min, lim_max)
ax.set_ylim(lim_min, lim_max)

ax.set_xlabel(
    "Rendimiento observado (kg/ha)"
)

ax.set_ylabel(
    "Rendimiento predicho (kg/ha)"
)

ax.set_title(
    "Test independiente 2023 – XGBoost"
)

texto = (
    f"MAE = {mae:.0f} kg/ha\n"
    f"RMSE = {rmse:.0f} kg/ha\n"
    f"R² = {r2:.3f}"
)

ax.text(
    0.05,
    0.95,
    texto,
    transform=ax.transAxes,
    verticalalignment="top",
    bbox=dict(
        boxstyle="round",
        alpha=0.15
    )
)

ax.grid(alpha=0.20)

plt.tight_layout()

f5 = f"{FIGURAS}/Figura_05_observado_predicho_2023_XGB.png"

plt.savefig(
    f5,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. FIGURA 6
# RESIDUOS XGBOOST 2023
# ============================================================

residuo = y_obs - y_pred

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    y_pred,
    residuo,
    alpha=0.7,
    s=45
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel(
    "Rendimiento predicho (kg/ha)"
)

ax.set_ylabel(
    "Residuo: observado - predicho (kg/ha)"
)

ax.set_title(
    "Distribución de residuos – XGBoost, test 2023"
)

ax.grid(alpha=0.20)

plt.tight_layout()

f6 = f"{FIGURAS}/Figura_06_residuos_2023_XGB.png"

plt.savefig(
    f6,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 10. FIGURA 7
# DISTRIBUCIÓN TRAIN vs TEST 2023
# ============================================================

train_target = df.loc[
    df["ano"] <= 2022,
    "rendimiento_kg_ha"
]

test_target = df.loc[
    df["ano"] == 2023,
    "rendimiento_kg_ha"
]

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    train_target,
    bins=25,
    alpha=0.55,
    density=True,
    label="2017–2022"
)

ax.hist(
    test_target,
    bins=20,
    alpha=0.55,
    density=True,
    label="2023"
)

ax.axvline(
    train_target.mean(),
    linestyle="--",
    linewidth=1.5
)

ax.axvline(
    test_target.mean(),
    linestyle=":",
    linewidth=2
)

ax.set_xlabel(
    "Rendimiento (kg/ha)"
)

ax.set_ylabel(
    "Densidad"
)

ax.set_title(
    "Cambio en la distribución del rendimiento: desarrollo vs test 2023"
)

ax.legend()
ax.grid(alpha=0.20)

plt.tight_layout()

f7 = f"{FIGURAS}/Figura_07_distribucion_target_train_vs_2023.png"

plt.savefig(
    f7,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 11. CONTROL
# ============================================================

print("\n" + "=" * 90)
print("FIGURAS GENERADAS")
print("=" * 90)

for f in [
    f1, f2, f3, f4, f5, f6, f7
]:
    print(
        os.path.basename(f),
        "->",
        os.path.exists(f)
    )

print("\n" + "=" * 90)
print("CONTROL METODOLÓGICO")
print("=" * 90)

print("Reentrenamiento de modelos: NO")
print("Tuning adicional: NO")
print("Selección de variables: NO")
print("2023: exclusivamente test independiente")
print(
    "Modelo mostrado en observado/predicho:",
    "XGB_tuned"
)

print("\nPASO 89 FINALIZADO")

# Figuras de interpretabilidad

In [ ]:
# ============================================================
# PASO 90
# FIGURAS FINALES DE INTERPRETABILIDAD
#
# NO reentrena modelos
# NO recalcula importancias
# Utiliza resultados guardados de los pasos 87 y 88
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

ROOT = "/content/TFM_Rendimiento_cultivos"
RESULTADOS = f"{ROOT}/05_Resultados"
FIGURAS = f"{RESULTADOS}/Figuras_finales"

os.makedirs(FIGURAS, exist_ok=True)

F_PERM = (
    f"{RESULTADOS}/"
    "paso87_permutation_importance_resumen.csv"
)

F_FAMILIAS = (
    f"{RESULTADOS}/"
    "paso87_importancia_por_fuente.csv"
)

F_SHAP = (
    f"{RESULTADOS}/"
    "paso88_SHAP_importancia_RF.csv"
)

# ============================================================
# 1. CARGAR
# ============================================================

perm = pd.read_csv(F_PERM)
familias = pd.read_csv(F_FAMILIAS)
shap_imp = pd.read_csv(F_SHAP)

print("=" * 90)
print("PASO 90 - FIGURAS DE INTERPRETABILIDAD")
print("=" * 90)

print("\nPermutation Importance:", perm.shape)
print("SHAP:", shap_imp.shape)
print("Fuentes:", familias.shape)

# ============================================================
# 2. FIGURA 8
# TOP 15 PERMUTATION IMPORTANCE
# ============================================================

top_perm = (
    perm
    .sort_values(
        "importancia_media",
        ascending=False
    )
    .head(15)
    .sort_values(
        "importancia_media",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(9, 7))

ax.barh(
    top_perm["variable"],
    top_perm["importancia_media"]
)

ax.set_xlabel(
    "Incremento medio del RMSE tras permutación"
)

ax.set_ylabel("Variable")

ax.set_title(
    "Importancia temporal de variables - Random Forest"
)

ax.grid(
    axis="x",
    alpha=0.20
)

plt.tight_layout()

f8 = (
    f"{FIGURAS}/"
    "Figura_08_permutation_importance_top15.png"
)

plt.savefig(
    f8,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 3. FIGURA 9
# TOP 15 SHAP
# ============================================================

top_shap = (
    shap_imp
    .sort_values(
        "mean_abs_SHAP",
        ascending=False
    )
    .head(15)
    .sort_values(
        "mean_abs_SHAP",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(9, 7))

ax.barh(
    top_shap["variable"],
    top_shap["mean_abs_SHAP"]
)

ax.set_xlabel(
    "Media del valor absoluto SHAP (kg/ha)"
)

ax.set_ylabel("Variable")

ax.set_title(
    "Importancia global SHAP - Random Forest"
)

ax.grid(
    axis="x",
    alpha=0.20
)

plt.tight_layout()

f9 = (
    f"{FIGURAS}/"
    "Figura_09_SHAP_top15.png"
)

plt.savefig(
    f9,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 4. FIGURA 10
# IMPORTANCIA POR FUENTE DE DATOS
# ============================================================

fuentes = (
    familias
    .sort_values(
        "importancia_media_total",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(
    fuentes["familia"],
    fuentes["importancia_media_total"]
)

ax.set_xlabel(
    "Importancia temporal agregada"
)

ax.set_ylabel(
    "Fuente de datos"
)

ax.set_title(
    "Contribución agregada por fuente de información"
)

ax.grid(
    axis="x",
    alpha=0.20
)

plt.tight_layout()

f10 = (
    f"{FIGURAS}/"
    "Figura_10_importancia_por_fuente.png"
)

plt.savefig(
    f10,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 5. TABLA RESUMEN DE CONCORDANCIA
# ============================================================

comp = (
    perm[
        [
            "variable",
            "importancia_media",
            "n_anos_positiva"
        ]
    ]
    .merge(
        shap_imp[
            [
                "variable",
                "mean_abs_SHAP"
            ]
        ],
        on="variable",
        how="inner"
    )
)

comp["rank_perm"] = (
    comp["importancia_media"]
    .rank(
        ascending=False,
        method="min"
    )
)

comp["rank_shap"] = (
    comp["mean_abs_SHAP"]
    .rank(
        ascending=False,
        method="min"
    )
)

comp["rank_medio"] = (
    comp["rank_perm"] +
    comp["rank_shap"]
) / 2

comp = (
    comp
    .sort_values(
        "rank_medio"
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 90)
print("TOP 15 VARIABLES CONSISTENTES ENTRE MÉTODOS")
print("=" * 90)

display(
    comp.head(15).round(3)
)

# ============================================================
# 6. GUARDAR TABLA
# ============================================================

F_COMP = (
    f"{RESULTADOS}/"
    "paso90_variables_consistentes_SHAP_permutation.csv"
)

comp.to_csv(
    F_COMP,
    index=False
)

# ============================================================
# 7. CONTROL FINAL
# ============================================================

print("\n" + "=" * 90)
print("FIGURAS GENERADAS")
print("=" * 90)

for f in [
    f8,
    f9,
    f10
]:
    print(
        os.path.basename(f),
        "->",
        os.path.exists(f)
    )

print("\nTabla guardada:")
print(F_COMP)

print("\nPASO 90 FINALIZADO")

# Consolidar las tablas definitivas del TFM

In [ ]:
# ============================================================
# PASO 91
# CONSOLIDACIÓN DE TABLAS FINALES PARA EL TFM
#
# NO modelos
# NO tuning
# NO nuevas decisiones metodológicas
# ============================================================

import os
import pandas as pd

ROOT = "/content/TFM_Rendimiento_cultivos"
RESULTADOS = f"{ROOT}/05_Resultados"

TABLAS = f"{RESULTADOS}/Tablas_finales"
os.makedirs(TABLAS, exist_ok=True)

# ============================================================
# 1. VALIDACIÓN TEMPORAL FINAL
# ============================================================

F_VALIDACION = (
    f"{RESULTADOS}/"
    "paso83_revalidacion_final_ERA5_DEFINITIVO.csv"
)

val = pd.read_csv(F_VALIDACION)

tabla_validacion = (
    val.groupby("modelo")
    .agg(
        MAE_medio=("MAE", "mean"),
        RMSE_medio=("RMSE", "mean"),
        R2_medio=("R2", "mean"),
        R2_std=("R2", "std"),
        R2_min=("R2", "min"),
        R2_max=("R2", "max")
    )
    .sort_values("RMSE_medio")
    .round(3)
    .reset_index()
)

print("=" * 90)
print("TABLA 1 - VALIDACIÓN TEMPORAL 2020-2022")
print("=" * 90)

display(tabla_validacion)

# ============================================================
# 2. TEST EXTERNO 2023
# ============================================================

F_TEST = (
    f"{RESULTADOS}/"
    "paso84_test_final_2023_metricas.csv"
)

tabla_test = (
    pd.read_csv(F_TEST)
    .sort_values("RMSE")
    .round(3)
)

print("\n" + "=" * 90)
print("TABLA 2 - TEST INDEPENDIENTE 2023")
print("=" * 90)

display(tabla_test)

# ============================================================
# 3. MODELOS vs BASELINES
# ============================================================

F_BASELINES = (
    f"{RESULTADOS}/"
    "paso85_diagnostico_target_y_baselines.csv"
)

tabla_baselines = (
    pd.read_csv(F_BASELINES)
    .sort_values("RMSE")
    .round(3)
)

print("\n" + "=" * 90)
print("TABLA 3 - MODELOS vs BASELINES 2023")
print("=" * 90)

display(tabla_baselines)

# ============================================================
# 4. RESULTADOS POR CULTIVO
# ============================================================

F_CULTIVO = (
    f"{RESULTADOS}/"
    "paso84_test_final_2023_por_cultivo.csv"
)

tabla_cultivo = (
    pd.read_csv(F_CULTIVO)
    .round(3)
)

print("\n" + "=" * 90)
print("TABLA 4 - RESULTADOS POR CULTIVO")
print("=" * 90)

display(tabla_cultivo)

# ============================================================
# 5. TOP VARIABLES CONSISTENTES
# ============================================================

F_VARIABLES = (
    f"{RESULTADOS}/"
    "paso90_variables_consistentes_SHAP_permutation.csv"
)

tabla_variables = (
    pd.read_csv(F_VARIABLES)
    .head(15)
    .round(3)
)

print("\n" + "=" * 90)
print("TABLA 5 - VARIABLES MÁS CONSISTENTES")
print("=" * 90)

display(tabla_variables)

# ============================================================
# 6. IMPORTANCIA POR FUENTE
# ============================================================

F_FUENTES = (
    f"{RESULTADOS}/"
    "paso87_importancia_por_fuente.csv"
)

tabla_fuentes = (
    pd.read_csv(F_FUENTES)
    .round(3)
)

print("\n" + "=" * 90)
print("TABLA 6 - IMPORTANCIA POR FUENTE DE DATOS")
print("=" * 90)

display(tabla_fuentes)

# ============================================================
# 7. GUARDAR TABLAS DEFINITIVAS
# ============================================================

tablas = {
    "Tabla_01_validacion_temporal.csv":
        tabla_validacion,

    "Tabla_02_test_2023.csv":
        tabla_test,

    "Tabla_03_modelos_vs_baselines.csv":
        tabla_baselines,

    "Tabla_04_resultados_por_cultivo.csv":
        tabla_cultivo,

    "Tabla_05_variables_consistentes.csv":
        tabla_variables,

    "Tabla_06_importancia_fuentes.csv":
        tabla_fuentes
}

for nombre, tabla in tablas.items():

    ruta = os.path.join(
        TABLAS,
        nombre
    )

    tabla.to_csv(
        ruta,
        index=False
    )

    print(
        nombre,
        "->",
        os.path.exists(ruta)
    )

print("\n" + "=" * 90)
print("PASO 91 FINALIZADO")
print("=" * 90)

print(
    "\nCarpeta de tablas finales:\n",
    TABLAS
)